In [4]:
import pandas
import numpy
import sklearn
import rapidfuzz
import unidecode

print("All libraries installed successfully!")

All libraries installed successfully!


In [5]:
import os

print(os.listdir("."))

['.bashrc', '.agent', '.kiro', '.claude', '.sagemaker_sql_editor_api_cache', '.local', '.ipython', '.npm', '.jupyter', 'untitled.chat', 'Untitled.ipynb', '.ipynb_checkpoints', '.cache', '.config', '.virtual_documents', 'business_entity_resolution.ipynb']


In [6]:
import sagemaker

session = sagemaker.Session()

print("Region:", session.boto_region_name)
print("S3 bucket:", session.default_bucket())

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│   1 import sagemaker                                                                             │
│   2                                                                                              │
│ ❱ 3 session = sagemaker.Session()                                                                │
│   4                                                                                              │
│   5 print("Region:", session.boto_region_name)                                                   │
│   6 print("S3 bucket:", session.default_bucket())                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
AttributeError: module 'sagemaker' has no attribute 'Session'

In [7]:
import boto3

session = boto3.Session()

print("Region:", session.region_name)

s3 = session.client("s3")
response = s3.list_buckets()

print("\nS3 buckets:")
for bucket in response["Buckets"]:
    print(bucket["Name"])

Region: eu-north-1

S3 buckets:
sagemaker-eu-north-1-152025327817


In [11]:
import boto3

s3 = boto3.client("s3")

bucket = "sagemaker-eu-north-1-152025327817"

response = s3.list_objects_v2(Bucket=bucket)

print("Files found:")
for obj in response.get("Contents", []):
    print(obj["Key"])

Files found:
dataset/.DS_Store
dataset/test/test_source1.tsv
dataset/test/test_source2.tsv
dataset/test/test_source3.tsv
dataset/train/train_ground_truth.tsv
dataset/train/train_source1.tsv
dataset/train/train_source2.tsv
dataset/train/train_source3.tsv


In [12]:
import pandas as pd

bucket = "sagemaker-eu-north-1-152025327817"

ground_truth = pd.read_csv(
    f"s3://{bucket}/dataset/train/train_ground_truth.tsv",
    sep="\t"
)

print("Shape:", ground_truth.shape)
print(ground_truth.head())

Shape: (2206821, 2)
  source1_entity_id                                 matched_entity_ids
0         S1-965667  S2-681193310,S2-743505751,S3-775321672,S3-1129...
1       S1-55344266  S2-249013014,S2-197070651,S3-478195123,S3-3843...
2      S1-343815751             S2-790675320,S2-479876582,S3-878454467
3      S1-656753428              S2-153058913,S2-24659151,S3-679606215
4      S1-102811957  S2-478959098,S2-553508714,S2-625774905,S3-7280...


In [13]:
import pandas as pd

bucket = "sagemaker-eu-north-1-152025327817"

for source in ["source1", "source2", "source3"]:
    path = f"s3://{bucket}/dataset/train/train_{source}.tsv"
    
    df = pd.read_csv(path, sep="\t", nrows=5)
    
    print(f"\n===== {source.upper()} =====")
    print("Columns:", list(df.columns))
    print(df)


===== SOURCE1 =====
Columns: ['entity_id', 'business_name', 'business_address', 'country']
      entity_id            business_name  \
0  S1-925783039      Orelee's Barbershop   
1  S1-773889195              Prime Money   
2  S1-377745466            B+ Retail Inc   
3  S1-133037285            Christ Chapel   
4  S1-755362802  Prabhav Business Center   

                                    business_address country  
0             1795 Westchester Drive, High Point, NC      US  
1                    17560 Ellis Road, Tahlequah, OK      US  
2                1712 Montebello Avenue, Phoenix, AZ      US  
3  2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD      US  
4  797, Lake Town Block A, Kolkata, Howrah, West ...   India  

===== SOURCE2 =====
Columns: ['entity_id', 'business_name', 'business_address', 'country']
      entity_id                    business_name  \
0  S2-166376419  राम मार्केटिंग प्राइवेट लिमिटेड   
1  S2-764573417     -- Holloway Peak Inc Seafood   
2  S2-639257739  

In [14]:
import psutil

ram_gb = psutil.virtual_memory().total / (1024 ** 3)

print(f"Total RAM: {ram_gb:.2f} GB")

Total RAM: 3.74 GB


In [15]:
import boto3

s3 = boto3.client("s3")
bucket = "sagemaker-eu-north-1-152025327817"

files = [
    "dataset/train/train_source1.tsv",
    "dataset/train/train_source2.tsv",
    "dataset/train/train_source3.tsv",
    "dataset/train/train_ground_truth.tsv"
]

for key in files:
    obj = s3.head_object(Bucket=bucket, Key=key)
    size_gb = obj["ContentLength"] / (1024**3)
    print(f"{key}: {size_gb:.2f} GB")

dataset/train/train_source1.tsv: 0.20 GB
dataset/train/train_source2.tsv: 0.46 GB
dataset/train/train_source3.tsv: 0.47 GB
dataset/train/train_ground_truth.tsv: 0.12 GB


In [16]:
import pandas as pd

source1_path = "s3://sagemaker-eu-north-1-152025327817/dataset/train/train_source1.tsv"

total_rows = 0

for chunk in pd.read_csv(
    source1_path,
    sep="\t",
    chunksize=100_000
):
    total_rows += len(chunk)

print("Total Source 1 rows:", total_rows)

Total Source 1 rows: 2206821


In [17]:
import pandas as pd

source1_path = "s3://sagemaker-eu-north-1-152025327817/dataset/train/train_source1.tsv"

sample = pd.read_csv(
    source1_path,
    sep="\t",
    nrows=20
)

print(sample[["entity_id", "business_name", "business_address", "country"]].to_string(index=False))

   entity_id                            business_name                                                                                                          business_address country
S1-925783039                      Orelee's Barbershop                                                                                    1795 Westchester Drive, High Point, NC      US
S1-773889195                              Prime Money                                                                                           17560 Ellis Road, Tahlequah, OK      US
S1-377745466                            B+ Retail Inc                                                                                       1712 Montebello Avenue, Phoenix, AZ      US
S1-133037285                            Christ Chapel                                                                         2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD      US
S1-755362802                  Prabhav Business Center                           

In [18]:
sample_ids = sample["entity_id"].tolist()

sample_truth = ground_truth[
    ground_truth["source1_entity_id"].isin(sample_ids)
]

print(sample_truth.to_string(index=False))

source1_entity_id                                                                         matched_entity_ids
     S1-236312382                                                      S2-734049056,S2-69010626,S3-591515133
     S1-629417405 S2-522538408,S2-928426462,S3-613837325,S3-606982601,S3-962449159,S3-438606634,S3-786131946
     S1-309349399                                                                                        NaN
     S1-773889195              S2-970528089,S2-374107005,S3-189140669,S3-161452820,S3-202893869,S3-622232873
      S1-27541239                                         S2-199341916,S2-85756610,S3-893131105,S3-739757636
     S1-377745466                                         S2-203037501,S3-70942743,S3-402918963,S3-836170816
     S1-851869949                           S2-915265864,S2-801813753,S2-961342067,S3-949828938,S3-807524060
     S1-133037285               S2-407207105,S2-103437512,S3-476003339,S3-33144658,S3-940895145,S3-183080822
     S1-626914593  

In [23]:
import re
import unicodedata
from unidecode import unidecode


def normalize_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    
    # Convert accented/non-Latin characters where possible
    text = unidecode(text)
    
    # Remove punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def normalize_name(name):
    text = normalize_text(name)
    
    # Common legal/business suffixes
    suffixes = [
        "private limited",
        "pvt ltd",
        "pvt",
        "limited",
        "ltd",
        "llc",
        "incorporated",
        "inc",
        "corporation",
        "corp",
        "company",
        "co"
    ]
    
    for suffix in suffixes:
        text = re.sub(rf"\b{suffix}\b", " ", text)
    
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def normalize_address(address):
    text = normalize_text(address)
    
    # Common address abbreviations
    replacements = {
        "street": "st",
        "road": "rd",
        "avenue": "ave",
        "boulevard": "blvd",
        "drive": "dr",
        "lane": "ln",
        "highway": "hwy",
        "apartment": "apt",
        "suite": "ste",
        "floor": "fl"
    }
    
    for full, short in replacements.items():
        text = re.sub(rf"\b{full}\b", short, text)
    
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [22]:
test_names = [
    "Kelly Advisory, Inc.",
    "Kelly Advisory Inc",
    "Kelly Advisory INC.",
    "Crystal Lending PC",
    "Prabhav Business Center"
]

for name in test_names:
    print(f"{name}  ->  {normalize_name(name)}")

Kelly Advisory, Inc.  ->  kelly advisory
Kelly Advisory Inc  ->  kelly advisory
Kelly Advisory INC.  ->  kelly advisory
Crystal Lending PC  ->  crystal lending pc
Prabhav Business Center  ->  prabhav business center


In [24]:
def normalize_name(name):
    text = normalize_text(name)
    
    suffixes = [
        "private limited",
        "pvt ltd",
        "pvt",
        "limited",
        "ltd",
        "llc",
        "incorporated",
        "inc",
        "corporation",
        "corp",
        "company",
        "co",
        "pc"
    ]
    
    for suffix in suffixes:
        text = re.sub(rf"\b{suffix}\b", " ", text)
    
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [25]:
print(normalize_name("Crystal Lending PC"))

crystal lending


In [26]:
sample["name_norm"] = sample["business_name"].apply(normalize_name)
sample["address_norm"] = sample["business_address"].apply(normalize_address)

print(
    sample[
        ["business_name", "name_norm", "business_address", "address_norm"]
    ].to_string(index=False)
)

                           business_name                  name_norm                                                                                                          business_address                                                                                                    address_norm
                     Orelee's Barbershop        orelee s barbershop                                                                                    1795 Westchester Drive, High Point, NC                                                                               1795 westchester dr high point nc
                             Prime Money                prime money                                                                                           17560 Ellis Road, Tahlequah, OK                                                                                     17560 ellis rd tahlequah ok
                           B+ Retail Inc                   b retail                           

In [27]:
# Create a simple blocking key
sample["block_key"] = (
    sample["country"].fillna("").str.lower().str.strip()
    + "_"
    + sample["name_norm"].str.replace(" ", "", regex=False).str[:4]
)

print(
    sample[
        ["entity_id", "business_name", "country", "block_key"]
    ].to_string(index=False)
)

   entity_id                            business_name country  block_key
S1-925783039                      Orelee's Barbershop      US    us_orel
S1-773889195                              Prime Money      US    us_prim
S1-377745466                            B+ Retail Inc      US    us_bret
S1-133037285                            Christ Chapel      US    us_chri
S1-755362802                  Prabhav Business Center   India india_prab
S1-851869949               Custom Wealth Services LLC      US    us_cust
S1-785847572 Consulting Nyasa Nursing Private Limited   India india_cons
 S1-27541239                        Nexus Anchor Rain      US    us_nexu
S1-629417405                        Moore Bitwise Inc      US    us_moor
 S1-22305073               Dermatology Green Medicine      US    us_derm
S1-504790211                       Crystal Lending PC      US    us_crys
S1-564729135               Dream Construction Limited   India india_drea
S1-236312382                      Kelly Advisory, I

In [28]:
# Get the true matches for our 20 sample Source 1 records
truth_dict = dict(
    zip(
        sample_truth["source1_entity_id"],
        sample_truth["matched_entity_ids"].fillna("")
    )
)

# Show how many true matches each sample entity has
for entity_id in sample["entity_id"]:
    matches = truth_dict.get(entity_id, "")
    
    if matches:
        match_list = [x.strip() for x in matches.split(",") if x.strip()]
        print(entity_id, "->", len(match_list), "true matches")
    else:
        print(entity_id, "-> 0 true matches")

S1-925783039 -> 4 true matches
S1-773889195 -> 6 true matches
S1-377745466 -> 4 true matches
S1-133037285 -> 6 true matches
S1-755362802 -> 2 true matches
S1-851869949 -> 5 true matches
S1-785847572 -> 4 true matches
S1-27541239 -> 4 true matches
S1-629417405 -> 7 true matches
S1-22305073 -> 4 true matches
S1-504790211 -> 7 true matches
S1-564729135 -> 6 true matches
S1-236312382 -> 3 true matches
S1-865131206 -> 4 true matches
S1-626914593 -> 2 true matches
S1-765734254 -> 4 true matches
S1-82053739 -> 2 true matches
S1-309349399 -> 0 true matches
S1-221443075 -> 3 true matches
S1-341506266 -> 3 true matches


In [29]:
# Test our blocking method against the known true matches

sample_block_keys = set(sample["block_key"])

candidate_ids = {entity_id: [] for entity_id in sample["entity_id"]}

for source in ["source2", "source3"]:
    path = f"s3://{bucket}/dataset/train/train_{source}.tsv"

    print(f"Scanning {source}...")

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_name", "country"],
        chunksize=100_000
    ):
        # Normalize names
        chunk["name_norm"] = chunk["business_name"].apply(normalize_name)

        # Create the same blocking key
        chunk["block_key"] = (
            chunk["country"].fillna("").str.lower().str.strip()
            + "_"
            + chunk["name_norm"]
                .str.replace(" ", "", regex=False)
                .str[:4]
        )

        # Keep only records belonging to our 20 sample blocks
        matches = chunk[chunk["block_key"].isin(sample_block_keys)]

        for _, row in matches.iterrows():
            for entity_id in sample["entity_id"]:
                if row["block_key"] == sample.loc[
                    sample["entity_id"] == entity_id, "block_key"
                ].iloc[0]:
                    candidate_ids[entity_id].append(row["entity_id"])

    print(f"Finished {source}")

print("\nBlocking test complete!")

Scanning source2...
Finished source2
Scanning source3...
Finished source3

Blocking test complete!


In [30]:
for entity_id, ids in candidate_ids.items():
    print(entity_id, "->", len(ids), "candidates")

S1-925783039 -> 967 candidates
S1-773889195 -> 32645 candidates
S1-377745466 -> 241 candidates
S1-133037285 -> 8463 candidates
S1-755362802 -> 1065 candidates
S1-851869949 -> 10987 candidates
S1-785847572 -> 13258 candidates
S1-27541239 -> 723 candidates
S1-629417405 -> 5676 candidates
S1-22305073 -> 21935 candidates
S1-504790211 -> 10721 candidates
S1-564729135 -> 8260 candidates
S1-236312382 -> 4938 candidates
S1-865131206 -> 2083 candidates
S1-626914593 -> 12718 candidates
S1-765734254 -> 6526 candidates
S1-82053739 -> 7412 candidates
S1-309349399 -> 1581 candidates
S1-221443075 -> 389 candidates
S1-341506266 -> 2384 candidates


In [31]:
# Check whether every known true match was retrieved
# by our current blocking method.

total_true = 0
total_found = 0

for entity_id in sample["entity_id"]:
    truth = truth_dict.get(entity_id, "")
    
    true_ids = {
        x.strip()
        for x in truth.split(",")
        if x.strip()
    }
    
    candidates = set(candidate_ids[entity_id])
    
    found = true_ids & candidates
    
    total_true += len(true_ids)
    total_found += len(found)
    
    print(
        entity_id,
        f"true={len(true_ids)}",
        f"found={len(found)}",
        f"recall={len(found)/len(true_ids) if true_ids else 1.0:.2%}"
    )

print("\nOverall blocking recall:",
      f"{total_found / total_true:.2%}")

S1-925783039 true=4 found=4 recall=100.00%
S1-773889195 true=6 found=6 recall=100.00%
S1-377745466 true=4 found=3 recall=75.00%
S1-133037285 true=6 found=5 recall=83.33%
S1-755362802 true=2 found=1 recall=50.00%
S1-851869949 true=5 found=5 recall=100.00%
S1-785847572 true=4 found=4 recall=100.00%
S1-27541239 true=4 found=4 recall=100.00%
S1-629417405 true=7 found=7 recall=100.00%
S1-22305073 true=4 found=3 recall=75.00%
S1-504790211 true=7 found=7 recall=100.00%
S1-564729135 true=6 found=3 recall=50.00%
S1-236312382 true=3 found=3 recall=100.00%
S1-865131206 true=4 found=3 recall=75.00%
S1-626914593 true=2 found=2 recall=100.00%
S1-765734254 true=4 found=4 recall=100.00%
S1-82053739 true=2 found=1 recall=50.00%
S1-309349399 true=0 found=0 recall=100.00%
S1-221443075 true=3 found=3 recall=100.00%
S1-341506266 true=3 found=3 recall=100.00%

Overall blocking recall: 88.75%


In [32]:
# Find the true matches that our blocking method missed

missed_ids = set()

for entity_id in sample["entity_id"]:
    truth = truth_dict.get(entity_id, "")
    
    true_ids = {
        x.strip()
        for x in truth.split(",")
        if x.strip()
    }
    
    candidates = set(candidate_ids[entity_id])
    missed_ids.update(true_ids - candidates)

print("Number of missed true-match IDs:", len(missed_ids))
print("Missed IDs:", missed_ids)


# Retrieve the missed records from Source 2 and Source 3
for source in ["source2", "source3"]:
    
    path = f"s3://{bucket}/dataset/train/train_{source}.tsv"
    
    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_name", "business_address", "country"],
        chunksize=100_000
    ):
        
        found = chunk[chunk["entity_id"].isin(missed_ids)]
        
        if len(found) > 0:
            print(f"\n===== MISSED RECORDS FROM {source.upper()} =====")
            print(found.to_string(index=False))

Number of missed true-match IDs: 9
Missed IDs: {'S2-256254095', 'S3-70942743', 'S3-652513823', 'S3-440254853', 'S3-250737963', 'S3-317068674', 'S3-912418512', 'S3-183080822', 'S2-327309238'}

===== MISSED RECORDS FROM SOURCE2 =====
   entity_id                  business_name                                     business_address country
S2-327309238 డ్రీమ్ కన్‌స్ట్రక్షన్ లిమిటెడ్ H.NO 16-11-23/37/A, OSARAMBAGH, HYDERABAD, Telangana   India

===== MISSED RECORDS FROM SOURCE2 =====
   entity_id                 business_name                                                                                           business_address country
S2-256254095 Mr Dream Limited Construction NO 16-11-23/37/A , 2ND FLOOR, FLAT NO.207, SAGAR HOTEL BUILDING, OPP.RTA OFFICE, MO, OSARAMBAGH, Telangana   India

===== MISSED RECORDS FROM SOURCE3 =====
   entity_id                business_name                                             business_address country
S3-440254853 Mr Prabhav Business Services Plot 33

In [33]:
def address_block_keys(address):
    text = normalize_address(address)
    
    if not text:
        return []
    
    tokens = text.split()
    
    keys = set()
    
    # House/building number, when available
    numbers = re.findall(r"\b\d+[a-z]?\b", text)
    for num in numbers[:3]:
        keys.add("num_" + num)
    
    # Last few location tokens can be useful
    if len(tokens) >= 2:
        keys.add("loc_" + "_".join(tokens[-2:]))
    
    # First few meaningful tokens
    if len(tokens) >= 3:
        keys.add("start_" + "_".join(tokens[:3]))
    
    return list(keys)


for _, row in sample.iterrows():
    keys = address_block_keys(row["business_address"])
    print(row["entity_id"], "->", keys)

S1-925783039 -> ['num_1795', 'start_1795_westchester_dr', 'loc_point_nc']
S1-773889195 -> ['start_17560_ellis_rd', 'loc_tahlequah_ok', 'num_17560']
S1-377745466 -> ['start_1712_montebello_ave', 'loc_phoenix_az', 'num_1712']
S1-133037285 -> ['loc_dundalk_md', 'num_2100', 'start_2100_cameron_dr']
S1-755362802 -> ['num_797', 'start_797_lake_town', 'loc_west_bengal']
S1-851869949 -> ['start_oh_columbus_5559', 'loc_orville_ave', 'num_5559']
S1-785847572 -> ['start_2505_tower_1', 'num_1', 'loc_mumbai_maharashtra', 'num_2505']
S1-27541239 -> ['loc_nashville_tn', 'num_2007', 'num_1111', 'start_1111_church_st']
S1-629417405 -> ['num_337', 'start_337_oakland_ave', 'loc_city_in']
S1-22305073 -> ['start_294_meadowcreek_dr', 'num_2', 'loc_pewaukee_wi', 'num_294']
S1-504790211 -> ['start_11643_prosperity_rd', 'num_11643', 'loc_jordan_ut']
S1-564729135 -> ['start_h_no_16', 'num_23', 'num_16', 'num_11', 'loc_hyderabad_telangana']
S1-236312382 -> ['loc_chokio_mn', 'start_301_1st_st', 'num_301']
S1-8651

In [34]:
# Create address block keys for the 20 Source 1 sample records
sample["address_keys"] = sample["business_address"].apply(address_block_keys)

# Collect all address keys used by our sample
sample_address_keys = set(
    key
    for keys in sample["address_keys"]
    for key in keys
)

print("Unique address block keys:", len(sample_address_keys))

Unique address block keys: 68


In [35]:
address_candidate_ids = {entity_id: set() for entity_id in sample["entity_id"]}

# Map address block key -> Source 1 entity IDs
address_key_to_entities = {}

for _, row in sample.iterrows():
    for key in row["address_keys"]:
        address_key_to_entities.setdefault(key, []).append(row["entity_id"])


for source in ["source2", "source3"]:
    path = f"s3://{bucket}/dataset/train/train_{source}.tsv"
    
    print(f"Scanning {source}...")
    
    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_address"],
        chunksize=100_000
    ):
        chunk["address_keys"] = chunk["business_address"].apply(address_block_keys)

        for _, row in chunk.iterrows():
            for key in row["address_keys"]:
                if key in address_key_to_entities:
                    for s1_id in address_key_to_entities[key]:
                        address_candidate_ids[s1_id].add(row["entity_id"])
    
    print(f"Finished {source}")

Scanning source2...
Finished source2
Scanning source3...
Finished source3


In [36]:
missed_ids = {
    'S2-256254095',
    'S3-70942743',
    'S3-652513823',
    'S3-440254853',
    'S3-250737963',
    'S3-317068674',
    'S3-912418512',
    'S3-183080822',
    'S2-327309238'
}

for s1_id in sample["entity_id"]:
    recovered = [
        x for x in address_candidate_ids[s1_id]
        if x in missed_ids
    ]
    
    if recovered:
        print(s1_id, "->", recovered)

S1-377745466 -> ['S3-70942743']
S1-133037285 -> ['S3-183080822']
S1-755362802 -> ['S3-440254853']
S1-22305073 -> ['S3-317068674']
S1-564729135 -> ['S3-912418512', 'S2-256254095', 'S2-327309238']
S1-865131206 -> ['S3-652513823']
S1-82053739 -> ['S3-250737963']


In [37]:
# Combine name-block candidates + address-block candidates
combined_candidate_ids = {}

for s1_id in sample["entity_id"]:
    name_candidates = set(candidate_ids[s1_id])
    address_candidates = address_candidate_ids[s1_id]
    
    combined_candidate_ids[s1_id] = name_candidates | address_candidates

# Show candidate counts
for s1_id in sample["entity_id"]:
    print(
        s1_id,
        "->",
        len(combined_candidate_ids[s1_id]),
        "candidates"
    )

S1-925783039 -> 3565 candidates
S1-773889195 -> 33000 candidates
S1-377745466 -> 13771 candidates
S1-133037285 -> 10980 candidates
S1-755362802 -> 91747 candidates
S1-851869949 -> 11141 candidates
S1-785847572 -> 640038 candidates
S1-27541239 -> 14633 candidates
S1-629417405 -> 13857 candidates
S1-22305073 -> 453946 candidates
S1-504790211 -> 12209 candidates
S1-564729135 -> 257917 candidates
S1-236312382 -> 28846 candidates
S1-865131206 -> 22482 candidates
S1-626914593 -> 14647 candidates
S1-765734254 -> 39297 candidates
S1-82053739 -> 209901 candidates
S1-309349399 -> 3910 candidates
S1-221443075 -> 152640 candidates
S1-341506266 -> 40207 candidates


In [38]:
candidate_counts = [
    len(combined_candidate_ids[s1_id])
    for s1_id in sample["entity_id"]
]

print("Average:", sum(candidate_counts) / len(candidate_counts))
print("Minimum:", min(candidate_counts))
print("Maximum:", max(candidate_counts))

Average: 103436.7
Minimum: 3565
Maximum: 640038


In [39]:
from collections import Counter

# Count how many candidate records each address key produces
address_key_counts = Counter()

for s1_id in sample["entity_id"]:
    for key in sample.loc[
        sample["entity_id"] == s1_id, "address_keys"
    ].iloc[0]:
        address_key_counts[key] += len(address_candidate_ids[s1_id])

print("Sample address key counts:")
for key, count in address_key_counts.most_common(30):
    print(key, "->", count)

Sample address key counts:
start_2505_tower_1 -> 628158
num_1 -> 628158
loc_mumbai_maharashtra -> 628158
num_2505 -> 628158
start_294_meadowcreek_dr -> 432588
num_2 -> 432588
loc_pewaukee_wi -> 432588
num_294 -> 432588
start_h_no_16 -> 250068
num_23 -> 250068
num_16 -> 250068
num_11 -> 250068
loc_hyderabad_telangana -> 250068
num_303 -> 202795
num_5 -> 202795
start_303_3rd_fl -> 202795
loc_ahmedabad_gujarat -> 202795
loc_delhi_delhi -> 152274
num_777 -> 152274
start_house_no_777 -> 152274
num_797 -> 90697
start_797_lake_town -> 90697
loc_west_bengal -> 90697
start_secunderabad_hyderabad_plot -> 37838
num_53 -> 37838
loc_53_1 -> 37838
num_402 -> 37838
start_601_oleander_circle -> 32801
num_601 -> 32801
loc_city_va -> 32801


In [40]:
for s1_id in sample["entity_id"]:
    print("\n", s1_id)
    
    keys = sample.loc[
        sample["entity_id"] == s1_id, "address_keys"
    ].iloc[0]
    
    for key in keys:
        # Count candidates produced by this key
        matching = 0
        for entity_id in address_candidate_ids[s1_id]:
            matching += 1
        
        print(key)
    
    print("Total address candidates:", len(address_candidate_ids[s1_id]))


 S1-925783039
num_1795
start_1795_westchester_dr
loc_point_nc
Total address candidates: 2602

 S1-773889195
start_17560_ellis_rd
loc_tahlequah_ok
num_17560
Total address candidates: 359

 S1-377745466
start_1712_montebello_ave
loc_phoenix_az
num_1712
Total address candidates: 13533

 S1-133037285
loc_dundalk_md
num_2100
start_2100_cameron_dr
Total address candidates: 2524

 S1-755362802
num_797
start_797_lake_town
loc_west_bengal
Total address candidates: 90697

 S1-851869949
start_oh_columbus_5559
loc_orville_ave
num_5559
Total address candidates: 155

 S1-785847572
start_2505_tower_1
num_1
loc_mumbai_maharashtra
num_2505
Total address candidates: 628158

 S1-27541239
loc_nashville_tn
num_2007
num_1111
start_1111_church_st
Total address candidates: 13915

 S1-629417405
num_337
start_337_oakland_ave
loc_city_in
Total address candidates: 8185

 S1-22305073
start_294_meadowcreek_dr
num_2
loc_pewaukee_wi
num_294
Total address candidates: 432588

 S1-504790211
start_11643_prosperity_rd
nu

In [41]:
for s1_id in sample["entity_id"]:
    keys = sample.loc[
        sample["entity_id"] == s1_id, "address_keys"
    ].iloc[0]
    
    print("\n", s1_id)
    for key in keys:
        print(key, "->", address_key_frequency[key])


 S1-925783039


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:8                                                                                    │
│                                                                                                  │
│   5 │                                                                                            │
│   6 │   print("\n", s1_id)                                                                       │
│   7 │   for key in keys:                                                                         │
│ ❱ 8 │   │   print(key, "->", address_key_frequency[key])                                         │
│   9                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'address_key_frequency' is not defined

In [42]:
def selective_address_keys(address):
    text = normalize_address(address)
    
    if not text:
        return []
    
    tokens = text.split()
    keys = set()
    
    # Strong key: first 3 address tokens
    if len(tokens) >= 3:
        keys.add("start_" + "_".join(tokens[:3]))
    
    # Strong location key: last 2 tokens
    if len(tokens) >= 2:
        keys.add("loc_" + "_".join(tokens[-2:]))
    
    return list(keys)


# Test the new keys
for _, row in sample.iterrows():
    print(
        row["entity_id"],
        "->",
        selective_address_keys(row["business_address"])
    )

S1-925783039 -> ['loc_point_nc', 'start_1795_westchester_dr']
S1-773889195 -> ['loc_tahlequah_ok', 'start_17560_ellis_rd']
S1-377745466 -> ['start_1712_montebello_ave', 'loc_phoenix_az']
S1-133037285 -> ['start_2100_cameron_dr', 'loc_dundalk_md']
S1-755362802 -> ['start_797_lake_town', 'loc_west_bengal']
S1-851869949 -> ['start_oh_columbus_5559', 'loc_orville_ave']
S1-785847572 -> ['start_2505_tower_1', 'loc_mumbai_maharashtra']
S1-27541239 -> ['loc_nashville_tn', 'start_1111_church_st']
S1-629417405 -> ['start_337_oakland_ave', 'loc_city_in']
S1-22305073 -> ['loc_pewaukee_wi', 'start_294_meadowcreek_dr']
S1-504790211 -> ['start_11643_prosperity_rd', 'loc_jordan_ut']
S1-564729135 -> ['start_h_no_16', 'loc_hyderabad_telangana']
S1-236312382 -> ['loc_chokio_mn', 'start_301_1st_st']
S1-865131206 -> ['loc_bridgeport_ct', 'start_66_edgewood_st']
S1-626914593 -> ['start_unit_building_3030', 'loc_middle_river']
S1-765734254 -> ['start_601_oleander_circle', 'loc_city_va']
S1-82053739 -> ['star

In [43]:
for s1_id in sample["entity_id"]:
    recovered = [
        x for x in selective_address_candidate_ids[s1_id]
        if x in missed_ids
    ]

    print(s1_id, "->", recovered)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│   1 for s1_id in sample["entity_id"]:                                                            │
│   2 │   recovered = [                                                                            │
│ ❱ 3 │   │   x for x in selective_address_candidate_ids[s1_id]                                    │
│   4 │   │   if x in missed_ids                                                                   │
│   5 │   ]                                                                                        │
│   6                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'selective_address_candidate_ids' is not defined

In [44]:
selective_address_candidate_ids = {
    entity_id: set() for entity_id in sample["entity_id"]
}

selective_key_to_entities = {}

# Create mapping from address key -> Source 1 entities
for _, row in sample.iterrows():
    for key in selective_address_keys(row["business_address"]):
        selective_key_to_entities.setdefault(key, []).append(
            row["entity_id"]
        )

# Scan Source 2 and Source 3
for source in ["source2", "source3"]:
    path = f"s3://{bucket}/dataset/train/train_{source}.tsv"

    print("Scanning", source)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_address"],
        chunksize=100_000
    ):
        chunk["address_keys"] = chunk["business_address"].apply(
            selective_address_keys
        )

        for _, row in chunk.iterrows():
            for key in row["address_keys"]:
                if key in selective_key_to_entities:
                    for s1_id in selective_key_to_entities[key]:
                        selective_address_candidate_ids[s1_id].add(
                            row["entity_id"]
                        )

    print("Finished", source)

print("Done!")

Scanning source2
Finished source2
Scanning source3
Finished source3
Done!


In [45]:
for s1_id in sample["entity_id"]:
    recovered = [
        x for x in selective_address_candidate_ids[s1_id]
        if x in missed_ids
    ]

    if recovered:
        print(s1_id, "->", recovered)

S1-133037285 -> ['S3-183080822']
S1-22305073 -> ['S3-317068674']
S1-564729135 -> ['S2-327309238']
S1-82053739 -> ['S3-250737963']


In [46]:
selective_counts = [
    len(selective_address_candidate_ids[s1_id])
    for s1_id in sample["entity_id"]
]

print("Average:", sum(selective_counts) / len(selective_counts))
print("Minimum:", min(selective_counts))
print("Maximum:", max(selective_counts))

Average: 22034.8
Minimum: 2
Maximum: 150661


In [47]:
def frequency_aware_address_keys(address, max_frequency=5000):
    text = normalize_address(address)

    if not text:
        return []

    tokens = text.split()
    keys = set()

    # Specific address-start key
    if len(tokens) >= 3:
        key = "start_" + "_".join(tokens[:3])
        if address_key_frequency.get(key, 0) <= max_frequency:
            keys.add(key)

    # Specific location key
    if len(tokens) >= 2:
        key = "loc_" + "_".join(tokens[-2:])
        if address_key_frequency.get(key, 0) <= max_frequency:
            keys.add(key)

    return list(keys)

In [48]:
for _, row in sample.iterrows():
    print(
        row["entity_id"],
        "->",
        frequency_aware_address_keys(row["business_address"])
    )

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:5                                                                                    │
│                                                                                                  │
│   2 │   print(                                                                                   │
│   3 │   │   row["entity_id"],                                                                    │
│   4 │   │   "->",                                                                                │
│ ❱ 5 │   │   frequency_aware_address_keys(row["business_address"])                                │
│   6 │   )                                                                                        │
│   7                                                                                              │
│                                                                                                  │
│ in frequency_aware_address_keys:13                                                               │
│                                                                                                  │
│   10 │   # Specific address-start key                                                            │
│   11 │   if len(tokens) >= 3:                                                                    │
│   12 │   │   key = "start_" + "_".join(tokens[:3])                                               │
│ ❱ 13 │   │   if address_key_frequency.get(key, 0) <= max_frequency:                              │
│   14 │   │   │   keys.add(key)                                                                   │
│   15 │                                                                                           │
│   16 │   # Specific location key                                                                 │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'address_key_frequency' is not defined

In [49]:
from collections import Counter

address_key_frequency = Counter()

for source in ["source2", "source3"]:
    path = f"s3://{bucket}/dataset/train/train_{source}.tsv"

    print("Scanning", source)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["business_address"],
        chunksize=100_000
    ):
        for address in chunk["business_address"]:
            keys = address_block_keys(address)

            for key in keys:
                address_key_frequency[key] += 1

    print("Finished", source)

print("Total unique address keys:", len(address_key_frequency))

Scanning source2
Finished source2
Scanning source3
Finished source3
Total unique address keys: 6132683


In [50]:
for key in [
    "num_1",
    "num_2",
    "num_23",
    "num_797",
    "start_17560_ellis_rd",
    "start_1712_montebello_ave"
]:
    print(key, "->", address_key_frequency.get(key, 0))

num_1 -> 546698
num_2 -> 429101
num_23 -> 51887
num_797 -> 1280
start_17560_ellis_rd -> 4
start_1712_montebello_ave -> 2


In [51]:
def frequency_aware_address_keys(address, max_frequency=5000):
    text = normalize_address(address)

    if not text:
        return []

    tokens = text.split()
    keys = set()

    # Specific address-start key
    if len(tokens) >= 3:
        key = "start_" + "_".join(tokens[:3])

        if address_key_frequency.get(key, 0) <= max_frequency:
            keys.add(key)

    # Specific location key
    if len(tokens) >= 2:
        key = "loc_" + "_".join(tokens[-2:])

        if address_key_frequency.get(key, 0) <= max_frequency:
            keys.add(key)

    return list(keys)

In [52]:
for _, row in sample.iterrows():
    print(
        row["entity_id"],
        "->",
        frequency_aware_address_keys(row["business_address"])
    )

S1-925783039 -> ['loc_point_nc', 'start_1795_westchester_dr']
S1-773889195 -> ['loc_tahlequah_ok', 'start_17560_ellis_rd']
S1-377745466 -> ['start_1712_montebello_ave']
S1-133037285 -> ['start_2100_cameron_dr', 'loc_dundalk_md']
S1-755362802 -> ['start_797_lake_town']
S1-851869949 -> ['start_oh_columbus_5559', 'loc_orville_ave']
S1-785847572 -> ['start_2505_tower_1']
S1-27541239 -> ['start_1111_church_st']
S1-629417405 -> ['start_337_oakland_ave', 'loc_city_in']
S1-22305073 -> ['loc_pewaukee_wi', 'start_294_meadowcreek_dr']
S1-504790211 -> ['start_11643_prosperity_rd', 'loc_jordan_ut']
S1-564729135 -> ['start_h_no_16']
S1-236312382 -> ['loc_chokio_mn', 'start_301_1st_st']
S1-865131206 -> ['loc_bridgeport_ct', 'start_66_edgewood_st']
S1-626914593 -> ['start_unit_building_3030', 'loc_middle_river']
S1-765734254 -> ['start_601_oleander_circle']
S1-82053739 -> ['start_303_3rd_fl']
S1-309349399 -> ['loc_reliance_st', 'start_charlotte_nc_833']
S1-221443075 -> ['start_house_no_777']
S1-341506

In [53]:
for s1_id in sample["entity_id"]:
    recovered = [
        x for x in frequency_address_candidate_ids[s1_id]
        if x in missed_ids
    ]

    if recovered:
        print(s1_id, "->", recovered)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│   1 for s1_id in sample["entity_id"]:                                                            │
│   2 │   recovered = [                                                                            │
│ ❱ 3 │   │   x for x in frequency_address_candidate_ids[s1_id]                                    │
│   4 │   │   if x in missed_ids                                                                   │
│   5 │   ]                                                                                        │
│   6                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'frequency_address_candidate_ids' is not defined

In [54]:
frequency_address_candidate_ids = {
    entity_id: set() for entity_id in sample["entity_id"]
}

frequency_key_to_entities = {}

# Map each useful address key to Source 1 entities
for _, row in sample.iterrows():
    for key in frequency_aware_address_keys(row["business_address"]):
        frequency_key_to_entities.setdefault(key, []).append(
            row["entity_id"]
        )


# Scan Source 2 and Source 3
for source in ["source2", "source3"]:
    path = f"s3://{bucket}/dataset/train/train_{source}.tsv"

    print("Scanning", source)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_address"],
        chunksize=100_000
    ):
        chunk["address_keys"] = chunk["business_address"].apply(
            frequency_aware_address_keys
        )

        for _, row in chunk.iterrows():
            for key in row["address_keys"]:
                if key in frequency_key_to_entities:
                    for s1_id in frequency_key_to_entities[key]:
                        frequency_address_candidate_ids[s1_id].add(
                            row["entity_id"]
                        )

    print("Finished", source)

print("Done!")

Scanning source2
Finished source2
Scanning source3
Finished source3
Done!


In [55]:
for s1_id in sample["entity_id"]:
    recovered = [
        x for x in frequency_address_candidate_ids[s1_id]
        if x in missed_ids
    ]

    if recovered:
        print(s1_id, "->", recovered)

S1-133037285 -> ['S3-183080822']
S1-22305073 -> ['S3-317068674']
S1-564729135 -> ['S2-327309238']
S1-82053739 -> ['S3-250737963']


In [56]:
frequency_counts = [
    len(frequency_address_candidate_ids[s1_id])
    for s1_id in sample["entity_id"]
]

print("Average:", sum(frequency_counts) / len(frequency_counts))
print("Minimum:", min(frequency_counts))
print("Maximum:", max(frequency_counts))

Average: 614.05
Minimum: 0
Maximum: 4173


In [58]:
def frequency_aware_address_keys_v2(address, max_frequency=5000):
    text = normalize_address(address)
    if not text:
        return []

    tokens = text.split()
    keys = set()

    # Consecutive 3-word address chunks
    for i in range(len(tokens) - 2):
        key = "tri_" + "_".join(tokens[i:i+3])
        freq = address_key_frequency.get(key, 0)

        if 0 < freq <= max_frequency:
            keys.add(key)

    # Keep useful start/end keys too
    if len(tokens) >= 3:
        key = "start_" + "_".join(tokens[:3])
        if address_key_frequency.get(key, 0) <= max_frequency:
            keys.add(key)

    if len(tokens) >= 2:
        key = "loc_" + "_".join(tokens[-2:])
        if address_key_frequency.get(key, 0) <= max_frequency:
            keys.add(key)

    return list(keys)

In [59]:
for s1_id in sample["entity_id"]:
    keys = frequency_aware_address_keys_v2(
        sample.loc[sample["entity_id"] == s1_id, "business_address"].iloc[0]
    )
    
    if s1_id in [
        "S1-377745466",
        "S1-755362802",
        "S1-564729135",
        "S1-865131206",
        "S1-82053739"
    ]:
        print(s1_id, "->", keys)

S1-377745466 -> ['start_1712_montebello_ave']
S1-755362802 -> ['start_797_lake_town']
S1-564729135 -> ['start_h_no_16']
S1-865131206 -> ['loc_bridgeport_ct', 'start_66_edgewood_st']
S1-82053739 -> ['start_303_3rd_fl']


In [60]:
from rapidfuzz.fuzz import ratio, token_set_ratio, WRatio

def similarity_score(s1_name, s1_address, s2_name, s2_address):
    name1 = normalize_name(s1_name)
    name2 = normalize_name(s2_name)

    addr1 = normalize_address(s1_address)
    addr2 = normalize_address(s2_address)

    # Name similarities
    name_ratio = ratio(name1, name2)
    name_token = token_set_ratio(name1, name2)
    name_wratio = WRatio(name1, name2)

    # Address similarities
    addr_ratio = ratio(addr1, addr2)
    addr_token = token_set_ratio(addr1, addr2)

    # Exact normalized matches
    exact_name = int(name1 != "" and name1 == name2)
    exact_address = int(addr1 != "" and addr1 == addr2)

    # Combined score
    score = (
        0.45 * name_wratio +
        0.25 * name_token +
        0.20 * addr_token +
        0.10 * addr_ratio
    )

    # Strong exact evidence
    if exact_name:
        score += 15

    if exact_address:
        score += 15

    return {
        "name_ratio": name_ratio,
        "name_token": name_token,
        "name_wratio": name_wratio,
        "addr_ratio": addr_ratio,
        "addr_token": addr_token,
        "exact_name": exact_name,
        "exact_address": exact_address,
        "score": score
    }

In [61]:
print(similarity_score(
    "Dream Construction Limited",
    "H.NO 16-11-23/37/A, OSARAMBAGH, HYDERABAD, Telangana",
    "Mr Dream Limited Construction",
    "H.NO 16-11-23/37/A, OSARAMBAGH, HYDERABAD, Telangana"
))

{'name_ratio': 92.3076923076923, 'name_token': 100.0, 'name_wratio': 95.0, 'addr_ratio': 100.0, 'addr_token': 100.0, 'exact_name': 0, 'exact_address': 1, 'score': 112.75}


In [62]:
print(similarity_score(
    "Dream Construction Limited",
    "H.NO 16-11-23/37/A, OSARAMBAGH, HYDERABAD, Telangana",
    "Orelee's Barbershop",
    "1795 Westchester Dr, Point, NC"
))

{'name_ratio': 32.432432432432435, 'name_token': 27.02702702702703, 'name_wratio': 32.432432432432435, 'addr_ratio': 28.57142857142857, 'addr_token': 33.76623376623377, 'exact_name': 0, 'exact_address': 0, 'score': 30.961740961740965}


In [63]:
thresholds = [50, 60, 70, 80, 90, 100]

for t in thresholds:
    print(f"Threshold {t}:")
    print("  Scores >= threshold are considered matches")

Threshold 50:
  Scores >= threshold are considered matches
Threshold 60:
  Scores >= threshold are considered matches
Threshold 70:
  Scores >= threshold are considered matches
Threshold 80:
  Scores >= threshold are considered matches
Threshold 90:
  Scores >= threshold are considered matches
Threshold 100:
  Scores >= threshold are considered matches


In [64]:
# Build a lookup of the 20 sample Source 1 records
sample_lookup = sample.set_index("entity_id").to_dict("index")

# Collect all candidate IDs from our two existing blockers
all_candidate_ids = {}

for s1_id in sample["entity_id"]:
    ids = set()

    # Name candidates
    ids.update(name_candidate_ids.get(s1_id, []))

    # Frequency-aware address candidates
    ids.update(frequency_address_candidate_ids.get(s1_id, []))

    all_candidate_ids[s1_id] = ids

print("Candidate sets prepared.")

for s1_id, ids in all_candidate_ids.items():
    print(s1_id, "->", len(ids))

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:11                                                                                   │
│                                                                                                  │
│    8 │   ids = set()                                                                             │
│    9 │                                                                                           │
│   10 │   # Name candidates                                                                       │
│ ❱ 11 │   ids.update(name_candidate_ids.get(s1_id, []))                                           │
│   12 │                                                                                           │
│   13 │   # Frequency-aware address candidates                                                    │
│   14 │   ids.update(frequency_address_candidate_ids.get(s1_id, []))                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'name_candidate_ids' is not defined

In [65]:
# Check which candidate dictionaries currently exist
print("frequency_address_candidate_ids:", 
      "frequency_address_candidate_ids" in globals())

print("Available candidate-related variables:")
for name in globals():
    if "candidate" in name.lower():
        print(" -", name)

frequency_address_candidate_ids: True
Available candidate-related variables:
 - candidate_ids
 - candidates
 - address_candidate_ids
 - combined_candidate_ids
 - name_candidates
 - address_candidates
 - candidate_counts
 - selective_address_candidate_ids
 - frequency_address_candidate_ids
 - all_candidate_ids


In [66]:
# Combine the existing name + frequency-aware address candidates
all_candidate_ids = {}

for s1_id in sample["entity_id"]:
    ids = set()

    # Existing name candidates
    ids.update(name_candidates.get(s1_id, []))

    # Existing frequency-aware address candidates
    ids.update(frequency_address_candidate_ids.get(s1_id, []))

    all_candidate_ids[s1_id] = ids

print("Candidate sets prepared.")

for s1_id, ids in all_candidate_ids.items():
    print(s1_id, "->", len(ids))

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:8                                                                                    │
│                                                                                                  │
│    5 │   ids = set()                                                                             │
│    6 │                                                                                           │
│    7 │   # Existing name candidates                                                              │
│ ❱  8 │   ids.update(name_candidates.get(s1_id, []))                                              │
│    9 │                                                                                           │
│   10 │   # Existing frequency-aware address candidates                                           │
│   11 │   ids.update(frequency_address_candidate_ids.get(s1_id, []))                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
AttributeError: 'set' object has no attribute 'get'

In [68]:
# Use the already-created combined candidates
all_candidate_ids = combined_candidate_ids

print("Candidate sets ready.")

for s1_id in sample["entity_id"]:
    print(s1_id, "->", len(all_candidate_ids.get(s1_id, [])))

Candidate sets ready.
S1-925783039 -> 3565
S1-773889195 -> 33000
S1-377745466 -> 13771
S1-133037285 -> 10980
S1-755362802 -> 91747
S1-851869949 -> 11141
S1-785847572 -> 640038
S1-27541239 -> 14633
S1-629417405 -> 13857
S1-22305073 -> 453946
S1-504790211 -> 12209
S1-564729135 -> 257917
S1-236312382 -> 28846
S1-865131206 -> 22482
S1-626914593 -> 14647
S1-765734254 -> 39297
S1-82053739 -> 209901
S1-309349399 -> 3910
S1-221443075 -> 152640
S1-341506266 -> 40207


In [1]:
print("Frequency-aware candidate counts:")
for s1_id in sample["entity_id"]:
    print(
        s1_id,
        "->",
        len(frequency_address_candidate_ids.get(s1_id, []))
    )

Frequency-aware candidate counts:


NameError: name 'sample' is not defined

In [2]:
import re
import pandas as pd
from unidecode import unidecode
from collections import Counter

bucket = "sagemaker-eu-north-1-152025327817"

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = unidecode(text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_address(address):
    text = normalize_text(address)

    replacements = {
        "street": "st",
        "road": "rd",
        "avenue": "ave",
        "boulevard": "blvd",
        "drive": "dr",
        "lane": "ln",
        "highway": "hwy",
        "apartment": "apt",
        "suite": "ste",
        "floor": "fl"
    }

    for full, short in replacements.items():
        text = re.sub(rf"\b{full}\b", short, text)

    return re.sub(r"\s+", " ", text).strip()

def normalize_name(name):
    text = normalize_text(name)

    suffixes = [
        "private limited", "pvt ltd", "pvt",
        "limited", "ltd", "llc",
        "incorporated", "inc",
        "corporation", "corp",
        "company", "co", "pc"
    ]

    for suffix in suffixes:
        text = re.sub(rf"\b{suffix}\b", " ", text)

    return re.sub(r"\s+", " ", text).strip()

print("Setup complete")

Setup complete


In [4]:
sample_ids = [
    "S1-925783039",
    "S1-773889195",
    "S1-377745466",
    "S1-133037285",
    "S1-755362802",
    "S1-851869949",
    "S1-785847572",
    "S1-27541239",
    "S1-629417405",
    "S1-22305073",
    "S1-504790211",
    "S1-564729135",
    "S1-236312382",
    "S1-865131206",
    "S1-626914593",
    "S1-765734254",
    "S1-82053739",
    "S1-309349399",
    "S1-221443075",
    "S1-341506266"
]

sample = pd.read_csv(
    f"s3://{bucket}/dataset/train/train_source1.tsv",
    sep="\t",
    usecols=["entity_id", "business_name", "business_address", "country"]
)

sample = sample[sample["entity_id"].isin(sample_ids)].copy()

print("Sample rows:", len(sample))

Sample rows: 20


In [5]:
address_key_frequency = Counter()

def frequency_aware_address_keys(address, max_frequency=5000):
    text = normalize_address(address)

    if not text:
        return []

    tokens = text.split()
    keys = set()

    if len(tokens) >= 3:
        key = "start_" + "_".join(tokens[:3])
        if address_key_frequency.get(key, 0) <= max_frequency:
            keys.add(key)

    if len(tokens) >= 2:
        key = "loc_" + "_".join(tokens[-2:])
        if address_key_frequency.get(key, 0) <= max_frequency:
            keys.add(key)

    return list(keys)


for source_file in [
    "train_source2.tsv",
    "train_source3.tsv"
]:
    print("Scanning", source_file)

    for chunk in pd.read_csv(
        f"s3://{bucket}/dataset/train/{source_file}",
        sep="\t",
        usecols=["business_address"],
        chunksize=100000
    ):
        for address in chunk["business_address"]:
            text = normalize_address(address)

            if not text:
                continue

            tokens = text.split()

            if len(tokens) >= 3:
                key = "start_" + "_".join(tokens[:3])
                address_key_frequency[key] += 1

            if len(tokens) >= 2:
                key = "loc_" + "_".join(tokens[-2:])
                address_key_frequency[key] += 1

print("Address index built.")
print("Unique address keys:", len(address_key_frequency))

Scanning train_source2.tsv
Scanning train_source3.tsv


KeyboardInterrupt: 

In [6]:
def normalize_address_fast(series):
    s = series.fillna("").astype(str).str.lower()

    # Transliteration
    s = s.map(unidecode)

    # Remove punctuation
    s = s.str.replace(r"[^a-z0-9\s]", " ", regex=True)

    # Common address abbreviations
    replacements = {
        r"\bstreet\b": "st",
        r"\broad\b": "rd",
        r"\bavenue\b": "ave",
        r"\bboulevard\b": "blvd",
        r"\bdrive\b": "dr",
        r"\blane\b": "ln",
        r"\bhighway\b": "hwy",
        r"\bapartment\b": "apt",
        r"\bsuite\b": "ste",
        r"\bfloor\b": "fl",
    }

    for pattern, replacement in replacements.items():
        s = s.str.replace(pattern, replacement, regex=True)

    return s.str.replace(r"\s+", " ", regex=True).str.strip()

print("Fast address normalizer ready")

Fast address normalizer ready


In [7]:
test_addresses = pd.Series([
    "H.NO 16-11-23/37/A, OSARAMBAGH, HYDERABAD, Telangana",
    "66 Edgewood Street, Bridgeport, Connecticut",
    "1712 Montebello Avenue"
])

print(normalize_address_fast(test_addresses).tolist())

['h no 16 11 23 37 a osarambagh hyderabad telangana', '66 edgewood st bridgeport connecticut', '1712 montebello ave']


In [ ]:
from collections import Counter

address_key_frequency = Counter()

for source_file in [
    "train_source2.tsv",
    "train_source3.tsv"
]:
    print("Scanning", source_file)

    for chunk in pd.read_csv(
        f"s3://{bucket}/dataset/train/{source_file}",
        sep="\t",
        usecols=["business_address"],
        chunksize=100000
    ):
        addresses = normalize_address_fast(chunk["business_address"])

        for text in addresses:
            if not text:
                continue

            tokens = text.split()

            if len(tokens) >= 3:
                address_key_frequency[
                    "start_" + "_".join(tokens[:3])
                ] += 1

            if len(tokens) >= 2:
                address_key_frequency[
                    "loc_" + "_".join(tokens[-2:])
                ] += 1

print("Address index built.")
print("Unique address keys:", len(address_key_frequency))

In [12]:
print("TEST")


TEST


In [13]:
from rapidfuzz.fuzz import ratio, token_set_ratio, WRatio

def calculate_match_features(
    s1_name,
    s1_address,
    candidate_name,
    candidate_address
):
    name1 = normalize_name(s1_name)
    name2 = normalize_name(candidate_name)

    addr1 = normalize_address(s1_address)
    addr2 = normalize_address(candidate_address)

    name_w = WRatio(name1, name2)
    name_token = token_set_ratio(name1, name2)

    addr_ratio = ratio(addr1, addr2)
    addr_token = token_set_ratio(addr1, addr2)

    exact_name = name1 != "" and name1 == name2
    exact_address = addr1 != "" and addr1 == addr2

    score = (
        0.45 * name_w +
        0.25 * name_token +
        0.20 * addr_token +
        0.10 * addr_ratio
    )

    return {
        "name_w": name_w,
        "name_token": name_token,
        "addr_ratio": addr_ratio,
        "addr_token": addr_token,
        "exact_name": exact_name,
        "exact_address": exact_address,
        "score": score
    }

print("Scoring function ready")

Scoring function ready


In [14]:
true_example = calculate_match_features(
    "Dream Construction Limited",
    "H.NO 16-11-23/37/A, OSARAMBAGH, HYDERABAD, Telangana",
    "Mr Dream Limited Construction",
    "H.NO 16-11-23/37/A, OSARAMBAGH, HYDERABAD, Telangana"
)

false_example = calculate_match_features(
    "Dream Construction Limited",
    "H.NO 16-11-23/37/A, OSARAMBAGH, HYDERABAD, Telangana",
    "Orelee's Barbershop",
    "1795 Westchester Dr, Point, NC"
)

print("TRUE MATCH")
print(true_example)

print("\nFALSE MATCH")
print(false_example)

TRUE MATCH
{'name_w': 95.0, 'name_token': 100.0, 'addr_ratio': 100.0, 'addr_token': 100.0, 'exact_name': False, 'exact_address': True, 'score': 97.75}

FALSE MATCH
{'name_w': 32.432432432432435, 'name_token': 27.02702702702703, 'addr_ratio': 28.57142857142857, 'addr_token': 33.76623376623377, 'exact_name': False, 'exact_address': False, 'score': 30.961740961740965}


In [15]:
def is_match(features):
    name_w = features["name_w"]
    name_token = features["name_token"]
    addr_ratio = features["addr_ratio"]
    addr_token = features["addr_token"]
    exact_name = features["exact_name"]
    exact_address = features["exact_address"]

    # Exact normalized name + compatible address
    if exact_name and (exact_address or addr_token >= 60):
        return True

    # Exact address + strong name similarity
    if exact_address and name_w >= 70:
        return True

    # Strong agreement on both fields
    if name_w >= 85 and addr_token >= 75:
        return True

    # Very strong name + reasonably compatible address
    if name_w >= 92 and addr_token >= 55:
        return True

    return False


print("Matching rule ready")

Matching rule ready


In [16]:
print("True example:", is_match(true_example))
print("False example:", is_match(false_example))

True example: True
False example: False


In [18]:
# Normalize the sample columns safely, row by row

sample["name_norm"] = sample["business_name"].apply(normalize_name)
sample["address_norm"] = sample["business_address"].apply(normalize_address)

print("Normalization completed successfully.")

display(sample[[
    "entity_id",
    "business_name",
    "business_address",
    "country",
    "name_norm",
    "address_norm"
]].head(10))

Normalization completed successfully.


,entity_id,business_name,business_address,country,name_norm,address_norm
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,1795 westchester dr high point nc
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,17560 ellis rd tahlequah ok
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b retail,1712 montebello ave phoenix az
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,christ chapel,2100 cameron dr unit apt g dundalk md
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,prabhav business center,797 lake town block a kolkata howrah west bengal
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US,custom wealth services,oh columbus 5559 orville ave
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India,consulting nyasa nursing,2505 tower 1 oakwood runwal greens mulund gore...
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US,nexus anchor rain,1111 church st unit 2007 nashville tn
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US,moore bitwise,337 oakland ave michigan city in
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US,dermatology green medicine,294 meadowcreek dr unit unit 2 village of pewa...


In [19]:
# Create compact blocking keys for the sample entities

def make_block_keys(name, address, country):
    keys = set()

    name = str(name).strip()
    address = str(address).strip()
    country = str(country).strip().lower()

    # Name-based keys
    name_tokens = name.split()

    if name:
        keys.add(f"{country}|name|{name}")

    if len(name_tokens) >= 2:
        keys.add(f"{country}|name2|{' '.join(name_tokens[:2])}")

    if len(name_tokens) >= 1:
        keys.add(f"{country}|name1|{name_tokens[0]}")

    # Address number + location
    address_tokens = address.split()

    numbers = re.findall(r"\b\d+[a-z]?\b", address)

    for number in numbers[:2]:
        keys.add(f"{country}|num|{number}")

    if len(address_tokens) >= 2:
        keys.add(
            f"{country}|addr_last2|{' '.join(address_tokens[-2:])}"
        )

    if len(address_tokens) >= 3:
        keys.add(
            f"{country}|addr_first3|{' '.join(address_tokens[:3])}"
        )

    return keys


sample["block_keys"] = sample.apply(
    lambda row: make_block_keys(
        row["name_norm"],
        row["address_norm"],
        row["country"]
    ),
    axis=1
)

print("Blocking keys created successfully.")
print("Example keys for first entity:")
for key in sorted(sample.iloc[0]["block_keys"]):
    print(key)

Blocking keys created successfully.
Example keys for first entity:
us|addr_first3|1795 westchester dr
us|addr_last2|point nc
us|name1|orelee
us|name2|orelee s
us|name|orelee s barbershop
us|num|1795


In [21]:
# Recreate ground_truth safely and verify the sample IDs

import pandas as pd

bucket = "sagemaker-eu-north-1-152025327817"

# Load only the ground-truth file
ground_truth = pd.read_csv(
    f"s3://{bucket}/dataset/train/train_ground_truth.tsv",
    sep="\t"
)

# Make sure sample_ids exists
if "sample_ids" not in globals():
    sample_ids = [
        "S1-925783039",
        "S1-773889195",
        "S1-377745466",
        "S1-133037285",
        "S1-755362802",
        "S1-851869949",
        "S1-785847572",
        "S1-27541239",
        "S1-629417405",
        "S1-22305073",
        "S1-504790211",
        "S1-564729135",
        "S1-236312382",
        "S1-865131206",
        "S1-626914593",
        "S1-765734254",
        "S1-82053739",
        "S1-309349399",
        "S1-221443075",
        "S1-341506266"
    ]

sample_gt = ground_truth[
    ground_truth["source1_entity_id"].isin(sample_ids)
].copy()

def parse_matches(value):
    if pd.isna(value) or str(value).strip() == "":
        return set()
    return {
        x.strip()
        for x in str(value).split(",")
        if x.strip()
    }

sample_gt["true_matches"] = sample_gt["matched_entity_ids"].apply(parse_matches)

print("Ground truth loaded successfully.")
print("Ground truth rows:", len(ground_truth))
print("Sample ground-truth rows:", len(sample_gt))
print("Sample IDs found:", len(sample_gt))

display(
    sample_gt[
        ["source1_entity_id", "matched_entity_ids"]
    ].head(20)
)

Ground truth loaded successfully.
Ground truth rows: 2206821
Sample ground-truth rows: 20
Sample IDs found: 20


,source1_entity_id,matched_entity_ids
91073,S1-236312382,"S2-734049056,S2-69010626,S3-591515133"
137307,S1-629417405,"S2-522538408,S2-928426462,S3-613837325,S3-6069..."
169210,S1-309349399,NaN
357062,S1-773889195,"S2-970528089,S2-374107005,S3-189140669,S3-1614..."
409706,S1-27541239,"S2-199341916,S2-85756610,S3-893131105,S3-73975..."
444731,S1-377745466,"S2-203037501,S3-70942743,S3-402918963,S3-83617..."
505834,S1-851869949,"S2-915265864,S2-801813753,S2-961342067,S3-9498..."
557004,S1-133037285,"S2-407207105,S2-103437512,S3-476003339,S3-3314..."
574234,S1-626914593,"S2-563003232,S3-594660686"
664353,S1-504790211,"S2-830972710,S2-482219248,S2-340882825,S2-3842..."


In [22]:
# Check that every true match shares at least one blocking key
# with its Source 1 entity.

def get_block_keys_for_record(name, address, country):
    name_norm = normalize_name(name)
    address_norm = normalize_address(address)

    return make_block_keys(
        name_norm,
        address_norm,
        country
    )


# Build a lookup of the blocking keys for our 20 Source 1 records
s1_keys = {
    row["entity_id"]: row["block_keys"]
    for _, row in sample.iterrows()
}

print("Sample Source 1 blocking keys ready.")
print("Entities checked:", len(s1_keys))

# Show the number of keys for each entity
for entity_id, keys in s1_keys.items():
    print(entity_id, "->", len(keys), "keys")

Sample Source 1 blocking keys ready.
Entities checked: 20
S1-925783039 -> 6 keys
S1-773889195 -> 6 keys
S1-377745466 -> 6 keys
S1-133037285 -> 6 keys
S1-755362802 -> 6 keys
S1-851869949 -> 6 keys
S1-785847572 -> 7 keys
S1-27541239 -> 7 keys
S1-629417405 -> 6 keys
S1-22305073 -> 7 keys
S1-504790211 -> 6 keys
S1-564729135 -> 7 keys
S1-236312382 -> 6 keys
S1-865131206 -> 5 keys
S1-626914593 -> 7 keys
S1-765734254 -> 6 keys
S1-82053739 -> 7 keys
S1-309349399 -> 6 keys
S1-221443075 -> 6 keys
S1-341506266 -> 7 keys


In [23]:
# Test blocking-key recall against the 20 known Source 1 entities

sample_key_to_s1 = {}

for entity_id, keys in s1_keys.items():
    for key in keys:
        sample_key_to_s1.setdefault(key, set()).add(entity_id)

recovered_matches = {sid: set() for sid in sample_ids}

for source_name in ["source2", "source3"]:
    path = f"s3://{bucket}/dataset/train/train_{source_name}.tsv"

    print(f"Scanning {source_name}...")

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=100000,
        dtype=str
    ):
        chunk["name_norm"] = chunk["business_name"].apply(normalize_name)
        chunk["address_norm"] = chunk["business_address"].apply(normalize_address)

        for _, row in chunk.iterrows():
            keys = make_block_keys(
                row["name_norm"],
                row["address_norm"],
                row["country"]
            )

            for key in keys:
                matching_s1 = sample_key_to_s1.get(key)

                if matching_s1:
                    for s1_id in matching_s1:
                        recovered_matches[s1_id].add(row["entity_id"])

    print(f"{source_name} scan complete.")

print("\nBlocking recall results:")
print("-" * 70)

total_true = 0
total_recovered = 0

for _, row in sample_gt.iterrows():
    s1_id = row["source1_entity_id"]
    true_ids = row["true_matches"]
    recovered = recovered_matches[s1_id]

    found = true_ids & recovered
    missed = true_ids - recovered

    total_true += len(true_ids)
    total_recovered += len(found)

    print(f"\n{s1_id}")
    print(f"True matches: {len(true_ids)}")
    print(f"Recovered:    {len(found)}")
    print(f"Missed:       {len(missed)}")

    if missed:
        print("Missed IDs:", sorted(missed))

print("\n" + "=" * 70)
print(f"TOTAL TRUE MATCHES: {total_true}")
print(f"TOTAL RECOVERED:    {total_recovered}")

if total_true > 0:
    print(f"RECALL:             {total_recovered / total_true:.2%}")

Scanning source2...
source2 scan complete.
Scanning source3...


KeyboardInterrupt: 

In [24]:
# Prepare a fast lookup for the 20 sample entities

sample_key_to_s1 = {}

for entity_id, keys in s1_keys.items():
    for key in keys:
        if key not in sample_key_to_s1:
            sample_key_to_s1[key] = set()
        sample_key_to_s1[key].add(entity_id)

print("Fast blocking lookup ready.")
print("Number of unique blocking keys:", len(sample_key_to_s1))

Fast blocking lookup ready.
Number of unique blocking keys: 126


In [25]:
# Fast blocking recall test
# Uses vectorized string operations instead of row-by-row processing.

recovered_matches = {sid: set() for sid in sample_ids}

# Create regex patterns from our 126 blocking keys
name_keys = []
addr_first_keys = []
addr_last_keys = []
num_keys = []

for key in sample_key_to_s1:
    parts = key.split("|", 2)

    if len(parts) != 3:
        continue

    key_type = parts[1]
    value = parts[2]

    if key_type == "name":
        name_keys.append((key, value))
    elif key_type == "name2":
        name_keys.append((key, value))
    elif key_type == "name1":
        name_keys.append((key, value))
    elif key_type == "addr_first3":
        addr_first_keys.append((key, value))
    elif key_type == "addr_last2":
        addr_last_keys.append((key, value))
    elif key_type == "num":
        num_keys.append((key, value))


def check_chunk(chunk):
    # Work only with the fields needed
    names = chunk["business_name"].fillna("").astype(str).str.lower()
    addresses = chunk["business_address"].fillna("").astype(str).str.lower()
    countries = chunk["country"].fillna("").astype(str).str.lower()

    for key, value in name_keys:
        country = key.split("|")[0]

        mask = (
            countries.eq(country)
            & names.str.contains(
                re.escape(value),
                regex=True,
                na=False
            )
        )

        if mask.any():
            for entity_id in sample_key_to_s1[key]:
                recovered_matches[entity_id].update(
                    chunk.loc[mask, "entity_id"].tolist()
                )

    for key, value in addr_first_keys:
        country = key.split("|")[0]

        mask = (
            countries.eq(country)
            & addresses.str.contains(
                re.escape(value),
                regex=True,
                na=False
            )
        )

        if mask.any():
            for entity_id in sample_key_to_s1[key]:
                recovered_matches[entity_id].update(
                    chunk.loc[mask, "entity_id"].tolist()
                )

    for key, value in addr_last_keys:
        country = key.split("|")[0]

        mask = (
            countries.eq(country)
            & addresses.str.contains(
                re.escape(value),
                regex=True,
                na=False
            )
        )

        if mask.any():
            for entity_id in sample_key_to_s1[key]:
                recovered_matches[entity_id].update(
                    chunk.loc[mask, "entity_id"].tolist()
                )


for source_name in ["source2", "source3"]:

    path = f"s3://{bucket}/dataset/train/train_{source_name}.tsv"

    print(f"Scanning {source_name}...")

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=200000,
        dtype=str,
        usecols=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    ):
        check_chunk(chunk)

    print(f"{source_name} complete.")


# Compare with ground truth
print("\nBLOCKING RECALL")
print("=" * 60)

total_true = 0
total_found = 0

for _, row in sample_gt.iterrows():

    s1_id = row["source1_entity_id"]
    true_ids = row["true_matches"]
    found_ids = recovered_matches[s1_id]

    recovered = true_ids & found_ids
    missed = true_ids - found_ids

    total_true += len(true_ids)
    total_found += len(recovered)

    print(
        f"{s1_id}: "
        f"{len(recovered)}/{len(true_ids)} recovered"
    )

    if missed:
        print("  Missed:", sorted(missed))

print("=" * 60)
print("TOTAL TRUE:", total_true)
print("TOTAL RECOVERED:", total_found)

if total_true:
    print("RECALL:", f"{total_found / total_true:.2%}")

Scanning source2...
source2 complete.
Scanning source3...
source3 complete.

BLOCKING RECALL
S1-236312382: 3/3 recovered
S1-629417405: 7/7 recovered
S1-309349399: 0/0 recovered
S1-773889195: 6/6 recovered
S1-27541239: 4/4 recovered
S1-377745466: 4/4 recovered
S1-851869949: 5/5 recovered
S1-133037285: 6/6 recovered
S1-626914593: 2/2 recovered
S1-504790211: 6/7 recovered
  Missed: ['S2-482219248']
S1-341506266: 3/3 recovered
S1-865131206: 4/4 recovered
S1-755362802: 2/2 recovered
S1-221443075: 3/3 recovered
S1-925783039: 4/4 recovered
S1-765734254: 4/4 recovered
S1-82053739: 1/2 recovered
  Missed: ['S3-250737963']
S1-22305073: 4/4 recovered
S1-564729135: 4/6 recovered
  Missed: ['S2-327309238', 'S3-912418512']
S1-785847572: 4/4 recovered
TOTAL TRUE: 80
TOTAL RECOVERED: 76
RECALL: 95.00%


In [27]:
# Inspect the 4 missed true matches

missed_ids = {
    "S2-482219248",
    "S3-250737963",
    "S2-327309238",
    "S3-912418512"
}

missed_rows = []

for source_name in ["source2", "source3"]:

    path = f"s3://{bucket}/dataset/train/train_{source_name}.tsv"

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=200000,
        dtype=str,
        usecols=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    ):
        found = chunk[chunk["entity_id"].isin(missed_ids)]

        if len(found) > 0:
            missed_rows.append(found)

missed_df = pd.concat(missed_rows, ignore_index=True)

print("Missed true-match records found:")
display(missed_df)

Missed true-match records found:


,entity_id,business_name,business_address,country
0,S2-482219248,Crysta1 Lending PC,"11643-11645 PROSPERITY ROAD, SOUTH JORDAN, UT",US
1,S2-327309238,డ్రీమ్ కన్‌స్ట్రక్షన్ లిమిటెడ్,"H.NO 16-11-23/37/A, OSARAMBAGH, HYDERABAD, Tel...",India
2,S3-250737963,Kelonyla,"303, 3Rd Floor Sakar 5 B/H Natraj Cinema Ashra...",India
3,S3-912418512,డ్రీమ్ కన్‌స్ట్రక్షన్ లిమిటెడ్,"D/16-11-23/37/a, 2Nd Floor, Flat No.207, Sagar...",India


In [28]:
# Show the Source 1 records corresponding to the 4 missed matches

missed_s1_ids = [
    "S1-504790211",   # Crystal Lending
    "S1-564729135",   # Dream Construction
    "S1-82053739"     # Smart Healthcare
]

s1_missed = sample[
    sample["entity_id"].isin(missed_s1_ids)
][[
    "entity_id",
    "business_name",
    "business_address",
    "country",
    "name_norm",
    "address_norm"
]].copy()

print("Source 1 records:")
display(s1_missed)

print("\nMissed Source 2/3 records:")
display(
    missed_df.assign(
        name_norm=missed_df["business_name"].apply(normalize_name),
        address_norm=missed_df["business_address"].apply(normalize_address)
    )
)

Source 1 records:


,entity_id,business_name,business_address,country,name_norm,address_norm
10,S1-504790211,Crystal Lending PC,"11643 Prosperity Road, South Jordan, UT",US,crystal lending,11643 prosperity rd south jordan ut
11,S1-564729135,Dream Construction Limited,"H.No.16-11-23/37/A, 2Nd Floor, Flat No.207, Sa...",India,dream construction,h no 16 11 23 37 a 2nd fl flat no 207 sagar ho...
16,S1-82053739,Smart Healthcare Private Limited,"303, 3Rd Floor Sakar 5 B/H Natraj Cinema Ashra...",India,smart healthcare,303 3rd fl sakar 5 b h natraj cinema ashram rd...



Missed Source 2/3 records:


,entity_id,business_name,business_address,country,name_norm,address_norm
0,S2-482219248,Crysta1 Lending PC,"11643-11645 PROSPERITY ROAD, SOUTH JORDAN, UT",US,crysta1 lending,11643 11645 prosperity rd south jordan ut
1,S2-327309238,డ్రీమ్ కన్‌స్ట్రక్షన్ లిమిటెడ్,"H.NO 16-11-23/37/A, OSARAMBAGH, HYDERABAD, Tel...",India,ddriim knsttrkssn limittedd,h no 16 11 23 37 a osarambagh hyderabad telangana
2,S3-250737963,Kelonyla,"303, 3Rd Floor Sakar 5 B/H Natraj Cinema Ashra...",India,kelonyla,303 3rd fl sakar 5 b h natraj cinema ashram rd...
3,S3-912418512,డ్రీమ్ కన్‌స్ట్రక్షన్ లిమిటెడ్,"D/16-11-23/37/a, 2Nd Floor, Flat No.207, Sagar...",India,ddriim knsttrkssn limittedd,d 16 11 23 37 a 2nd fl flat no 207 sagar hotel...


In [29]:
# Test an additional address-based blocking strategy
# specifically for cases where the business name is unreliable.

def extra_address_keys(address):
    address = normalize_address(address)

    if not address:
        return set()

    tokens = address.split()
    keys = set()

    # Street/building numbers
    numbers = re.findall(r"\b\d+[a-z]?\b", address)

    for number in numbers[:4]:
        keys.add("num|" + number)

    # First 4 address tokens
    if len(tokens) >= 4:
        keys.add("first4|" + " ".join(tokens[:4]))

    # First 5 address tokens
    if len(tokens) >= 5:
        keys.add("first5|" + " ".join(tokens[:5]))

    # Last 3 address tokens
    if len(tokens) >= 3:
        keys.add("last3|" + " ".join(tokens[-3:]))

    return keys


print("Additional address-key test")
print("=" * 60)

for _, row in pd.concat([
    s1_missed,
    missed_df.assign(
        name_norm=missed_df["business_name"].apply(normalize_name),
        address_norm=missed_df["business_address"].apply(normalize_address)
    )
]).iterrows():

    keys = extra_address_keys(row["address_norm"])

    print("\n", row["entity_id"])
    print(row["business_name"])
    print("Keys:")
    for key in sorted(keys):
        print(" ", key)

Additional address-key test

 S1-504790211
Crystal Lending PC
Keys:
  first4|11643 prosperity rd south
  first5|11643 prosperity rd south jordan
  last3|south jordan ut
  num|11643

 S1-564729135
Dream Construction Limited
Keys:
  first4|h no 16 11
  first5|h no 16 11 23
  last3|osarambagh hyderabad telangana
  num|11
  num|16
  num|23
  num|37

 S1-82053739
Smart Healthcare Private Limited
Keys:
  first4|303 3rd fl sakar
  first5|303 3rd fl sakar 5
  last3|rd ahmedabad gujarat
  num|303
  num|5

 S2-482219248
Crysta1 Lending  PC
Keys:
  first4|11643 11645 prosperity rd
  first5|11643 11645 prosperity rd south
  last3|south jordan ut
  num|11643
  num|11645

 S2-327309238
డ్రీమ్ కన్‌స్ట్రక్షన్ లిమిటెడ్
Keys:
  first4|h no 16 11
  first5|h no 16 11 23
  last3|osarambagh hyderabad telangana
  num|11
  num|16
  num|23
  num|37

 S3-250737963
Kelonyla
Keys:
  first4|303 3rd fl sakar
  first5|303 3rd fl sakar 5
  last3|rd ahmedabad gujarat
  num|303
  num|5

 S3-912418512
డ్రీమ్ కన్‌స్ట్రక్

In [30]:
# Create a more robust address-core representation

def address_core(address):
    text = normalize_address(address)

    # Remove common structural labels
    text = re.sub(
        r"\b(h|hno|no|floor|fl|flat|unit|apt|apartment|suite|ste|b|bh)\b",
        " ",
        text
    )

    # Normalize repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


print("Address-core comparison")
print("=" * 60)

comparison = pd.concat([
    s1_missed,
    missed_df.assign(
        name_norm=missed_df["business_name"].apply(normalize_name),
        address_norm=missed_df["business_address"].apply(normalize_address)
    )
]).copy()

comparison["address_core"] = comparison["address_norm"].apply(address_core)

display(
    comparison[
        ["entity_id", "business_name", "address_core"]
    ]
)

Address-core comparison


,entity_id,business_name,address_core
10,S1-504790211,Crystal Lending PC,11643 prosperity rd south jordan ut
11,S1-564729135,Dream Construction Limited,16 11 23 37 a 2nd 207 sagar hotel building opp...
16,S1-82053739,Smart Healthcare Private Limited,303 3rd sakar 5 natraj cinema ashram rd ahmeda...
0,S2-482219248,Crysta1 Lending PC,11643 11645 prosperity rd south jordan ut
1,S2-327309238,డ్రీమ్ కన్‌స్ట్రక్షన్ లిమిటెడ్,16 11 23 37 a osarambagh hyderabad telangana
2,S3-250737963,Kelonyla,303 3rd sakar 5 natraj cinema ashram rd ahmeda...
3,S3-912418512,డ్రీమ్ కన్‌స్ట్రక్షన్ లిమిటెడ్,d 16 11 23 37 a 2nd 207 sagar hotel building o...


In [31]:
# Test address-core similarity on the 4 missed pairs

from rapidfuzz.fuzz import token_set_ratio, ratio

test_pairs = [
    ("S1-504790211", "S2-482219248"),
    ("S1-564729135", "S2-327309238"),
    ("S1-564729135", "S3-912418512"),
    ("S1-82053739", "S3-250737963"),
]

# Combine the known records
all_test_records = pd.concat([
    sample[[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]],
    missed_df[[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]]
]).drop_duplicates("entity_id")

records = all_test_records.set_index("entity_id")

print("Address-core similarity:")
print("=" * 70)

for s1_id, candidate_id in test_pairs:

    s1_core = address_core(
        records.loc[s1_id, "business_address"]
    )

    candidate_core = address_core(
        records.loc[candidate_id, "business_address"]
    )

    similarity = token_set_ratio(s1_core, candidate_core)

    print(f"\n{s1_id}  <->  {candidate_id}")
    print(f"Similarity: {similarity:.1f}")
    print("S1: ", s1_core)
    print("S2/3:", candidate_core)

Address-core similarity:

S1-504790211  <->  S2-482219248
Similarity: 100.0
S1:  11643 prosperity rd south jordan ut
S2/3: 11643 11645 prosperity rd south jordan ut

S1-564729135  <->  S2-327309238
Similarity: 100.0
S1:  16 11 23 37 a 2nd 207 sagar hotel building opp rta office mo osarambagh hyderabad telangana
S2/3: 16 11 23 37 a osarambagh hyderabad telangana

S1-564729135  <->  S3-912418512
Similarity: 97.0
S1:  16 11 23 37 a 2nd 207 sagar hotel building opp rta office mo osarambagh hyderabad telangana
S2/3: d 16 11 23 37 a 2nd 207 sagar hotel building opp rta office mo osarambagh hyderabad tg

S1-82053739  <->  S3-250737963
Similarity: 100.0
S1:  303 3rd sakar 5 natraj cinema ashram rd ahmedabad gujarat
S2/3: 303 3rd sakar 5 natraj cinema ashram rd ahmedabad gujarat


In [32]:
# Improved address-core blocking keys

def address_core_block_keys(address, country):
    core = address_core(address)

    if not core:
        return set()

    tokens = core.split()
    country = str(country).strip().lower()

    keys = set()

    # Specific beginning of address
    if len(tokens) >= 4:
        keys.add(
            f"{country}|core_first4|{' '.join(tokens[:4])}"
        )

    # Specific ending/location
    if len(tokens) >= 3:
        keys.add(
            f"{country}|core_last3|{' '.join(tokens[-3:])}"
        )

    # Strong address-number combination
    numbers = re.findall(r"\b\d+[a-z]?\b", core)

    if len(numbers) >= 2 and len(tokens) >= 4:
        keys.add(
            f"{country}|core_nums|{' '.join(numbers[:3])}|{' '.join(tokens[-3:])}"
        )

    return keys


print("Testing improved address-core keys")
print("=" * 60)

for _, row in comparison.iterrows():
    keys = address_core_block_keys(
        row["address_norm"],
        row["country"]
    )

    print(row["entity_id"], "->", len(keys), "keys")
    for key in sorted(keys):
        print(" ", key)

Testing improved address-core keys
S1-504790211 -> 2 keys
  us|core_first4|11643 prosperity rd south
  us|core_last3|south jordan ut
S1-564729135 -> 3 keys
  india|core_first4|16 11 23 37
  india|core_last3|osarambagh hyderabad telangana
  india|core_nums|16 11 23|osarambagh hyderabad telangana
S1-82053739 -> 3 keys
  india|core_first4|303 3rd sakar 5
  india|core_last3|rd ahmedabad gujarat
  india|core_nums|303 5|rd ahmedabad gujarat
S2-482219248 -> 3 keys
  us|core_first4|11643 11645 prosperity rd
  us|core_last3|south jordan ut
  us|core_nums|11643 11645|south jordan ut
S2-327309238 -> 3 keys
  india|core_first4|16 11 23 37
  india|core_last3|osarambagh hyderabad telangana
  india|core_nums|16 11 23|osarambagh hyderabad telangana
S3-250737963 -> 3 keys
  india|core_first4|303 3rd sakar 5
  india|core_last3|rd ahmedabad gujarat
  india|core_nums|303 5|rd ahmedabad gujarat
S3-912418512 -> 3 keys
  india|core_first4|d 16 11 23
  india|core_last3|osarambagh hyderabad tg
  india|core_num

In [33]:
# Verify that the improved address blocker connects
# every previously missed true match.

s1_core_keys = {}

for _, row in s1_missed.iterrows():
    s1_core_keys[row["entity_id"]] = address_core_block_keys(
        row["address_norm"],
        row["country"]
    )

candidate_core_keys = {}

missed_with_norm = missed_df.copy()
missed_with_norm["address_norm"] = missed_with_norm[
    "business_address"
].apply(normalize_address)

for _, row in missed_with_norm.iterrows():
    candidate_core_keys[row["entity_id"]] = address_core_block_keys(
        row["address_norm"],
        row["country"]
    )


test_pairs = [
    ("S1-504790211", "S2-482219248"),
    ("S1-564729135", "S2-327309238"),
    ("S1-564729135", "S3-912418512"),
    ("S1-82053739", "S3-250737963"),
]

print("Improved blocker recall test")
print("=" * 60)

for s1_id, candidate_id in test_pairs:

    common = (
        s1_core_keys[s1_id]
        &
        candidate_core_keys[candidate_id]
    )

    print(
        f"{s1_id} -> {candidate_id}:",
        "RECOVERED" if common else "MISSED"
    )

    if common:
        for key in sorted(common):
            print("  Shared:", key)

Improved blocker recall test
S1-504790211 -> S2-482219248: RECOVERED
  Shared: us|core_last3|south jordan ut
S1-564729135 -> S2-327309238: RECOVERED
  Shared: india|core_first4|16 11 23 37
  Shared: india|core_last3|osarambagh hyderabad telangana
  Shared: india|core_nums|16 11 23|osarambagh hyderabad telangana
S1-564729135 -> S3-912418512: MISSED
S1-82053739 -> S3-250737963: RECOVERED
  Shared: india|core_first4|303 3rd sakar 5
  Shared: india|core_last3|rd ahmedabad gujarat
  Shared: india|core_nums|303 5|rd ahmedabad gujarat


In [34]:
def address_number_location_key(address, country):
    core = address_core(address)
    country = str(country).strip().lower()

    numbers = re.findall(r"\b\d+[a-z]?\b", core)

    if len(numbers) < 3:
        return None

    # Use the first 3 address numbers + country.
    # This handles formatting differences such as:
    # H.No.16-11-23/37/A
    # D/16-11-23/37/A
    return f"{country}|addr_nums|{' '.join(numbers[:3])}"


print("Testing number-based address key")
print("=" * 60)

for s1_id, candidate_id in [
    ("S1-564729135", "S3-912418512"),
]:
    s1_row = sample[sample["entity_id"] == s1_id].iloc[0]
    c_row = missed_df[missed_df["entity_id"] == candidate_id].iloc[0]

    s1_key = address_number_location_key(
        s1_row["address_norm"],
        s1_row["country"]
    )

    c_key = address_number_location_key(
        c_row["business_address"],
        c_row["country"]
    )

    print(s1_id, "->", s1_key)
    print(candidate_id, "->", c_key)
    print("RECOVERED:", s1_key == c_key)

Testing number-based address key
S1-564729135 -> india|addr_nums|16 11 23
S3-912418512 -> india|addr_nums|16 11 23
RECOVERED: True


In [35]:
# Final blocking-key test on the 20-entity validation sample

def final_block_keys(name, address, country):
    keys = set()

    name = normalize_name(name)
    address = normalize_address(address)
    country = str(country).strip().lower()

    # Existing name-based keys
    name_tokens = name.split()

    if name:
        keys.add(f"{country}|name|{name}")

    if len(name_tokens) >= 2:
        keys.add(f"{country}|name2|{' '.join(name_tokens[:2])}")

    if len(name_tokens) >= 1:
        keys.add(f"{country}|name1|{name_tokens[0]}")

    # Improved address-core keys
    keys.update(
        address_core_block_keys(address, country)
    )

    # Address number key
    number_key = address_number_location_key(address, country)

    if number_key:
        keys.add(number_key)

    return keys


print("Testing final blocking rules")
print("=" * 60)

for _, row in sample.iterrows():
    keys = final_block_keys(
        row["business_name"],
        row["business_address"],
        row["country"]
    )

    print(
        row["entity_id"],
        "->",
        len(keys),
        "keys"
    )

print("=" * 60)
print("Total validation entities:", len(sample))

Testing final blocking rules
S1-925783039 -> 5 keys
S1-773889195 -> 5 keys
S1-377745466 -> 5 keys
S1-133037285 -> 5 keys
S1-755362802 -> 5 keys
S1-851869949 -> 5 keys
S1-785847572 -> 6 keys
S1-27541239 -> 6 keys
S1-629417405 -> 5 keys
S1-22305073 -> 6 keys
S1-504790211 -> 5 keys
S1-564729135 -> 7 keys
S1-236312382 -> 5 keys
S1-865131206 -> 4 keys
S1-626914593 -> 6 keys
S1-765734254 -> 5 keys
S1-82053739 -> 6 keys
S1-309349399 -> 5 keys
S1-221443075 -> 5 keys
S1-341506266 -> 7 keys
Total validation entities: 20


In [43]:
# Final blocking recall test on the 20 validation entities
# Uses chunked vectorized scanning to keep RAM usage low.

sample_key_to_s1_final = {}

for _, row in sample.iterrows():
    for key in final_block_keys(
        row["business_name"],
        row["business_address"],
        row["country"]
    ):
        sample_key_to_s1_final.setdefault(key, set()).add(
            row["entity_id"]
        )


recovered_final = {
    s1_id: set()
    for s1_id in sample_ids
}

def scan_final_blocking(path, source_label):
    print(f"Scanning {source_label}...")

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=200_000,
        dtype=str,
        keep_default_na=False
    ):
        chunk["name_norm"] = chunk["business_name"].map(normalize_name)
        chunk["address_norm"] = normalize_address_fast(
            chunk["business_address"]
        )

        for _, row in chunk.iterrows():

            keys = final_block_keys(
                row["name_norm"],
                row["address_norm"],
                row["country"]
            )

            matched_s1 = set()

            for key in keys:
                if key in sample_key_to_s1_final:
                    matched_s1.update(
                        sample_key_to_s1_final[key]
                    )

            for s1_id in matched_s1:
                recovered_final[s1_id].add(
                    row["entity_id"]
                )

    print(f"{source_label} complete.")


scan_final_blocking(
    f"s3://{bucket}/dataset/train/train_source2.tsv",
    "source2"
)

scan_final_blocking(
    f"s3://{bucket}/dataset/train/train_source3.tsv",
    "source3"
)


print()
print("=" * 70)
print("FINAL BLOCKING RECALL")
print("=" * 70)

total_true = 0
total_recovered = 0

for _, row in sample_gt.iterrows():

    s1_id = row["source1_entity_id"]
    true_matches = row["true_matches"]

    recovered = recovered_final.get(s1_id, set())

    found = true_matches & recovered
    missed = true_matches - recovered

    total_true += len(true_matches)
    total_recovered += len(found)

    print(
        f"{s1_id}: "
        f"{len(found)}/{len(true_matches)} recovered"
    )

    if missed:
        print("  MISSED:", sorted(missed))

print("=" * 70)
print("TOTAL TRUE:", total_true)
print("TOTAL RECOVERED:", total_recovered)

if total_true > 0:
    print(
        "RECALL:",
        f"{100 * total_recovered / total_true:.2f}%"
    )

Scanning source2...


KeyboardInterrupt: 

In [ ]:
# FAST final blocking recall test
# No iterrows() over the large datasets.

print("Preparing validation keys...")

# Map each blocking key -> Source1 IDs
sample_key_to_s1_final = {}

for _, row in sample.iterrows():
    keys = final_block_keys(
        row["business_name"],
        row["business_address"],
        row["country"]
    )

    for key in keys:
        sample_key_to_s1_final.setdefault(key, set()).add(
            row["entity_id"]
        )

# Convert keys into groups for fast vectorized matching
key_to_s1 = sample_key_to_s1_final

recovered_final = {
    s1_id: set()
    for s1_id in sample_ids
}


def scan_fast(path, source_label):

    print(f"\nScanning {source_label}...")

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            path,
            sep="\t",
            chunksize=200_000,
            dtype=str,
            keep_default_na=False
        ),
        start=1
    ):

        # Normalize columns
        chunk["name_norm"] = (
            chunk["business_name"]
            .map(normalize_name)
        )

        chunk["address_norm"] = normalize_address_fast(
            chunk["business_address"]
        )

        # Country
        chunk["country_norm"] = (
            chunk["country"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        # ------------------------------------------------
        # 1. Exact normalized name blocking
        # ------------------------------------------------
        name_keys = (
            chunk["country_norm"]
            + "|name|"
            + chunk["name_norm"]
        )

        for key, s1_ids in key_to_s1.items():

            mask = name_keys.eq(key)

            if mask.any():
                ids = chunk.loc[mask, "entity_id"].tolist()

                for s1_id in s1_ids:
                    recovered_final[s1_id].update(ids)

        # ------------------------------------------------
        # 2. Address-core blocking
        # ------------------------------------------------
        cores = (
            chunk["address_norm"]
            .str.replace(
                r"\b(h|hno|no|floor|fl|flat|unit|apt|apartment|suite|ste|b|bh)\b",
                " ",
                regex=True
            )
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

        core_first4 = (
            chunk["country_norm"]
            + "|core_first4|"
            + cores.str.split().str[:4].str.join(" ")
        )

        core_last3 = (
            chunk["country_norm"]
            + "|core_last3|"
            + cores.str.split().str[-3:].str.join(" ")
        )

        for key in key_to_s1:

            if (
                "|core_first4|" not in key
                and "|core_last3|" not in key
            ):
                continue

            if "|core_first4|" in key:
                mask = core_first4.eq(key)
            else:
                mask = core_last3.eq(key)

            if mask.any():
                ids = chunk.loc[mask, "entity_id"].tolist()

                for s1_id in key_to_s1[key]:
                    recovered_final[s1_id].update(ids)

        if chunk_no % 5 == 0:
            print(
                f"  processed {chunk_no * 200_000:,} rows..."
            )

    print(f"{source_label} complete.")


scan_fast(
    f"s3://{bucket}/dataset/train/train_source2.tsv",
    "source2"
)

scan_fast(
    f"s3://{bucket}/dataset/train/train_source3.tsv",
    "source3"
)


# ------------------------------------------------
# Recall calculation
# ------------------------------------------------

print()
print("=" * 70)
print("FINAL BLOCKING RECALL")
print("=" * 70)

total_true = 0
total_recovered = 0

for _, row in sample_gt.iterrows():

    s1_id = row["source1_entity_id"]
    true_matches = row["true_matches"]

    recovered = recovered_final.get(
        s1_id,
        set()
    )

    found = true_matches & recovered
    missed = true_matches - recovered

    total_true += len(true_matches)
    total_recovered += len(found)

    print(
        f"{s1_id}: "
        f"{len(found)}/{len(true_matches)} recovered"
    )

    if missed:
        print(
            "  MISSED:",
            sorted(missed)
        )

print("=" * 70)
print("TOTAL TRUE:", total_true)
print("TOTAL RECOVERED:", total_recovered)

if total_true:
    print(
        "RECALL:",
        f"{100 * total_recovered / total_true:.2f}%"
    )

Preparing validation keys...

Scanning source2...


In [1]:
import os

print("Checking candidate index files...")

if os.path.exists("candidate_index"):
    print("Folder exists.")
    for f in os.listdir("candidate_index"):
        path = os.path.join("candidate_index", f)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{f} -> {size_mb:.2f} MB")
else:
    print("candidate_index folder does NOT exist.")

Checking candidate index files...
Folder exists.
source3_index.tsv -> 205.14 MB


In [2]:
import os

path = "candidate_index/source2_index.tsv"

if os.path.exists(path):
    print("Source 2 index EXISTS")
    print("Size:", round(os.path.getsize(path) / (1024 * 1024), 2), "MB")
else:
    print("Source 2 index NOT FOUND")

Source 2 index NOT FOUND


In [3]:
import os
import pandas as pd
import re
from unidecode import unidecode

BUCKET = "sagemaker-eu-north-1-152025327817"
TEST_S2 = f"s3://{BUCKET}/dataset/test/test_source2.tsv"

INDEX_DIR = "candidate_index"
S2_INDEX = os.path.join(INDEX_DIR, "source2_index.tsv")

os.makedirs(INDEX_DIR, exist_ok=True)

def clean_series(s):
    s = s.fillna("").astype(str).str.lower()
    s = s.map(unidecode)
    s = s.str.replace(r"[^a-z0-9\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

def make_source2_index(chunk):
    name = clean_series(chunk["business_name"])
    address = clean_series(chunk["business_address"])
    country = clean_series(chunk["country"])

    rows = []

    for i in range(len(chunk)):
        eid = chunk.iloc[i]["entity_id"]
        c = country.iloc[i]
        n = name.iloc[i]
        a = address.iloc[i]

        keys = set()

        if n:
            keys.add(f"{c}|name|{n}")

        nt = n.split()
        if len(nt) >= 2:
            keys.add(f"{c}|name2|{' '.join(nt[:2])}")

        if nt:
            keys.add(f"{c}|name1|{nt[0]}")

        at = a.split()

        if len(at) >= 4:
            keys.add(f"{c}|addr_first4|{' '.join(at[:4])}")

        if len(at) >= 3:
            keys.add(f"{c}|addr_last3|{' '.join(at[-3:])}")

        nums = re.findall(r"\b\d+[a-z]?\b", a)

        if len(nums) >= 3:
            keys.add(f"{c}|addr_nums|{' '.join(nums[:3])}")

        for key in keys:
            rows.append((key, eid))

    return pd.DataFrame(rows, columns=["block_key", "entity_id"])


# Remove incomplete old file if one somehow exists
if os.path.exists(S2_INDEX):
    os.remove(S2_INDEX)

total = 0
first_write = True

print("Starting Source 2 index...")
print("This may take some time, but progress will be printed after each chunk.")

for chunk in pd.read_csv(
    TEST_S2,
    sep="\t",
    dtype=str,
    chunksize=100000
):
    idx = make_source2_index(chunk)

    idx.to_csv(
        S2_INDEX,
        sep="\t",
        index=False,
        mode="w" if first_write else "a",
        header=first_write
    )

    first_write = False
    total += len(chunk)

    print(
        f"Processed {total:,} Source 2 rows | "
        f"index rows written: {len(idx):,}",
        flush=True
    )

print("DONE")
print("File:", S2_INDEX)
print("Size:", round(os.path.getsize(S2_INDEX) / (1024 * 1024), 2), "MB")

Starting Source 2 index...
This may take some time, but progress will be printed after each chunk.
Processed 100,000 Source 2 rows | index rows written: 503,653
Processed 200,000 Source 2 rows | index rows written: 504,029
Processed 300,000 Source 2 rows | index rows written: 504,012
Processed 400,000 Source 2 rows | index rows written: 504,261
Processed 500,000 Source 2 rows | index rows written: 503,789
Processed 600,000 Source 2 rows | index rows written: 504,089
Processed 700,000 Source 2 rows | index rows written: 504,164
Processed 800,000 Source 2 rows | index rows written: 504,367
Processed 900,000 Source 2 rows | index rows written: 504,057
Processed 1,000,000 Source 2 rows | index rows written: 504,223
Processed 1,100,000 Source 2 rows | index rows written: 503,996
Processed 1,200,000 Source 2 rows | index rows written: 504,191
Processed 1,300,000 Source 2 rows | index rows written: 503,788
Processed 1,400,000 Source 2 rows | index rows written: 503,964
Processed 1,500,000 Sou

In [4]:
import shutil

total, used, free = shutil.disk_usage(".")

print("Total:", round(total / (1024**3), 2), "GB")
print("Used :", round(used / (1024**3), 2), "GB")
print("Free :", round(free / (1024**3), 2), "GB")

Total: 4.94 GB
Used : 2.24 GB
Free : 2.7 GB


In [ ]:
import os
import re
import pandas as pd
from unidecode import unidecode

# Paths
BUCKET = "sagemaker-eu-north-1-152025327817"

TEST_S1 = f"s3://{BUCKET}/dataset/test/test_source1.tsv"

S2_INDEX = "candidate_index/source2_index.tsv"
S3_INDEX = "candidate_index/source3_index.tsv"

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CANDIDATE_FILE = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")

# Safety checks
print("Checking files...")

for path in [S2_INDEX, S3_INDEX]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing index: {path}")
    print(path, "OK")

print("Starting candidate generation...")
print("This uses existing indexes; no new large index will be created.")

# Same normalization used when the indexes were created
def clean_series(s):
    s = s.fillna("").astype(str).str.lower()
    s = s.map(unidecode)
    s = s.str.replace(r"[^a-z0-9\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

def make_keys(chunk):
    name = clean_series(chunk["business_name"])
    address = clean_series(chunk["business_address"])
    country = clean_series(chunk["country"])

    all_keys = []

    for i in range(len(chunk)):
        c = country.iloc[i]
        n = name.iloc[i]
        a = address.iloc[i]

        keys = set()

        if n:
            keys.add(f"{c}|name|{n}")

        nt = n.split()

        if len(nt) >= 2:
            keys.add(f"{c}|name2|{' '.join(nt[:2])}")

        if nt:
            keys.add(f"{c}|name1|{nt[0]}")

        at = a.split()

        if len(at) >= 4:
            keys.add(f"{c}|addr_first4|{' '.join(at[:4])}")

        if len(at) >= 3:
            keys.add(f"{c}|addr_last3|{' '.join(at[-3:])}")

        nums = re.findall(r"\b\d+[a-z]?\b", a)

        if len(nums) >= 3:
            keys.add(f"{c}|addr_nums|{' '.join(nums[:3])}")

        all_keys.append(keys)

    return all_keys


def add_index_matches(index_path, key_to_rows):
    """
    Read an existing index in chunks and add matching
    Source2/Source3 entity IDs to the current Source1 rows.
    """
    for idx_chunk in pd.read_csv(
        index_path,
        sep="\t",
        dtype=str,
        chunksize=200000
    ):
        matched = idx_chunk[idx_chunk["block_key"].isin(key_to_rows)]

        if len(matched) == 0:
            continue

        for key, eid in zip(
            matched["block_key"].tolist(),
            matched["entity_id"].tolist()
        ):
            for row_num in key_to_rows[key]:
                candidate_sets[row_num].add(eid)


# Remove an incomplete candidate file if it exists
if os.path.exists(CANDIDATE_FILE):
    os.remove(CANDIDATE_FILE)

first_write = True
total_s1 = 0

for s1_chunk in pd.read_csv(
    TEST_S1,
    sep="\t",
    dtype=str,
    chunksize=100000
):

    keys_per_row = make_keys(s1_chunk)

    # key -> Source1 row positions
    key_to_rows = {}

    for row_num, keys in enumerate(keys_per_row):
        for key in keys:
            key_to_rows.setdefault(key, []).append(row_num)

    # One set of candidates for every Source1 row
    candidate_sets = [set() for _ in range(len(s1_chunk))]

    # Search existing Source2 and Source3 indexes
    add_index_matches(S2_INDEX, key_to_rows)
    add_index_matches(S3_INDEX, key_to_rows)

    # Build output for this chunk
    output = pd.DataFrame({
        "source1_entity_id": s1_chunk["entity_id"].values,
        "candidate_entity_ids": [
            ",".join(sorted(x)) for x in candidate_sets
        ]
    })

    output.to_csv(
        CANDIDATE_FILE,
        sep="\t",
        index=False,
        mode="w" if first_write else "a",
        header=first_write
    )

    first_write = False
    total_s1 += len(s1_chunk)

    print(
        f"Processed {total_s1:,} Source 1 rows",
        flush=True
    )

print("\nDONE")
print("Candidate file:", CANDIDATE_FILE)
print(
    "Size:",
    round(os.path.getsize(CANDIDATE_FILE) / (1024**2), 2),
    "MB"
)

Checking files...
candidate_index/source2_index.tsv OK
candidate_index/source3_index.tsv OK
Starting candidate generation...
This uses existing indexes; no new large index will be created.


In [1]:
import os

path = "output/candidate_pairs.tsv"

if os.path.exists(path):
    print("Partial candidate file exists")
    print("Size:", round(os.path.getsize(path) / (1024**2), 2), "MB")
else:
    print("No candidate file was created")

No candidate file was created


In [2]:
import os
import shutil

for f in [
    "candidate_index/source2_index.tsv",
    "candidate_index/source3_index.tsv"
]:
    print(f, round(os.path.getsize(f) / (1024**3), 2), "GB")

free = shutil.disk_usage(".").free / (1024**3)
print("Free space:", round(free, 2), "GB")

candidate_index/source2_index.tsv 0.99 GB
candidate_index/source3_index.tsv 0.2 GB
Free space: 3.66 GB


In [3]:
import os

for f in [
    "candidate_index/source2_index.tsv",
    "candidate_index/source3_index.tsv"
]:
    with open(f, "rb") as file:
        lines = sum(1 for _ in file)

    print(os.path.basename(f), "rows:", f"{lines:,}")

source2_index.tsv rows: 24,633,019
source3_index.tsv rows: 5,046,021


In [4]:
import importlib.util

print("duckdb installed:", importlib.util.find_spec("duckdb") is not None)

duckdb installed: True


In [5]:
import duckdb
import os

TEST_S1 = "s3://sagemaker-eu-north-152025327817/dataset/test/test_source1.tsv"

In [6]:
import duckdb
import os

BUCKET = "sagemaker-eu-north-1-152025327817"

TEST_S1 = f"s3://{BUCKET}/dataset/test/test_source1.tsv"

print("DuckDB:", duckdb.__version__)
print("Source 1:", TEST_S1)
print("Ready.")

DuckDB: 1.5.1
Source 1: s3://sagemaker-eu-north-1-152025327817/dataset/test/test_source1.tsv
Ready.


In [7]:
import os
import re
import pandas as pd
from unidecode import unidecode

BUCKET = "sagemaker-eu-north-1-152025327817"
TEST_S1 = f"s3://{BUCKET}/dataset/test/test_source1.tsv"

KEY_FILE = "candidate_index/source1_keys.tsv"

def clean_series(s):
    s = s.fillna("").astype(str).str.lower()
    s = s.map(unidecode)
    s = s.str.replace(r"[^a-z0-9\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

if os.path.exists(KEY_FILE):
    os.remove(KEY_FILE)

first_write = True
total_rows = 0
total_keys = 0

print("Creating Source 1 blocking keys...")

for chunk in pd.read_csv(
    TEST_S1,
    sep="\t",
    dtype=str,
    chunksize=100000
):
    names = clean_series(chunk["business_name"])
    addresses = clean_series(chunk["business_address"])
    countries = clean_series(chunk["country"])

    rows = []

    for i in range(len(chunk)):
        eid = chunk.iloc[i]["entity_id"]
        n = names.iloc[i]
        a = addresses.iloc[i]
        c = countries.iloc[i]

        keys = set()

        if n:
            keys.add(f"{c}|name|{n}")

        nt = n.split()

        if len(nt) >= 2:
            keys.add(f"{c}|name2|{' '.join(nt[:2])}")

        if nt:
            keys.add(f"{c}|name1|{nt[0]}")

        at = a.split()

        if len(at) >= 4:
            keys.add(f"{c}|addr_first4|{' '.join(at[:4])}")

        if len(at) >= 3:
            keys.add(f"{c}|addr_last3|{' '.join(at[-3:])}")

        nums = re.findall(r"\b\d+[a-z]?\b", a)

        if len(nums) >= 3:
            keys.add(f"{c}|addr_nums|{' '.join(nums[:3])}")

        for key in keys:
            rows.append((eid, key))

    out = pd.DataFrame(
        rows,
        columns=["source1_entity_id", "block_key"]
    )

    out.to_csv(
        KEY_FILE,
        sep="\t",
        index=False,
        mode="w" if first_write else "a",
        header=first_write
    )

    first_write = False
    total_rows += len(chunk)
    total_keys += len(out)

    print(
        f"Processed {total_rows:,} Source 1 rows | "
        f"{total_keys:,} keys written",
        flush=True
    )

print("\nDONE")
print("File:", KEY_FILE)
print(
    "Size:",
    round(os.path.getsize(KEY_FILE) / (1024**2), 2),
    "MB"
)

Creating Source 1 blocking keys...
Processed 100,000 Source 1 rows | 508,813 keys written
Processed 200,000 Source 1 rows | 1,017,632 keys written
Processed 300,000 Source 1 rows | 1,526,543 keys written
Processed 400,000 Source 1 rows | 2,035,432 keys written
Processed 500,000 Source 1 rows | 2,544,230 keys written
Processed 600,000 Source 1 rows | 3,053,006 keys written
Processed 700,000 Source 1 rows | 3,561,940 keys written
Processed 800,000 Source 1 rows | 4,071,016 keys written
Processed 900,000 Source 1 rows | 4,579,734 keys written
Processed 1,000,000 Source 1 rows | 5,088,588 keys written
Processed 1,100,000 Source 1 rows | 5,597,493 keys written
Processed 1,200,000 Source 1 rows | 6,106,425 keys written
Processed 1,300,000 Source 1 rows | 6,615,438 keys written
Processed 1,400,000 Source 1 rows | 7,124,171 keys written
Processed 1,500,000 Source 1 rows | 7,633,053 keys written
Processed 1,600,000 Source 1 rows | 8,141,977 keys written
Processed 1,700,000 Source 1 rows | 8,650

In [ ]:
import duckdb
import os
import shutil

S1_KEYS = "candidate_index/source1_keys.tsv"
S2_INDEX = "candidate_index/source2_index.tsv"
S3_INDEX = "candidate_index/source3_index.tsv"

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CANDIDATE_FILE = "output/candidate_pairs.tsv"

# Remove only an old/incomplete candidate file if present
if os.path.exists(CANDIDATE_FILE):
    os.remove(CANDIDATE_FILE)

print("Starting DuckDB candidate join...")
print("Source 1 keys :", round(os.path.getsize(S1_KEYS)/(1024**2), 2), "MB")
print("Source 2 index:", round(os.path.getsize(S2_INDEX)/(1024**2), 2), "MB")
print("Source 3 index:", round(os.path.getsize(S3_INDEX)/(1024**2), 2), "MB")
print("Free space    :", round(shutil.disk_usage(".").free/(1024**3), 2), "GB")

con = duckdb.connect()

# Keep memory controlled
con.execute("SET memory_limit='2GB'")
con.execute("SET threads=2")

# Use disk only if DuckDB needs temporary space
con.execute("SET temp_directory='candidate_index'")

print("Joining Source 1 keys with Source 2 and Source 3 indexes...")
print("The broad name1 block is excluded to prevent huge false candidate sets.")

query = f"""
COPY (
    SELECT
        s1.source1_entity_id,
        COALESCE(
            string_agg(DISTINCT candidates.entity_id, ',' ORDER BY candidates.entity_id),
            ''
        ) AS candidate_entity_ids
    FROM read_csv(
        '{S1_KEYS}',
        delim='\\t',
        header=true,
        columns={{
            'source1_entity_id':'VARCHAR',
            'block_key':'VARCHAR'
        }}
    ) s1
    LEFT JOIN (
        SELECT block_key, entity_id
        FROM read_csv(
            '{S2_INDEX}',
            delim='\\t',
            header=true,
            columns={{
                'block_key':'VARCHAR',
                'entity_id':'VARCHAR'
            }}
        )
        WHERE block_key NOT LIKE '%|name1|%'

        UNION ALL

        SELECT block_key, entity_id
        FROM read_csv(
            '{S3_INDEX}',
            delim='\\t',
            header=true,
            columns={{
                'block_key':'VARCHAR',
                'entity_id':'VARCHAR'
            }}
        )
        WHERE block_key NOT LIKE '%|name1|%'
    ) candidates
    ON s1.block_key = candidates.block_key
    GROUP BY s1.source1_entity_id
    ORDER BY s1.source1_entity_id
) TO '{CANDIDATE_FILE}'
WITH (
    HEADER,
    DELIMITER '\\t'
)
"""

con.execute(query)

con.close()

print()
print("DONE")
print("Candidate file:", CANDIDATE_FILE)
print(
    "Size:",
    round(os.path.getsize(CANDIDATE_FILE)/(1024**2), 2),
    "MB"
)
print(
    "Free space:",
    round(shutil.disk_usage(".").free/(1024**3), 2),
    "GB"
)

Starting DuckDB candidate join...
Source 1 keys : 364.01 MB
Source 2 index: 1016.95 MB
Source 3 index: 205.14 MB
Free space    : 3.27 GB
Joining Source 1 keys with Source 2 and Source 3 indexes...
The broad name1 block is excluded to prevent huge false candidate sets.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [1]:
import os
import shutil

print("Kernel check: OK")

path = "output/candidate_pairs.tsv"

if os.path.exists(path):
    print("Candidate file exists:", round(os.path.getsize(path)/(1024**2), 2), "MB")
else:
    print("No candidate file")

print("Free space:", round(shutil.disk_usage(".").free/(1024**3), 2), "GB")

Kernel check: OK
Candidate file exists: 0.0 MB
Free space: 3.31 GB


In [2]:
import duckdb
import os
import shutil

SOURCE2 = "candidate_index/source2_index.tsv"
FILTERED_S2 = "candidate_index/source2_filtered.tsv"

if os.path.exists(FILTERED_S2):
    os.remove(FILTERED_S2)

print("Creating filtered Source 2 index...")

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=2")

con.execute(f"""
COPY (
    SELECT block_key, entity_id
    FROM read_csv(
        '{SOURCE2}',
        delim='\\t',
        header=true,
        columns={{
            'block_key':'VARCHAR',
            'entity_id':'VARCHAR'
        }}
    )
    WHERE block_key NOT LIKE '%|name1|%'
)
TO '{FILTERED_S2}'
WITH (
    HEADER,
    DELIMITER '\\t'
)
""")

con.close()

print("DONE")
print(
    "Filtered Source 2 size:",
    round(os.path.getsize(FILTERED_S2)/(1024**2), 2),
    "MB"
)
print(
    "Free space:",
    round(shutil.disk_usage(".").free/(1024**3), 2),
    "GB"
)

Creating filtered Source 2 index...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DONE
Filtered Source 2 size: 872.75 MB
Free space: 2.45 GB


In [3]:
import os
import shutil

original = "candidate_index/source2_index.tsv"
filtered = "candidate_index/source2_filtered.tsv"

print("Before:")
print("Original exists:", os.path.exists(original))
print("Filtered exists:", os.path.exists(filtered))

os.remove(original)
os.rename(filtered, original)

print("\nReplacement complete.")
print("Source 2 index size:",
      round(os.path.getsize(original) / (1024**2), 2), "MB")
print("Free space:",
      round(shutil.disk_usage(".").free / (1024**3), 2), "GB")

Before:
Original exists: True
Filtered exists: True

Replacement complete.
Source 2 index size: 872.75 MB
Free space: 2.45 GB


In [4]:
import duckdb

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=2")

print("Checking Source 2 key frequencies...")

result = con.execute("""
SELECT
    CASE
        WHEN block_key LIKE '%|name|%' THEN 'name'
        WHEN block_key LIKE '%|name2|%' THEN 'name2'
        WHEN block_key LIKE '%|name1|%' THEN 'name1'
        WHEN block_key LIKE '%|addr_first4|%' THEN 'addr_first4'
        WHEN block_key LIKE '%|addr_last3|%' THEN 'addr_last3'
        WHEN block_key LIKE '%|addr_nums|%' THEN 'addr_nums'
        ELSE 'other'
    END AS key_type,
    COUNT(*) AS rows,
    COUNT(DISTINCT block_key) AS distinct_keys,
    MAX(cnt) AS largest_key
FROM (
    SELECT
        block_key,
        COUNT(*) AS cnt
    FROM read_csv(
        'candidate_index/source2_index.tsv',
        delim='\t',
        header=true,
        columns={
            'block_key':'VARCHAR',
            'entity_id':'VARCHAR'
        }
    )
    GROUP BY block_key
) x
GROUP BY key_type
ORDER BY rows DESC
""").fetchdf()

print(result.to_string(index=False))

con.close()

Checking Source 2 key frequencies...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   key_type    rows  distinct_keys  largest_key
       name 3931312        3931312          300
addr_first4 3477466        3477466         4171
      name2 1789116        1789116        29181
 addr_last3 1004371        1004371        81688
  addr_nums  276210         276210         1695


In [5]:
import duckdb

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=2")

print("Checking Source 3 key frequencies...")

result = con.execute("""
SELECT
    CASE
        WHEN block_key LIKE '%|name|%' THEN 'name'
        WHEN block_key LIKE '%|name2|%' THEN 'name2'
        WHEN block_key LIKE '%|name1|%' THEN 'name1'
        WHEN block_key LIKE '%|addr_first4|%' THEN 'addr_first4'
        WHEN block_key LIKE '%|addr_last3|%' THEN 'addr_last3'
        WHEN block_key LIKE '%|addr_nums|%' THEN 'addr_nums'
        ELSE 'other'
    END AS key_type,
    COUNT(*) AS rows,
    COUNT(DISTINCT block_key) AS distinct_keys,
    MAX(cnt) AS largest_key
FROM (
    SELECT
        block_key,
        COUNT(*) AS cnt
    FROM read_csv(
        'candidate_index/source3_index.tsv',
        delim='\t',
        header=true,
        columns={
            'block_key':'VARCHAR',
            'entity_id':'VARCHAR'
        }
    )
    GROUP BY block_key
) x
GROUP BY key_type
ORDER BY rows DESC
""").fetchdf()

print(result.to_string(index=False))

con.close()

Checking Source 3 key frequencies...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 key_type    rows  distinct_keys  largest_key
    other 1231480        1231480        16790
     name  862245         862245           76
    name2  600336         600336         6681
    name1  145938         145938        11015
addr_nums   79173          79173          306


In [6]:
import duckdb

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=2")

print("Estimating safe candidate volume...")
print("No output file will be created.")

query = """
WITH
s1 AS (
    SELECT source1_entity_id, block_key
    FROM read_csv(
        'candidate_index/source1_keys.tsv',
        delim='\t',
        header=true,
        columns={
            'source1_entity_id':'VARCHAR',
            'block_key':'VARCHAR'
        }
    )
),

s2 AS (
    SELECT block_key, entity_id
    FROM read_csv(
        'candidate_index/source2_index.tsv',
        delim='\t',
        header=true,
        columns={
            'block_key':'VARCHAR',
            'entity_id':'VARCHAR'
        }
    )
),

s3 AS (
    SELECT block_key, entity_id
    FROM read_csv(
        'candidate_index/source3_index.tsv',
        delim='\t',
        header=true,
        columns={
            'block_key':'VARCHAR',
            'entity_id':'VARCHAR'
        }
    )
),

s2_freq AS (
    SELECT block_key, COUNT(*) AS n
    FROM s2
    GROUP BY block_key
),

s3_freq AS (
    SELECT block_key, COUNT(*) AS n
    FROM s3
    GROUP BY block_key
),

pairs2 AS (
    SELECT s1.source1_entity_id, s2.entity_id
    FROM s1
    JOIN s2 ON s1.block_key = s2.block_key
    JOIN s2_freq f ON s2.block_key = f.block_key
    WHERE
        (
            s1.block_key LIKE '%|name|%'
            AND f.n <= 300
        )
        OR
        (
            s1.block_key LIKE '%|name2|%'
            AND f.n <= 500
        )
        OR
        (
            s1.block_key LIKE '%|addr_first4|%'
            AND f.n <= 500
        )
        OR
        (
            s1.block_key LIKE '%|addr_last3|%'
            AND f.n <= 500
        )
        OR
        (
            s1.block_key LIKE '%|addr_nums|%'
            AND f.n <= 500
        )
),

pairs3 AS (
    SELECT s1.source1_entity_id, s3.entity_id
    FROM s1
    JOIN s3 ON s1.block_key = s3.block_key
    JOIN s3_freq f ON s3.block_key = f.block_key
    WHERE
        (
            s1.block_key LIKE '%|name|%'
            AND f.n <= 100
        )
        OR
        (
            s1.block_key LIKE '%|name2|%'
            AND f.n <= 500
        )
        OR
        (
            s1.block_key LIKE '%|addr_nums|%'
            AND f.n <= 500
        )
        OR
        (
            s1.block_key NOT LIKE '%|name1|%'
            AND s1.block_key NOT LIKE '%|name2|%'
            AND s1.block_key NOT LIKE '%|name|%'
            AND f.n <= 500
        )
)

SELECT
    COUNT(*) AS raw_candidate_pairs,
    COUNT(DISTINCT source1_entity_id) AS source1_entities_with_candidates,
    COUNT(DISTINCT entity_id) AS distinct_candidate_entities
FROM (
    SELECT * FROM pairs2
    UNION ALL
    SELECT * FROM pairs3
) x
"""

result = con.execute(query).fetchdf()

print(result.to_string(index=False))

con.close()

Estimating safe candidate volume...
No output file will be created.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OutOfMemoryException: Out of Memory Error: failed to offload data block of size 256.0 KiB (3.1 GiB/3.1 GiB used).
This limit was set by the 'max_temp_directory_size' setting.
By default, this setting utilizes the available disk space on the drive where the 'temp_directory' is located.
You can adjust this setting, by using (for example) PRAGMA max_temp_directory_size='10GiB'

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

In [7]:
import os
import shutil

print("Checking temporary files...")

if os.path.exists("candidate_index"):
    for name in os.listdir("candidate_index"):
        if name.startswith("duckdb"):
            path = os.path.join("candidate_index", name)
            if os.path.isdir(path):
                shutil.rmtree(path)
                print("Removed:", name)

print(
    "Free space:",
    round(shutil.disk_usage(".").free / (1024**3), 2),
    "GB"
)

Checking temporary files...
Free space: 3.45 GB


In [8]:
import duckdb
import shutil

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")

print("Testing exact-name join only...")

result = con.execute("""
SELECT COUNT(*) AS candidate_pairs
FROM read_csv(
    'candidate_index/source1_keys.tsv',
    delim='\t',
    header=true,
    columns={
        'source1_entity_id':'VARCHAR',
        'block_key':'VARCHAR'
    }
) s1
JOIN read_csv(
    'candidate_index/source2_index.tsv',
    delim='\t',
    header=true,
    columns={
        'block_key':'VARCHAR',
        'entity_id':'VARCHAR'
    }
) s2
ON s1.block_key = s2.block_key
WHERE s1.block_key LIKE '%|name|%'
""").fetchone()[0]

print("Source 1 → Source 2 exact-name candidate pairs:", f"{result:,}")
print("Free space:", round(shutil.disk_usage(".").free/(1024**3), 2), "GB")

con.close()

Testing exact-name join only...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Source 1 → Source 2 exact-name candidate pairs: 5,719,476
Free space: 3.45 GB


In [9]:
import duckdb
import shutil

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")

print("Testing exact-name join: Source 1 → Source 3...")

result = con.execute("""
SELECT COUNT(*) AS candidate_pairs
FROM read_csv(
    'candidate_index/source1_keys.tsv',
    delim='\t',
    header=true,
    columns={
        'source1_entity_id':'VARCHAR',
        'block_key':'VARCHAR'
    }
) s1
JOIN read_csv(
    'candidate_index/source3_index.tsv',
    delim='\t',
    header=true,
    columns={
        'block_key':'VARCHAR',
        'entity_id':'VARCHAR'
    }
) s3
ON s1.block_key = s3.block_key
WHERE s1.block_key LIKE '%|name|%'
""").fetchone()[0]

print("Source 1 → Source 3 exact-name candidate pairs:", f"{result:,}")
print("Free space:", round(shutil.disk_usage(".").free/(1024**3), 2), "GB")

con.close()

Testing exact-name join: Source 1 → Source 3...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Source 1 → Source 3 exact-name candidate pairs: 1,400,827
Free space: 3.45 GB


In [10]:
import duckdb
import os
import shutil

RAW_FILE = "candidate_index/exact_name_candidates.tsv"

if os.path.exists(RAW_FILE):
    os.remove(RAW_FILE)

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")

print("Writing exact-name candidates...")

con.execute("""
COPY (
    SELECT
        s1.source1_entity_id,
        s2.entity_id AS candidate_entity_id
    FROM read_csv(
        'candidate_index/source1_keys.tsv',
        delim='\t',
        header=true,
        columns={
            'source1_entity_id':'VARCHAR',
            'block_key':'VARCHAR'
        }
    ) s1
    JOIN read_csv(
        'candidate_index/source2_index.tsv',
        delim='\t',
        header=true,
        columns={
            'block_key':'VARCHAR',
            'entity_id':'VARCHAR'
        }
    ) s2
    ON s1.block_key = s2.block_key
    WHERE s1.block_key LIKE '%|name|%'

    UNION ALL

    SELECT
        s1.source1_entity_id,
        s3.entity_id AS candidate_entity_id
    FROM read_csv(
        'candidate_index/source1_keys.tsv',
        delim='\t',
        header=true,
        columns={
            'source1_entity_id':'VARCHAR',
            'block_key':'VARCHAR'
        }
    ) s1
    JOIN read_csv(
        'candidate_index/source3_index.tsv',
        delim='\t',
        header=true,
        columns={
            'block_key':'VARCHAR',
            'entity_id':'VARCHAR'
        }
    ) s3
    ON s1.block_key = s3.block_key
    WHERE s1.block_key LIKE '%|name|%'
) TO 'candidate_index/exact_name_candidates.tsv'
WITH (
    HEADER,
    DELIMITER '\t'
)
""")

con.close()

size_mb = os.path.getsize(RAW_FILE) / (1024**2)
free_gb = shutil.disk_usage(".").free / (1024**3)

print("DONE")
print("File size:", round(size_mb, 2), "MB")
print("Free space:", round(free_gb, 2), "GB")

Writing exact-name candidates...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DONE
File size: 175.05 MB
Free space: 3.28 GB


In [11]:
import duckdb
import shutil

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")

print("Testing address-number candidates...")

result = con.execute("""
SELECT COUNT(*) AS candidate_pairs
FROM read_csv(
    'candidate_index/source1_keys.tsv',
    delim='\t',
    header=true,
    columns={
        'source1_entity_id':'VARCHAR',
        'block_key':'VARCHAR'
    }
) s1
JOIN read_csv(
    'candidate_index/source2_index.tsv',
    delim='\t',
    header=true,
    columns={
        'block_key':'VARCHAR',
        'entity_id':'VARCHAR'
    }
) s2
ON s1.block_key = s2.block_key
WHERE s1.block_key LIKE '%|addr_nums|%'
""").fetchone()[0]

result3 = con.execute("""
SELECT COUNT(*) AS candidate_pairs
FROM read_csv(
    'candidate_index/source1_keys.tsv',
    delim='\t',
    header=true,
    columns={
        'source1_entity_id':'VARCHAR',
        'block_key':'VARCHAR'
    }
) s1
JOIN read_csv(
    'candidate_index/source3_index.tsv',
    delim='\t',
    header=true,
    columns={
        'block_key':'VARCHAR',
        'entity_id':'VARCHAR'
    }
) s3
ON s1.block_key = s3.block_key
WHERE s1.block_key LIKE '%|addr_nums|%'
""").fetchone()[0]

print("Source 1 → Source 2 addr_nums candidates:", f"{result:,}")
print("Source 1 → Source 3 addr_nums candidates:", f"{result3:,}")
print("Total:", f"{result + result3:,}")
print("Free space:", round(shutil.disk_usage(".").free/(1024**3), 2), "GB")

con.close()

Testing address-number candidates...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Source 1 → Source 2 addr_nums candidates: 2,264,516
Source 1 → Source 3 addr_nums candidates: 413,574
Total: 2,678,090
Free space: 3.28 GB


In [12]:
import duckdb
import os
import shutil

RAW_FILE = "candidate_index/addr_nums_candidates.tsv"

if os.path.exists(RAW_FILE):
    os.remove(RAW_FILE)

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")

print("Writing address-number candidates...")

con.execute("""
COPY (
    SELECT
        s1.source1_entity_id,
        s2.entity_id AS candidate_entity_id
    FROM read_csv(
        'candidate_index/source1_keys.tsv',
        delim='\t',
        header=true,
        columns={
            'source1_entity_id':'VARCHAR',
            'block_key':'VARCHAR'
        }
    ) s1
    JOIN read_csv(
        'candidate_index/source2_index.tsv',
        delim='\t',
        header=true,
        columns={
            'block_key':'VARCHAR',
            'entity_id':'VARCHAR'
        }
    ) s2
    ON s1.block_key = s2.block_key
    WHERE s1.block_key LIKE '%|addr_nums|%'

    UNION ALL

    SELECT
        s1.source1_entity_id,
        s3.entity_id AS candidate_entity_id
    FROM read_csv(
        'candidate_index/source1_keys.tsv',
        delim='\t',
        header=true,
        columns={
            'source1_entity_id':'VARCHAR',
            'block_key':'VARCHAR'
        }
    ) s1
    JOIN read_csv(
        'candidate_index/source3_index.tsv',
        delim='\t',
        header=true,
        columns={
            'block_key':'VARCHAR',
            'entity_id':'VARCHAR'
        }
    ) s3
    ON s1.block_key = s3.block_key
    WHERE s1.block_key LIKE '%|addr_nums|%'
) TO 'candidate_index/addr_nums_candidates.tsv'
WITH (
    HEADER,
    DELIMITER '\t'
)
""")

con.close()

size_mb = os.path.getsize(RAW_FILE) / (1024**2)
free_gb = shutil.disk_usage(".").free / (1024**3)

print("DONE")
print("File size:", round(size_mb, 2), "MB")
print("Free space:", round(free_gb, 2), "GB")

Writing address-number candidates...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DONE
File size: 65.84 MB
Free space: 3.21 GB


In [13]:
import os

for f in [
    "candidate_index/exact_name_candidates.tsv",
    "candidate_index/addr_nums_candidates.tsv"
]:
    print(f, "->", round(os.path.getsize(f)/(1024**2), 2), "MB")

candidate_index/exact_name_candidates.tsv -> 175.05 MB
candidate_index/addr_nums_candidates.tsv -> 65.84 MB


In [14]:
import duckdb
import shutil

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")

print("Testing name2 candidates...")

s2 = con.execute("""
SELECT COUNT(*)
FROM read_csv(
    'candidate_index/source1_keys.tsv',
    delim='\t',
    header=true,
    columns={
        'source1_entity_id':'VARCHAR',
        'block_key':'VARCHAR'
    }
) s1
JOIN read_csv(
    'candidate_index/source2_index.tsv',
    delim='\t',
    header=true,
    columns={
        'block_key':'VARCHAR',
        'entity_id':'VARCHAR'
    }
) s2
ON s1.block_key = s2.block_key
WHERE s1.block_key LIKE '%|name2|%'
""").fetchone()[0]

s3 = con.execute("""
SELECT COUNT(*)
FROM read_csv(
    'candidate_index/source1_keys.tsv',
    delim='\t',
    header=true,
    columns={
        'source1_entity_id':'VARCHAR',
        'block_key':'VARCHAR'
    }
) s1
JOIN read_csv(
    'candidate_index/source3_index.tsv',
    delim='\t',
    header=true,
    columns={
        'block_key':'VARCHAR',
        'entity_id':'VARCHAR'
    }
) s3
ON s1.block_key = s3.block_key
WHERE s1.block_key LIKE '%|name2|%'
""").fetchone()[0]

print("Source 1 → Source 2 name2 candidates:", f"{s2:,}")
print("Source 1 → Source 3 name2 candidates:", f"{s3:,}")
print("Total:", f"{s2+s3:,}")
print("Free space:", round(shutil.disk_usage(".").free/(1024**3), 2), "GB")

con.close()

Testing name2 candidates...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Source 1 → Source 2 name2 candidates: 383,585,453
Source 1 → Source 3 name2 candidates: 78,499,129
Total: 462,084,582
Free space: 3.21 GB


In [15]:
import duckdb
import shutil

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")

print("Checking candidate overlap...")

result = con.execute("""
SELECT
    COUNT(*) AS total_pairs,
    COUNT(DISTINCT source1_entity_id || '|' || candidate_entity_id) AS unique_pairs
FROM (
    SELECT source1_entity_id, candidate_entity_id
    FROM read_csv(
        'candidate_index/exact_name_candidates.tsv',
        delim='\t',
        header=true,
        columns={
            'source1_entity_id':'VARCHAR',
            'candidate_entity_id':'VARCHAR'
        }
    )

    UNION ALL

    SELECT source1_entity_id, candidate_entity_id
    FROM read_csv(
        'candidate_index/addr_nums_candidates.tsv',
        delim='\t',
        header=true,
        columns={
            'source1_entity_id':'VARCHAR',
            'candidate_entity_id':'VARCHAR'
        }
    )
)
""").fetchone()

print("Total pairs:", f"{result[0]:,}")
print("Unique pairs:", f"{result[1]:,}")
print("Duplicate pairs:", f"{result[0] - result[1]:,}")
print("Free space:", round(shutil.disk_usage(".").free/(1024**3), 2), "GB")

con.close()

Checking candidate overlap...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pairs: 9,798,393
Unique pairs: 9,765,660
Duplicate pairs: 32,733
Free space: 3.21 GB


In [16]:
import duckdb
import os
import shutil

FINAL_CANDIDATES = "candidate_index/unique_candidates.tsv"

if os.path.exists(FINAL_CANDIDATES):
    os.remove(FINAL_CANDIDATES)

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")

print("Creating deduplicated candidate pairs...")

con.execute("""
COPY (
    SELECT DISTINCT
        source1_entity_id,
        candidate_entity_id
    FROM (
        SELECT
            source1_entity_id,
            candidate_entity_id
        FROM read_csv(
            'candidate_index/exact_name_candidates.tsv',
            delim='\t',
            header=true,
            columns={
                'source1_entity_id':'VARCHAR',
                'candidate_entity_id':'VARCHAR'
            }
        )

        UNION ALL

        SELECT
            source1_entity_id,
            candidate_entity_id
        FROM read_csv(
            'candidate_index/addr_nums_candidates.tsv',
            delim='\t',
            header=true,
            columns={
                'source1_entity_id':'VARCHAR',
                'candidate_entity_id':'VARCHAR'
            }
        )
    )
) TO 'candidate_index/unique_candidates.tsv'
WITH (
    HEADER,
    DELIMITER '\t'
)
""")

con.close()

size_mb = os.path.getsize(FINAL_CANDIDATES) / (1024**2)
free_gb = shutil.disk_usage(".").free / (1024**3)

print("DONE")
print("Unique candidate file:", round(size_mb, 2), "MB")
print("Free space:", round(free_gb, 2), "GB")

Creating deduplicated candidate pairs...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DONE
Unique candidate file: 240.09 MB
Free space: 2.98 GB


In [17]:
import duckdb

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")

checks = [
    ("S1-504790211", "S2-482219248"),
    ("S1-564729135", "S2-327309238"),
    ("S1-564729135", "S3-912418512"),
    ("S1-82053739", "S3-250737963"),
]

for s1_id, candidate_id in checks:
    found = con.execute("""
        SELECT COUNT(*)
        FROM read_csv(
            'candidate_index/unique_candidates.tsv',
            delim='\t',
            header=true,
            columns={
                'source1_entity_id':'VARCHAR',
                'candidate_entity_id':'VARCHAR'
            }
        )
        WHERE source1_entity_id = ?
          AND candidate_entity_id = ?
    """, [s1_id, candidate_id]).fetchone()[0]

    print(
        s1_id,
        "->",
        candidate_id,
        ":",
        "RECOVERED" if found else "NOT RECOVERED"
    )

con.close()

S1-504790211 -> S2-482219248 : NOT RECOVERED
S1-564729135 -> S2-327309238 : NOT RECOVERED
S1-564729135 -> S3-912418512 : NOT RECOVERED
S1-82053739 -> S3-250737963 : NOT RECOVERED


In [19]:
import duckdb

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")

pairs = [
    ("S1-504790211", "S2-482219248"),
    ("S1-564729135", "S2-327309238"),
    ("S1-564729135", "S3-912418512"),
    ("S1-82053739", "S3-250737963"),
]

print("Checking blocking keys for the 4 difficult pairs...\n")

# Get Source 1 keys
for s1_id, candidate_id in pairs:
    s1_keys = con.execute("""
        SELECT block_key
        FROM read_csv(
            'candidate_index/source1_keys.tsv',
            delim='\t',
            header=true,
            columns={
                'source1_entity_id':'VARCHAR',
                'block_key':'VARCHAR'
            }
        )
        WHERE source1_entity_id = ?
    """, [s1_id]).fetchall()

    # Choose the correct index
    index_file = (
        'candidate_index/source2_index.tsv'
        if candidate_id.startswith("S2-")
        else 'candidate_index/source3_index.tsv'
    )

    candidate_keys = con.execute("""
        SELECT block_key
        FROM read_csv(
            ?,
            delim='\t',
            header=true,
            columns={
                'block_key':'VARCHAR',
                'entity_id':'VARCHAR'
            }
        )
        WHERE entity_id = ?
    """, [index_file, candidate_id]).fetchall()

    s1_set = {x[0] for x in s1_keys}
    candidate_set = {x[0] for x in candidate_keys}
    common = sorted(s1_set & candidate_set)

    print("=" * 70)
    print(s1_id, "->", candidate_id)
    print("Common keys:", common if common else "NONE")

    print("\nSource 1 keys:")
    for k in sorted(s1_set):
        print(" ", k)

    print("\nCandidate keys:")
    for k in sorted(candidate_set):
        print(" ", k)

con.close()

Checking blocking keys for the 4 difficult pairs...



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S1-504790211 -> S2-482219248
Common keys: NONE

Source 1 keys:

Candidate keys:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S1-564729135 -> S2-327309238
Common keys: NONE

Source 1 keys:

Candidate keys:
S1-564729135 -> S3-912418512
Common keys: NONE

Source 1 keys:

Candidate keys:
S1-82053739 -> S3-250737963
Common keys: NONE

Source 1 keys:

Candidate keys:


In [ ]:
import pandas as pd
from rapidfuzz.fuzz import ratio, token_set_ratio
import unidecode
import re

# Read only 10,000 candidate pairs
candidates = pd.read_csv(
    "candidate_index/unique_candidates.tsv",
    sep="\t",
    nrows=10000
)

print("Loaded candidates:", len(candidates))

# Load the corresponding Source 1 and candidate records
s1 = pd.read_csv(
    "s3://sagemaker-eu-north-1-152025327817/dataset/test/test_source1.tsv",
    sep="\t",
    usecols=["entity_id", "business_name", "business_address", "country"]
)

# Only load candidate IDs appearing in our 10k sample
candidate_ids = set(candidates["candidate_entity_id"])

s2 = pd.read_csv(
    "s3://sagemaker-eu-north-1-152025327817/dataset/test/test_source2.tsv",
    sep="\t",
    usecols=["entity_id", "business_name", "business_address", "country"]
)

s3 = pd.read_csv(
    "s3://sagemaker-eu-north-1-152025327817/dataset/test/test_source3.tsv",
    sep="\t",
    usecols=["entity_id", "business_name", "business_address", "country"]
)

# Keep only records needed for this small test
s1 = s1[s1["entity_id"].isin(set(candidates["source1_entity_id"]))].copy()
s2 = s2[s2["entity_id"].isin(candidate_ids)].copy()
s3 = s3[s3["entity_id"].isin(candidate_ids)].copy()

s2 = pd.concat([s2, s3], ignore_index=True)

print("Source 1 records loaded:", len(s1))
print("Candidate records loaded:", len(s2))

def norm(text):
    if pd.isna(text):
        return ""
    text = unidecode.unidecode(str(text)).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

# Dictionaries for fast lookup
s1_map = s1.set_index("entity_id").to_dict("index")
cand_map = s2.set_index("entity_id").to_dict("index")

results = []

for row in candidates.itertuples(index=False):
    a = s1_map.get(row.source1_entity_id)
    b = cand_map.get(row.candidate_entity_id)

    if a is None or b is None:
        continue

    name_a = norm(a["business_name"])
    name_b = norm(b["business_name"])

    addr_a = norm(a["business_address"])
    addr_b = norm(b["business_address"])

    name_score = token_set_ratio(name_a, name_b)
    address_score = token_set_ratio(addr_a, addr_b)

    # Conservative combined score
    score = 0.6 * name_score + 0.4 * address_score

    results.append({
        "source1_entity_id": row.source1_entity_id,
        "candidate_entity_id": row.candidate_entity_id,
        "name_score": round(name_score, 1),
        "address_score": round(address_score, 1),
        "combined_score": round(score, 1)
    })

scores = pd.DataFrame(results)

print("\nScored pairs:", len(scores))
print("\nScore summary:")
print(scores["combined_score"].describe())

print("\nTop 20 candidates:")
print(
    scores.sort_values("combined_score", ascending=False)
          .head(20)
          .to_string(index=False)
)

Loaded candidates: 10000


In [1]:
print("Stopped safely. No candidate files were modified.")

Stopped safely. No candidate files were modified.


In [2]:
import duckdb
import pandas as pd
from rapidfuzz.fuzz import token_set_ratio
import unidecode
import re

con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")

# Read only 1,000 candidate pairs
candidates = pd.read_csv(
    "candidate_index/unique_candidates.tsv",
    sep="\t",
    nrows=1000
)

print("Candidate pairs:", len(candidates))

# Create a temporary candidate table
con.register("candidate_sample", candidates)

# Retrieve ONLY the required Source 1 records
s1 = con.execute("""
SELECT
    s.entity_id,
    s.business_name,
    s.business_address,
    s.country
FROM read_csv(
    's3://sagemaker-eu-north-1-152025327817/dataset/test/test_source1.tsv',
    delim='\t',
    header=true
) s
WHERE s.entity_id IN (
    SELECT DISTINCT source1_entity_id
    FROM candidate_sample
)
""").df()

print("Source 1 records:", len(s1))

# Retrieve ONLY the required S2/S3 records
others = con.execute("""
SELECT
    s.entity_id,
    s.business_name,
    s.business_address,
    s.country
FROM read_csv(
    's3://sagemaker-eu-north-1-152025327817/dataset/test/test_source2.tsv',
    delim='\t',
    header=true
) s
WHERE s.entity_id IN (
    SELECT DISTINCT candidate_entity_id
    FROM candidate_sample
)

UNION ALL

SELECT
    s.entity_id,
    s.business_name,
    s.business_address,
    s.country
FROM read_csv(
    's3://sagemaker-eu-north-1-152025327817/dataset/test/test_source3.tsv',
    delim='\t',
    header=true
) s
WHERE s.entity_id IN (
    SELECT DISTINCT candidate_entity_id
    FROM candidate_sample
)
""").df()

print("Candidate records:", len(others))

def norm(text):
    if pd.isna(text):
        return ""
    text = unidecode.unidecode(str(text)).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

s1_map = s1.set_index("entity_id").to_dict("index")
other_map = others.set_index("entity_id").to_dict("index")

scores = []

for row in candidates.itertuples(index=False):
    a = s1_map.get(row.source1_entity_id)
    b = other_map.get(row.candidate_entity_id)

    if a is None or b is None:
        continue

    name_score = token_set_ratio(
        norm(a["business_name"]),
        norm(b["business_name"])
    )

    address_score = token_set_ratio(
        norm(a["business_address"]),
        norm(b["business_address"])
    )

    combined = 0.6 * name_score + 0.4 * address_score

    scores.append({
        "source1_entity_id": row.source1_entity_id,
        "candidate_entity_id": row.candidate_entity_id,
        "name_score": round(name_score, 1),
        "address_score": round(address_score, 1),
        "combined_score": round(combined, 1)
    })

scores = pd.DataFrame(scores)

print("\nScored pairs:", len(scores))
print("\nTop 20:")
print(
    scores.sort_values(
        "combined_score",
        ascending=False
    ).head(20).to_string(index=False)
)

con.close()

Candidate pairs: 1000


HTTPException: HTTP Error: HTTP GET error reading 'https://sagemaker-eu-north-1-152025327817.s3.eu-north-1.amazonaws.com/dataset/test/test_source1.tsv' in region 'eu-north-1' (HTTP 403 Forbidden)

AccessDenied: Access Denied

Authentication Failure - this is usually caused by invalid or missing credentials.
* No credentials are provided.
* See https://duckdb.org/docs/stable/extensions/httpfs/s3api.html

Authentication Failure - this is usually caused by invalid or missing credentials.
* No credentials are provided.
* See https://duckdb.org/docs/stable/extensions/httpfs/s3api.html

LINE 7: FROM read_csv(
             ^

In [3]:
import boto3

s3 = boto3.client("s3", region_name="eu-north-1")

bucket = "sagemaker-eu-north-1-152025327817"
key = "dataset/test/test_source1.tsv"

obj = s3.head_object(Bucket=bucket, Key=key)

print("S3 access: OK")
print("File size:", round(obj["ContentLength"] / (1024**3), 2), "GB")

S3 access: OK
File size: 0.16 GB


In [4]:
import boto3
import pandas as pd
import io
import re
import unidecode
from rapidfuzz.fuzz import token_set_ratio

BUCKET = "sagemaker-eu-north-1-152025327817"

# --------------------------------------------------
# 1. Read a small candidate sample
# --------------------------------------------------

candidates = pd.read_csv(
    "candidate_index/unique_candidates.tsv",
    sep="\t",
    nrows=10000
)

s1_ids = set(candidates["source1_entity_id"])
candidate_ids = set(candidates["candidate_entity_id"])

print("Candidate pairs:", len(candidates))
print("Source 1 IDs needed:", len(s1_ids))
print("Candidate IDs needed:", len(candidate_ids))


# --------------------------------------------------
# 2. Function to scan one S3 TSV in chunks
# --------------------------------------------------

s3 = boto3.client("s3", region_name="eu-north-1")

def collect_records(key, wanted_ids, chunk_size=100000):
    found = {}

    obj = s3.get_object(Bucket=BUCKET, Key=key)

    for chunk in pd.read_csv(
        obj["Body"],
        sep="\t",
        chunksize=chunk_size,
        usecols=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    ):
        mask = chunk["entity_id"].isin(wanted_ids)

        if mask.any():
            selected = chunk.loc[mask]

            for row in selected.itertuples(index=False):
                found[row.entity_id] = {
                    "business_name": row.business_name,
                    "business_address": row.business_address,
                    "country": row.country
                }

        if len(found) == len(wanted_ids):
            break

    return found


# --------------------------------------------------
# 3. Collect Source 1 records
# --------------------------------------------------

print("\nReading required Source 1 records...")

s1_map = collect_records(
    "dataset/test/test_source1.tsv",
    s1_ids
)

print("Source 1 records found:", len(s1_map), "/", len(s1_ids))


# --------------------------------------------------
# 4. Split S2/S3 IDs
# --------------------------------------------------

s2_ids = {x for x in candidate_ids if str(x).startswith("S2-")}
s3_ids = {x for x in candidate_ids if str(x).startswith("S3-")}

print("S2 IDs needed:", len(s2_ids))
print("S3 IDs needed:", len(s3_ids))


# --------------------------------------------------
# 5. Collect Source 2 records
# --------------------------------------------------

print("\nReading required Source 2 records...")

s2_map = collect_records(
    "dataset/test/test_source2.tsv",
    s2_ids
)

print("Source 2 records found:", len(s2_map), "/", len(s2_ids))


# --------------------------------------------------
# 6. Collect Source 3 records
# --------------------------------------------------

print("\nReading required Source 3 records...")

s3_map = collect_records(
    "dataset/test/test_source3.tsv",
    s3_ids
)

print("Source 3 records found:", len(s3_map), "/", len(s3_ids))


# --------------------------------------------------
# 7. Normalization
# --------------------------------------------------

def norm(text):
    if pd.isna(text):
        return ""

    text = unidecode.unidecode(str(text)).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


# --------------------------------------------------
# 8. Score the 10,000 candidates
# --------------------------------------------------

scores = []

for row in candidates.itertuples(index=False):

    a = s1_map.get(row.source1_entity_id)

    if str(row.candidate_entity_id).startswith("S2-"):
        b = s2_map.get(row.candidate_entity_id)
    else:
        b = s3_map.get(row.candidate_entity_id)

    if a is None or b is None:
        continue

    name_score = token_set_ratio(
        norm(a["business_name"]),
        norm(b["business_name"])
    )

    address_score = token_set_ratio(
        norm(a["business_address"]),
        norm(b["business_address"])
    )

    combined_score = (
        0.6 * name_score +
        0.4 * address_score
    )

    scores.append({
        "source1_entity_id": row.source1_entity_id,
        "candidate_entity_id": row.candidate_entity_id,
        "name_score": round(name_score, 1),
        "address_score": round(address_score, 1),
        "combined_score": round(combined_score, 1)
    })


scores = pd.DataFrame(scores)

print("\n-----------------------------")
print("SCORING TEST COMPLETE")
print("-----------------------------")
print("Scored pairs:", len(scores))

if len(scores):
    print("\nScore summary:")
    print(scores["combined_score"].describe())

    print("\nTop 20:")
    print(
        scores.sort_values(
            "combined_score",
            ascending=False
        ).head(20).to_string(index=False)
    )

Candidate pairs: 10000
Source 1 IDs needed: 9835
Candidate IDs needed: 1616

Reading required Source 1 records...
Source 1 records found: 9835 / 9835
S2 IDs needed: 1616
S3 IDs needed: 0

Reading required Source 2 records...
Source 2 records found: 1616 / 1616

Reading required Source 3 records...
Source 3 records found: 0 / 0

-----------------------------
SCORING TEST COMPLETE
-----------------------------
Scored pairs: 10000

Score summary:
count    10000.000000
mean        79.507190
std          9.040882
min         60.000000
25%         73.900000
50%         76.600000
75%         83.400000
max        100.000000
Name: combined_score, dtype: float64

Top 20:
source1_entity_id candidate_entity_id  name_score  address_score  combined_score
     S1-352260214        S2-473996463       100.0          100.0           100.0
     S1-837928799        S2-491607559       100.0          100.0           100.0
     S1-966084304        S2-210791875       100.0          100.0           100.0
      

In [5]:
import pandas as pd

GT = pd.read_csv(
    "s3://sagemaker-eu-north-1-152025327817/dataset/train/train_ground_truth.tsv",
    sep="\t"
)

print("Ground-truth rows:", len(GT))
print(GT.head(10).to_string(index=False))

Ground-truth rows: 2206821
source1_entity_id                                                            matched_entity_ids
        S1-965667               S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364
      S1-55344266                           S2-249013014,S2-197070651,S3-478195123,S3-384364074
     S1-343815751                                        S2-790675320,S2-479876582,S3-878454467
     S1-656753428                                         S2-153058913,S2-24659151,S3-679606215
     S1-102811957 S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785
      S1-18727616                           S2-755677256,S3-187831601,S3-641489370,S3-476250621
     S1-318373630                                                     S2-660036492,S3-804600254
      S1-86989137                                                     S3-274817120,S3-312496301
      S1-29845983                                                     S2-648035184,S3-588502663
     S1-78900

In [6]:
import boto3
import pandas as pd
import re
import unidecode
from rapidfuzz.fuzz import token_set_ratio

BUCKET = "sagemaker-eu-north-1-152025327817"

# --------------------------------------------------
# 1. Take 100 labeled training entities
# --------------------------------------------------

gt = pd.read_csv(
    "s3://sagemaker-eu-north-1-152025327817/dataset/train/train_ground_truth.tsv",
    sep="\t",
    nrows=100
)

# Collect all true matched S2/S3 IDs
true_ids = set()

for value in gt["matched_entity_ids"].fillna(""):
    if str(value).strip():
        true_ids.update(str(value).split(","))

s1_ids = set(gt["source1_entity_id"])

print("Training Source 1 entities:", len(s1_ids))
print("True matched S2/S3 records:", len(true_ids))


# --------------------------------------------------
# 2. Authenticated S3 chunk reader
# --------------------------------------------------

s3 = boto3.client("s3", region_name="eu-north-1")

def collect_records(key, wanted_ids, chunk_size=100000):
    found = {}

    obj = s3.get_object(
        Bucket=BUCKET,
        Key=key
    )

    for chunk in pd.read_csv(
        obj["Body"],
        sep="\t",
        chunksize=chunk_size,
        usecols=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    ):
        selected = chunk[chunk["entity_id"].isin(wanted_ids)]

        for row in selected.itertuples(index=False):
            found[row.entity_id] = {
                "business_name": row.business_name,
                "business_address": row.business_address,
                "country": row.country
            }

        if len(found) == len(wanted_ids):
            break

    return found


# --------------------------------------------------
# 3. Get required Source 1 records
# --------------------------------------------------

print("\nReading Source 1...")

s1_map = collect_records(
    "dataset/train/train_source1.tsv",
    s1_ids
)

print("Source 1 found:", len(s1_map), "/", len(s1_ids))


# --------------------------------------------------
# 4. Get required S2/S3 records
# --------------------------------------------------

s2_ids = {x for x in true_ids if x.startswith("S2-")}
s3_ids = {x for x in true_ids if x.startswith("S3-")}

print("S2 records needed:", len(s2_ids))
print("S3 records needed:", len(s3_ids))

print("\nReading Source 2...")

s2_map = collect_records(
    "dataset/train/train_source2.tsv",
    s2_ids
)

print("Source 2 found:", len(s2_map), "/", len(s2_ids))

print("\nReading Source 3...")

s3_map = collect_records(
    "dataset/train/train_source3.tsv",
    s3_ids
)

print("Source 3 found:", len(s3_map), "/", len(s3_ids))


# --------------------------------------------------
# 5. Normalize + score true matches
# --------------------------------------------------

def norm(text):
    if pd.isna(text):
        return ""

    text = unidecode.unidecode(str(text)).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


scores = []

for row in gt.itertuples(index=False):

    source1_id = row.source1_entity_id
    a = s1_map.get(source1_id)

    if a is None:
        continue

    matched = str(row.matched_entity_ids)

    if not matched or matched == "nan":
        continue

    for candidate_id in matched.split(","):

        if candidate_id.startswith("S2-"):
            b = s2_map.get(candidate_id)
        else:
            b = s3_map.get(candidate_id)

        if b is None:
            continue

        name_score = token_set_ratio(
            norm(a["business_name"]),
            norm(b["business_name"])
        )

        address_score = token_set_ratio(
            norm(a["business_address"]),
            norm(b["business_address"])
        )

        combined = (
            0.6 * name_score +
            0.4 * address_score
        )

        scores.append({
            "source1_entity_id": source1_id,
            "candidate_entity_id": candidate_id,
            "name_score": round(name_score, 1),
            "address_score": round(address_score, 1),
            "combined_score": round(combined, 1)
        })


scores = pd.DataFrame(scores)

print("\n==============================")
print("TRUE-MATCH CALIBRATION")
print("==============================")
print("True matches scored:", len(scores))

if len(scores):
    print("\nCombined-score distribution:")
    print(scores["combined_score"].describe())

    print("\nLowest-scoring true matches:")
    print(
        scores.sort_values("combined_score")
        .head(20)
        .to_string(index=False)
    )

    print("\nTrue matches below thresholds:")
    for threshold in [70, 75, 80, 85, 90, 95]:
        count = (scores["combined_score"] < threshold).sum()
        print(
            f"Below {threshold}: {count} "
            f"({count / len(scores) * 100:.2f}%)"
        )

Training Source 1 entities: 100
True matched S2/S3 records: 343

Reading Source 1...
Source 1 found: 100 / 100
S2 records needed: 166
S3 records needed: 177

Reading Source 2...
Source 2 found: 166 / 166

Reading Source 3...
Source 3 found: 177 / 177

TRUE-MATCH CALIBRATION
True matches scored: 343

Combined-score distribution:
count    343.000000
mean      88.055977
std       13.748687
min       37.600000
25%       84.900000
50%       93.000000
75%       97.600000
max      100.000000
Name: combined_score, dtype: float64

Lowest-scoring true matches:
source1_entity_id candidate_entity_id  name_score  address_score  combined_score
     S1-553375105         S2-81940621        16.7           69.0            37.6
        S1-965667        S3-775321672        11.4           82.4            39.8
     S1-777597828        S3-413669121        20.0           73.0            41.2
     S1-161556164         S3-91054277         6.7          100.0            44.0
     S1-116043204         S3-85523430 

In [7]:
print("TRUE-MATCH RULE CALIBRATION")
print("=" * 45)

rules = {
    "Name >= 95": scores["name_score"] >= 95,
    "Name >= 90": scores["name_score"] >= 90,
    "Address >= 95": scores["address_score"] >= 95,
    "Address >= 90": scores["address_score"] >= 90,
    "Name >= 90 OR Address >= 90":
        (scores["name_score"] >= 90) |
        (scores["address_score"] >= 90),
    "Name >= 95 OR Address >= 95":
        (scores["name_score"] >= 95) |
        (scores["address_score"] >= 95),
    "Name >= 85 AND Address >= 70":
        (scores["name_score"] >= 85) &
        (scores["address_score"] >= 70),
}

total = len(scores)

for name, mask in rules.items():
    recovered = int(mask.sum())
    recall = recovered / total * 100

    print(
        f"{name:<35} "
        f"{recovered:>3}/{total} "
        f"({recall:.2f}%)"
    )

TRUE-MATCH RULE CALIBRATION
Name >= 95                          213/343 (62.10%)
Name >= 90                          238/343 (69.39%)
Address >= 95                       130/343 (37.90%)
Address >= 90                       214/343 (62.39%)
Name >= 90 OR Address >= 90         305/343 (88.92%)
Name >= 95 OR Address >= 95         262/343 (76.38%)
Name >= 85 AND Address >= 70        234/343 (68.22%)


In [8]:
print("TEST CANDIDATE SCORE ANALYSIS")
print("=" * 45)

total = len(scores)

rules = {
    "Name >= 95": scores["name_score"] >= 95,
    "Address >= 95": scores["address_score"] >= 95,
    "Name >= 90 OR Address >= 90":
        (scores["name_score"] >= 90) |
        (scores["address_score"] >= 90),
    "Name >= 95 OR Address >= 95":
        (scores["name_score"] >= 95) |
        (scores["address_score"] >= 95),
    "Both >= 80":
        (scores["name_score"] >= 80) &
        (scores["address_score"] >= 80),
}

for name, mask in rules.items():
    count = int(mask.sum())
    print(f"{name:<35} {count:>6}/{total}")

print("\nAmbiguous candidates (neither strong name nor strong address):")

ambiguous = scores[
    (scores["name_score"] < 90) &
    (scores["address_score"] < 90)
].sort_values("combined_score", ascending=False)

print(ambiguous.head(20).to_string(index=False))

TEST CANDIDATE SCORE ANALYSIS
Name >= 95                             213/343
Address >= 95                          130/343
Name >= 90 OR Address >= 90            305/343
Name >= 95 OR Address >= 95            262/343
Both >= 80                             228/343

Ambiguous candidates (neither strong name nor strong address):
source1_entity_id candidate_entity_id  name_score  address_score  combined_score
     S1-652339606        S3-626670274        89.5           89.7            89.5
     S1-102811957        S2-625774905        88.2           88.0            88.1
     S1-840162906        S3-800978181        88.2           88.0            88.1
     S1-989976700        S3-231189391        88.2           87.1            87.8
     S1-567308588        S2-226928496        89.7           85.0            87.8
      S1-26175016        S3-606010619        87.5           88.1            87.7
     S1-818149988        S3-965578558        85.7           87.5            86.4
      S1-67172650      

In [9]:
# STEP 1: Build labeled training candidates for calibration
# Safe chunked scan — does NOT load the full datasets into RAM.

import pandas as pd
import re
from unidecode import unidecode

# ---------------------------------------------------------
# 1. Get the 100 Source1 IDs from our existing ground truth
# ---------------------------------------------------------
gt_path = "s3://sagemaker-eu-north-1-152025327817/dataset/train/train_ground_truth.tsv"

gt100 = pd.read_csv(
    gt_path,
    sep="\t",
    nrows=100
)

train_s1_ids = set(gt100["source1_entity_id"].astype(str))

# True matched IDs for labeling
true_pairs = set()

for row in gt100.itertuples(index=False):
    s1 = str(row.source1_entity_id)
    matched = str(row.matched_entity_ids)

    if matched and matched != "nan":
        for cid in matched.split(","):
            cid = cid.strip()
            if cid:
                true_pairs.add((s1, cid))

print("Training Source1 entities:", len(train_s1_ids))
print("Known true pairs:", len(true_pairs))


# ---------------------------------------------------------
# 2. Read those 100 Source1 records
# ---------------------------------------------------------
s3 = boto3.client("s3", region_name="eu-north-1")

def collect_records_from_s3(key, wanted_ids, chunk_size=100000):
    found = {}

    obj = s3.get_object(
        Bucket=BUCKET,
        Key=key
    )

    for chunk in pd.read_csv(
        obj["Body"],
        sep="\t",
        chunksize=chunk_size,
        usecols=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    ):
        chunk["entity_id"] = chunk["entity_id"].astype(str)

        selected = chunk[chunk["entity_id"].isin(wanted_ids)]

        for row in selected.itertuples(index=False):
            found[row.entity_id] = {
                "business_name": row.business_name,
                "business_address": row.business_address,
                "country": row.country
            }

        if len(found) == len(wanted_ids):
            break

    return found


train_s1 = collect_records_from_s3(
    "dataset/train/train_source1.tsv",
    train_s1_ids
)

print("Source1 records found:", len(train_s1))


# ---------------------------------------------------------
# 3. Blocking keys
# ---------------------------------------------------------
def make_name_key(name):
    return normalize_name(name)


def make_addr_nums(address):
    text = normalize_address(address)

    nums = re.findall(r"\d+", text)

    # Keep unique numbers, preserving order
    seen = set()
    nums_unique = []

    for n in nums:
        if n not in seen:
            seen.add(n)
            nums_unique.append(n)

    # Need at least one number
    if not nums_unique:
        return ""

    return "|".join(nums_unique)


# Source1 lookup keys
name_to_s1 = {}
addr_to_s1 = {}

for s1_id, rec in train_s1.items():

    name_key = make_name_key(rec["business_name"])
    addr_key = make_addr_nums(rec["business_address"])

    if name_key:
        name_to_s1.setdefault(name_key, set()).add(s1_id)

    if addr_key:
        addr_to_s1.setdefault(addr_key, set()).add(s1_id)


print("Name blocking keys:", len(name_to_s1))
print("Address-number blocking keys:", len(addr_to_s1))


# ---------------------------------------------------------
# 4. Scan Source2 and Source3 ONCE
# ---------------------------------------------------------
def collect_training_candidates(key, source_label):

    candidates = []

    obj = s3.get_object(
        Bucket=BUCKET,
        Key=key
    )

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            obj["Body"],
            sep="\t",
            chunksize=100000,
            usecols=[
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ]
        )
    ):

        chunk["entity_id"] = chunk["entity_id"].astype(str)

        # Normalize names
        norm_names = chunk["business_name"].map(make_name_key)

        # Address-number keys
        addr_keys = chunk["business_address"].map(make_addr_nums)

        # Candidate rows through exact normalized name
        name_mask = norm_names.isin(name_to_s1)

        # Candidate rows through address numbers
        addr_mask = addr_keys.isin(addr_to_s1)

        selected_mask = name_mask | addr_mask

        selected = chunk[selected_mask].copy()

        if len(selected) == 0:
            continue

        for idx, row in selected.iterrows():

            cid = str(row["entity_id"])

            nk = norm_names.loc[idx]
            ak = addr_keys.loc[idx]

            matched_s1 = set()

            if nk in name_to_s1:
                matched_s1.update(name_to_s1[nk])

            if ak in addr_to_s1:
                matched_s1.update(addr_to_s1[ak])

            for s1_id in matched_s1:

                # Country must agree
                s1_country = str(train_s1[s1_id]["country"]).strip().lower()
                candidate_country = str(row["country"]).strip().lower()

                if s1_country != candidate_country:
                    continue

                candidates.append({
                    "source1_entity_id": s1_id,
                    "candidate_entity_id": cid,
                    "candidate_source": source_label,
                    "business_name": row["business_name"],
                    "business_address": row["business_address"],
                    "country": row["country"]
                })

        if chunk_no % 10 == 0:
            print(
                source_label,
                "chunks processed:",
                chunk_no,
                "| candidates so far:",
                len(candidates)
            )

    return candidates


train_candidates = []

train_candidates.extend(
    collect_training_candidates(
        "dataset/train/train_source2.tsv",
        "S2"
    )
)

train_candidates.extend(
    collect_training_candidates(
        "dataset/train/train_source3.tsv",
        "S3"
    )
)

calibration_candidates = pd.DataFrame(train_candidates)

print("\n====================================")
print("CALIBRATION CANDIDATES")
print("====================================")
print("Rows:", len(calibration_candidates))
print(
    "Unique Source1:",
    calibration_candidates["source1_entity_id"].nunique()
)
print(
    "Unique candidate IDs:",
    calibration_candidates["candidate_entity_id"].nunique()
)

# Label positives / negatives
calibration_candidates["is_true_match"] = [
    (s1, cid) in true_pairs
    for s1, cid in zip(
        calibration_candidates["source1_entity_id"],
        calibration_candidates["candidate_entity_id"]
    )
]

print("\nTrue matches:", calibration_candidates["is_true_match"].sum())
print("Non-matches:", (~calibration_candidates["is_true_match"]).sum())

Training Source1 entities: 100
Known true pairs: 343
Source1 records found: 100


NameError: name 'normalize_name' is not defined

In [10]:
# Restore the normalization functions needed for Step 1

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = unidecode(text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_name(name):
    text = normalize_text(name)

    suffixes = [
        "private limited", "pvt ltd", "pvt",
        "limited", "ltd", "llc",
        "incorporated", "inc",
        "corporation", "corp",
        "company", "co", "pc"
    ]

    for suffix in suffixes:
        text = re.sub(rf"\b{suffix}\b", " ", text)

    return re.sub(r"\s+", " ", text).strip()


def normalize_address(address):
    text = normalize_text(address)

    replacements = {
        "street": "st",
        "road": "rd",
        "avenue": "ave",
        "boulevard": "blvd",
        "drive": "dr",
        "lane": "ln",
        "highway": "hwy",
        "apartment": "apt",
        "suite": "ste",
        "floor": "fl"
    }

    for full, short in replacements.items():
        text = re.sub(rf"\b{full}\b", short, text)

    return re.sub(r"\s+", " ", text).strip()


print("Normalization functions restored successfully.")

Normalization functions restored successfully.


In [11]:
# STEP 1B: Build blocking keys for the 100 training Source1 entities

def make_name_key(name):
    return normalize_name(name)


def make_addr_nums(address):
    text = normalize_address(address)

    nums = re.findall(r"\d+", text)

    seen = set()
    nums_unique = []

    for n in nums:
        if n not in seen:
            seen.add(n)
            nums_unique.append(n)

    if not nums_unique:
        return ""

    return "|".join(nums_unique)


name_to_s1 = {}
addr_to_s1 = {}

for s1_id, rec in train_s1.items():

    name_key = make_name_key(rec["business_name"])
    addr_key = make_addr_nums(rec["business_address"])

    if name_key:
        name_to_s1.setdefault(name_key, set()).add(s1_id)

    if addr_key:
        addr_to_s1.setdefault(addr_key, set()).add(s1_id)


print("Source1 records:", len(train_s1))
print("Name blocking keys:", len(name_to_s1))
print("Address-number blocking keys:", len(addr_to_s1))

Source1 records: 100
Name blocking keys: 100
Address-number blocking keys: 92


In [12]:
# STEP 1C: Scan training Source2 and Source3 for calibration candidates

def collect_training_candidates(key, source_label):

    candidates = []

    obj = s3.get_object(
        Bucket=BUCKET,
        Key=key
    )

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            obj["Body"],
            sep="\t",
            chunksize=100000,
            usecols=[
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ]
        )
    ):

        chunk["entity_id"] = chunk["entity_id"].astype(str)

        norm_names = chunk["business_name"].map(make_name_key)
        addr_keys = chunk["business_address"].map(make_addr_nums)

        name_mask = norm_names.isin(name_to_s1)
        addr_mask = addr_keys.isin(addr_to_s1)

        selected = chunk[name_mask | addr_mask].copy()

        if len(selected) == 0:
            continue

        for idx, row in selected.iterrows():

            cid = str(row["entity_id"])

            matched_s1 = set()

            nk = norm_names.loc[idx]
            ak = addr_keys.loc[idx]

            if nk in name_to_s1:
                matched_s1.update(name_to_s1[nk])

            if ak in addr_to_s1:
                matched_s1.update(addr_to_s1[ak])

            for s1_id in matched_s1:

                s1_country = str(
                    train_s1[s1_id]["country"]
                ).strip().lower()

                candidate_country = str(
                    row["country"]
                ).strip().lower()

                if s1_country != candidate_country:
                    continue

                candidates.append({
                    "source1_entity_id": s1_id,
                    "candidate_entity_id": cid,
                    "candidate_source": source_label,
                    "business_name": row["business_name"],
                    "business_address": row["business_address"],
                    "country": row["country"]
                })

        if chunk_no % 10 == 0:
            print(
                source_label,
                "| chunks processed:",
                chunk_no,
                "| candidates:",
                len(candidates)
            )

    return candidates


# -----------------------------------------
# Scan Source 2
# -----------------------------------------
print("Starting Source2 scan...")

s2_calibration = collect_training_candidates(
    "dataset/train/train_source2.tsv",
    "S2"
)

print(
    "Source2 calibration candidates:",
    len(s2_calibration)
)


# -----------------------------------------
# Scan Source 3
# -----------------------------------------
print("\nStarting Source3 scan...")

s3_calibration = collect_training_candidates(
    "dataset/train/train_source3.tsv",
    "S3"
)

print(
    "Source3 calibration candidates:",
    len(s3_calibration)
)


# -----------------------------------------
# Combine
# -----------------------------------------
calibration_candidates = pd.DataFrame(
    s2_calibration + s3_calibration
)

print("\n====================================")
print("CALIBRATION CANDIDATES")
print("====================================")

print(
    "Rows:",
    len(calibration_candidates)
)

print(
    "Unique Source1:",
    calibration_candidates["source1_entity_id"].nunique()
)

print(
    "Unique candidate IDs:",
    calibration_candidates["candidate_entity_id"].nunique()
)


# -----------------------------------------
# Label true / false
# -----------------------------------------
calibration_candidates["is_true_match"] = [
    (s1, cid) in true_pairs
    for s1, cid in zip(
        calibration_candidates["source1_entity_id"],
        calibration_candidates["candidate_entity_id"]
    )
]

print(
    "True matches:",
    int(calibration_candidates["is_true_match"].sum())
)

print(
    "Non-matches:",
    int((~calibration_candidates["is_true_match"]).sum())
)

Starting Source2 scan...
S2 | chunks processed: 0 | candidates: 2369
S2 | chunks processed: 10 | candidates: 26149
S2 | chunks processed: 20 | candidates: 49679
S2 | chunks processed: 30 | candidates: 73818
S2 | chunks processed: 40 | candidates: 97533
S2 | chunks processed: 50 | candidates: 119764
Source2 calibration candidates: 119764

Starting Source3 scan...
S3 | chunks processed: 0 | candidates: 2611
S3 | chunks processed: 10 | candidates: 28118
S3 | chunks processed: 20 | candidates: 53889
S3 | chunks processed: 30 | candidates: 79612
S3 | chunks processed: 40 | candidates: 105309
S3 | chunks processed: 50 | candidates: 130618
Source3 calibration candidates: 135274

CALIBRATION CANDIDATES
Rows: 255038
Unique Source1: 98
Unique candidate IDs: 254973
True matches: 253
Non-matches: 254785


In [13]:
# STEP 2: Score the 255k calibration candidates
# No S3/S2 scan. Uses records already collected above.

from rapidfuzz.fuzz import token_set_ratio

def score_pair(s1_id, candidate_name, candidate_address):
    s1_rec = train_s1[s1_id]

    s1_name = normalize_name(s1_rec["business_name"])
    s1_addr = normalize_address(s1_rec["business_address"])

    cand_name = normalize_name(candidate_name)
    cand_addr = normalize_address(candidate_address)

    name_score = token_set_ratio(s1_name, cand_name)
    address_score = token_set_ratio(s1_addr, cand_addr)

    combined_score = (
        0.6 * name_score +
        0.4 * address_score
    )

    return name_score, address_score, combined_score


# Score each candidate
results = []

for row in calibration_candidates.itertuples(index=False):

    ns, ads, cs = score_pair(
        row.source1_entity_id,
        row.business_name,
        row.business_address
    )

    results.append((ns, ads, cs))


score_df = calibration_candidates.copy()

score_values = pd.DataFrame(
    results,
    columns=[
        "name_score",
        "address_score",
        "combined_score"
    ],
    index=score_df.index
)

score_df[
    ["name_score", "address_score", "combined_score"]
] = score_values


print("=============================================")
print("CALIBRATION SCORING COMPLETE")
print("=============================================")
print("Rows scored:", len(score_df))

print("\nTrue matches:")
print(
    score_df["is_true_match"].sum()
)

print("\nNon-matches:")
print(
    (~score_df["is_true_match"]).sum()
)

print("\nScore range:")
print(
    score_df[
        ["name_score", "address_score", "combined_score"]
    ].describe()
)

CALIBRATION SCORING COMPLETE
Rows scored: 255038

True matches:
253

Non-matches:
254785

Score range:
          name_score  address_score  combined_score
count  255038.000000  255038.000000   255038.000000
mean       32.079007      41.694173       35.925074
std        11.966076       7.979128        7.723507
min         0.000000       0.000000        8.372093
25%        25.806452      37.288136       31.601329
50%        30.769231      41.304348       35.046780
75%        35.714286      45.454545       38.768650
max       100.000000     100.000000      100.000000


In [14]:
# STEP 3: Measure precision/recall at different matching rules

def evaluate_rule(mask, label):
    total_true = int(score_df["is_true_match"].sum())

    selected = score_df[mask]

    tp = int(selected["is_true_match"].sum())
    fp = int((~selected["is_true_match"]).sum())
    fn = total_true - tp

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / total_true if total_true else 0

    beta = 0.5

    f05 = (
        (1 + beta**2) * precision * recall
        / ((beta**2 * precision) + recall)
        if (precision + recall) > 0
        else 0
    )

    print(
        f"{label:<40} "
        f"TP={tp:4d}  FP={fp:6d}  "
        f"Precision={precision:.4f}  "
        f"Recall={recall:.4f}  "
        f"F0.5={f05:.4f}"
    )


print("RULE CALIBRATION")
print("=" * 100)

# Combined-score thresholds
for threshold in [60, 65, 70, 75, 80, 85, 90, 95]:

    evaluate_rule(
        score_df["combined_score"] >= threshold,
        f"Combined >= {threshold}"
    )


print("\nNAME / ADDRESS RULES")
print("=" * 100)

evaluate_rule(
    (score_df["name_score"] >= 90) |
    (score_df["address_score"] >= 90),
    "Name >=90 OR Address >=90"
)

evaluate_rule(
    (score_df["name_score"] >= 95) |
    (score_df["address_score"] >= 95),
    "Name >=95 OR Address >=95"
)

evaluate_rule(
    (score_df["name_score"] >= 90) &
    (score_df["address_score"] >= 80),
    "Name >=90 AND Address >=80"
)

evaluate_rule(
    (score_df["name_score"] >= 85) &
    (score_df["address_score"] >= 85),
    "Name >=85 AND Address >=85"
)

evaluate_rule(
    (score_df["name_score"] >= 80) &
    (score_df["address_score"] >= 90),
    "Name >=80 AND Address >=90"
)

evaluate_rule(
    (score_df["name_score"] >= 90) &
    (score_df["address_score"] >= 70),
    "Name >=90 AND Address >=70"
)


print("\nTOP FALSE POSITIVES")
print("=" * 100)

false_positive = score_df[
    ~score_df["is_true_match"]
].sort_values(
    "combined_score",
    ascending=False
)

print(
    false_positive[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "name_score",
            "address_score",
            "combined_score"
        ]
    ].head(20).to_string(index=False)
)

RULE CALIBRATION
Combined >= 60                           TP= 244  FP=  4795  Precision=0.0484  Recall=0.9644  F0.5=0.0598
Combined >= 65                           TP= 232  FP=  3798  Precision=0.0576  Recall=0.9170  F0.5=0.0708
Combined >= 70                           TP= 224  FP=  3506  Precision=0.0601  Recall=0.8854  F0.5=0.0738
Combined >= 75                           TP= 218  FP=  1405  Precision=0.1343  Recall=0.8617  F0.5=0.1616
Combined >= 80                           TP= 211  FP=   113  Precision=0.6512  Recall=0.8340  F0.5=0.6811
Combined >= 85                           TP= 203  FP=    22  Precision=0.9022  Recall=0.8024  F0.5=0.8803
Combined >= 90                           TP= 188  FP=    18  Precision=0.9126  Recall=0.7431  F0.5=0.8728
Combined >= 95                           TP= 150  FP=     7  Precision=0.9554  Recall=0.5929  F0.5=0.8513

NAME / ADDRESS RULES
Name >=90 OR Address >=90                TP= 240  FP=  3906  Precision=0.0579  Recall=0.9486  F0.5=0.0713
Name >=

In [15]:
# STEP 4: Inspect exact-name collisions and false positives

# Add normalized name key
score_df["name_key"] = score_df["source1_entity_id"].map(
    lambda x: normalize_name(
        train_s1[x]["business_name"]
    )
)

# How many candidate rows share each Source1's normalized name?
name_candidate_counts = (
    score_df.groupby("source1_entity_id")
    .size()
    .sort_values(ascending=False)
)

print("CANDIDATE COUNT PER SOURCE1")
print("=" * 60)
print(name_candidate_counts.describe())


# ---------------------------------------------------------
# False positives where name is EXACTLY the same
# ---------------------------------------------------------
exact_name_fp = score_df[
    (~score_df["is_true_match"]) &
    (score_df["name_score"] == 100)
].copy()

print("\nFALSE POSITIVES WITH NAME SCORE = 100")
print("=" * 60)
print("Count:", len(exact_name_fp))

print(
    exact_name_fp[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "name_score",
            "address_score",
            "combined_score"
        ]
    ]
    .sort_values("address_score", ascending=False)
    .head(30)
    .to_string(index=False)
)


# ---------------------------------------------------------
# True matches with exact name but weak address
# ---------------------------------------------------------
exact_name_true = score_df[
    (score_df["is_true_match"]) &
    (score_df["name_score"] == 100)
].copy()

print("\nTRUE MATCHES WITH NAME SCORE = 100")
print("=" * 60)
print("Count:", len(exact_name_true))

print(
    exact_name_true["address_score"]
    .describe()
)


# ---------------------------------------------------------
# True matches with weak name but strong address
# ---------------------------------------------------------
weak_name_true = score_df[
    (score_df["is_true_match"]) &
    (score_df["name_score"] < 80)
].copy()

print("\nTRUE MATCHES WITH NAME < 80")
print("=" * 60)
print("Count:", len(weak_name_true))

print(
    weak_name_true[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "name_score",
            "address_score",
            "combined_score"
        ]
    ]
    .sort_values("address_score", ascending=False)
    .head(30)
    .to_string(index=False)
)

CANDIDATE COUNT PER SOURCE1
count       98.000000
mean      2602.428571
std       6452.374071
min          1.000000
25%         18.250000
50%        236.000000
75%       1952.500000
max      43500.000000
dtype: float64

FALSE POSITIVES WITH NAME SCORE = 100
Count: 3628
source1_entity_id candidate_entity_id  name_score  address_score  combined_score
     S1-731397769        S2-334516966       100.0      93.939394       97.575758
     S1-842533642        S2-656410013       100.0      93.333333       97.333333
     S1-553375105        S3-691302578       100.0      90.697674       96.279070
     S1-262997549        S2-879297655       100.0      90.196078       96.078431
     S1-262997549        S3-910786226       100.0      89.655172       95.862069
     S1-508022910        S3-877617107       100.0      88.571429       95.428571
     S1-365634347        S2-430083508       100.0      87.500000       95.000000
     S1-790589419        S3-727252499       100.0      86.400000       94.560000
 

In [16]:
# STEP 5: Create selective address-prefix blocking keys
# We use the first 4 normalized address tokens.

def make_addr_first4(address):
    text = normalize_address(address)
    tokens = text.split()

    if len(tokens) < 4:
        return ""

    return " ".join(tokens[:4])


addr4_to_s1 = {}

for s1_id, rec in train_s1.items():

    key = make_addr_first4(rec["business_address"])

    if key:
        addr4_to_s1.setdefault(key, set()).add(s1_id)


print("=============================================")
print("ADDRESS FIRST-4 BLOCKING")
print("=============================================")
print("Source1 entities:", len(train_s1))
print("Usable address4 keys:", len(addr4_to_s1))

print("\nExample keys:")
for key, ids in list(addr4_to_s1.items())[:10]:
    print(repr(key), "->", list(ids))

ADDRESS FIRST-4 BLOCKING
Source1 entities: 100
Usable address4 keys: 100

Example keys:
'1013 girard ave indianola' -> ['S1-262997549']
'no 35 brentwood apartments' -> ['S1-244810289']
'kolanghattil house ward13 no' -> ['S1-389414274']
'h no 847 s' -> ['S1-439203009']
'6267 passing sky dr' -> ['S1-862038247']
'356 bell factory rd' -> ['S1-541270202']
'335 terry blvd islip' -> ['S1-24234371']
'1401 14th fl plot' -> ['S1-9962387']
'2493 pennwood court round' -> ['S1-686913759']
'row house no b4' -> ['S1-491795528']


In [17]:
# STEP 6: Test selective address-first4 blocking
# This is calibration only.

def scan_addr4_candidates(key, source_label, max_key_frequency=100):

    candidates = []

    obj = s3.get_object(
        Bucket=BUCKET,
        Key=key
    )

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            obj["Body"],
            sep="\t",
            chunksize=100000,
            usecols=[
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ]
        )
    ):

        chunk["entity_id"] = chunk["entity_id"].astype(str)

        addr4 = chunk["business_address"].map(make_addr_first4)

        selected = chunk[addr4.isin(addr4_to_s1)].copy()

        if len(selected) == 0:
            continue

        for idx, row in selected.iterrows():

            key4 = addr4.loc[idx]
            matched_s1 = addr4_to_s1.get(key4, set())

            # Country filter
            for s1_id in matched_s1:

                s1_country = str(
                    train_s1[s1_id]["country"]
                ).strip().lower()

                candidate_country = str(
                    row["country"]
                ).strip().lower()

                if s1_country != candidate_country:
                    continue

                candidates.append({
                    "source1_entity_id": s1_id,
                    "candidate_entity_id": str(row["entity_id"]),
                    "candidate_source": source_label,
                    "business_name": row["business_name"],
                    "business_address": row["business_address"],
                    "country": row["country"]
                })

        if chunk_no % 10 == 0:
            print(
                source_label,
                "| chunks:",
                chunk_no,
                "| candidates:",
                len(candidates)
            )

    return candidates


print("Starting Source2 address4 scan...")

addr4_s2 = scan_addr4_candidates(
    "dataset/train/train_source2.tsv",
    "S2"
)

print(
    "\nSource2 address4 candidates:",
    len(addr4_s2)
)


print("\nStarting Source3 address4 scan...")

addr4_s3 = scan_addr4_candidates(
    "dataset/train/train_source3.tsv",
    "S3"
)

print(
    "\nSource3 address4 candidates:",
    len(addr4_s3)
)


# Combine and remove duplicates
addr4_candidates = pd.DataFrame(
    addr4_s2 + addr4_s3
).drop_duplicates(
    subset=[
        "source1_entity_id",
        "candidate_entity_id"
    ]
)

# Label known true matches
addr4_candidates["is_true_match"] = [
    (s1, cid) in true_pairs
    for s1, cid in zip(
        addr4_candidates["source1_entity_id"],
        addr4_candidates["candidate_entity_id"]
    )
]

print("\n=============================================")
print("ADDRESS4 RESULTS")
print("=============================================")
print("Candidate pairs:", len(addr4_candidates))
print(
    "Unique Source1:",
    addr4_candidates["source1_entity_id"].nunique()
)
print(
    "True matches recovered:",
    int(addr4_candidates["is_true_match"].sum())
)
print(
    "New true matches NOT in previous candidates:",
    int(
        addr4_candidates[
            addr4_candidates["is_true_match"]
        ]
        .apply(
            lambda r:
            not (
                (
                    r["source1_entity_id"],
                    r["candidate_entity_id"]
                )
                in set(
                    zip(
                        calibration_candidates["source1_entity_id"],
                        calibration_candidates["candidate_entity_id"]
                    )
                )
            ),
            axis=1
        )
        .sum()
    )
)

Starting Source2 address4 scan...
S2 | chunks: 0 | candidates: 1
S2 | chunks: 10 | candidates: 32
S2 | chunks: 20 | candidates: 78
S2 | chunks: 30 | candidates: 113
S2 | chunks: 40 | candidates: 146
S2 | chunks: 50 | candidates: 180

Source2 address4 candidates: 180

Starting Source3 address4 scan...
S3 | chunks: 0 | candidates: 1
S3 | chunks: 10 | candidates: 46
S3 | chunks: 20 | candidates: 77
S3 | chunks: 30 | candidates: 107
S3 | chunks: 40 | candidates: 133
S3 | chunks: 50 | candidates: 164

Source3 address4 candidates: 171

ADDRESS4 RESULTS
Candidate pairs: 351
Unique Source1: 57
True matches recovered: 113
New true matches NOT in previous candidates: 10


In [18]:
# STEP 7: Identify exactly which true matches are still missing
# No S3/S2 scan.

previous_pairs = set(
    zip(
        calibration_candidates["source1_entity_id"],
        calibration_candidates["candidate_entity_id"]
    )
)

addr4_pairs = set(
    zip(
        addr4_candidates["source1_entity_id"],
        addr4_candidates["candidate_entity_id"]
    )
)

all_recovered_pairs = previous_pairs | addr4_pairs

missing_true_pairs = [
    (s1, cid)
    for (s1, cid) in true_pairs
    if (s1, cid) not in all_recovered_pairs
]

print("=============================================")
print("MISSING TRUE MATCHES")
print("=============================================")
print("Total known true pairs:", len(true_pairs))
print("Recovered:", len(all_recovered_pairs & true_pairs))
print("Still missing:", len(missing_true_pairs))

print("\nFirst 30 missing pairs:")
for s1, cid in missing_true_pairs[:30]:
    print(s1, "->", cid)


# Which Source1 entities are affected?
missing_by_s1 = {}

for s1, cid in missing_true_pairs:
    missing_by_s1.setdefault(s1, []).append(cid)

print("\nSource1 entities with missing matches:")
print(len(missing_by_s1))

print("\nFirst 20:")
for s1, ids in list(missing_by_s1.items())[:20]:
    print(s1, "->", ids)

MISSING TRUE MATCHES
Total known true pairs: 343
Recovered: 263
Still missing: 80

First 30 missing pairs:
S1-72310418 -> S2-516305116
S1-518197819 -> S2-891724637
S1-9962387 -> S3-13637230
S1-502736054 -> S3-537652714
S1-72310418 -> S2-11779506
S1-567308588 -> S2-191200521
S1-116043204 -> S3-85523430
S1-170252930 -> S3-228972158
S1-67172650 -> S3-251407198
S1-550488938 -> S3-651095723
S1-72310418 -> S3-731160199
S1-656753428 -> S3-679606215
S1-389414274 -> S2-933610960
S1-389414274 -> S3-443891409
S1-212509374 -> S2-598327313
S1-731542669 -> S2-939487122
S1-876102895 -> S2-358560681
S1-9962387 -> S3-160760047
S1-876102895 -> S2-791711694
S1-965667 -> S2-681193310
S1-561341312 -> S2-483615364
S1-818149988 -> S3-965578558
S1-185938539 -> S3-880695057
S1-541270202 -> S3-31133661
S1-463669205 -> S2-301757320
S1-318373630 -> S2-660036492
S1-574308285 -> S3-875723263
S1-314714647 -> S3-192802051
S1-727602285 -> S3-204655096
S1-731542669 -> S3-988120505

Source1 entities with missing matches

In [19]:
# STEP 8: Inspect the 80 missing true matches
# Only retrieves the specific IDs we already know are true matches.

missing_s1_ids = set(s1 for s1, cid in missing_true_pairs)
missing_candidate_ids = set(cid for s1, cid in missing_true_pairs)

print("Missing Source1 IDs:", len(missing_s1_ids))
print("Missing candidate IDs:", len(missing_candidate_ids))


# ---------------------------------------------------------
# Retrieve the missing candidate records from S2 + S3
# ---------------------------------------------------------
def collect_specific_ids(key, wanted_ids, chunk_size=100000):

    found = {}

    obj = s3.get_object(
        Bucket=BUCKET,
        Key=key
    )

    for chunk in pd.read_csv(
        obj["Body"],
        sep="\t",
        chunksize=chunk_size,
        usecols=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    ):

        chunk["entity_id"] = chunk["entity_id"].astype(str)

        selected = chunk[
            chunk["entity_id"].isin(wanted_ids)
        ]

        for row in selected.itertuples(index=False):
            found[row.entity_id] = {
                "business_name": row.business_name,
                "business_address": row.business_address,
                "country": row.country
            }

        if len(found) == len(wanted_ids):
            break

    return found


missing_s2_ids = {
    cid for s1, cid in missing_true_pairs
    if cid.startswith("S2-")
}

missing_s3_ids = {
    cid for s1, cid in missing_true_pairs
    if cid.startswith("S3-")
}

print("\nS2 IDs:", len(missing_s2_ids))
print("S3 IDs:", len(missing_s3_ids))


missing_s2 = collect_specific_ids(
    "dataset/train/train_source2.tsv",
    missing_s2_ids
)

missing_s3 = collect_specific_ids(
    "dataset/train/train_source3.tsv",
    missing_s3_ids
)

missing_records = {
    **missing_s2,
    **missing_s3
}

print("S2 records found:", len(missing_s2))
print("S3 records found:", len(missing_s3))


# ---------------------------------------------------------
# Analyze every missing true pair
# ---------------------------------------------------------
missing_analysis = []

for s1_id, cid in missing_true_pairs:

    s1_rec = train_s1[s1_id]
    cand_rec = missing_records[cid]

    s1_name = normalize_name(s1_rec["business_name"])
    cand_name = normalize_name(cand_rec["business_name"])

    s1_addr = normalize_address(s1_rec["business_address"])
    cand_addr = normalize_address(cand_rec["business_address"])

    name_score = token_set_ratio(
        s1_name,
        cand_name
    )

    address_score = token_set_ratio(
        s1_addr,
        cand_addr
    )

    combined_score = (
        0.6 * name_score +
        0.4 * address_score
    )

    s1_name_key = make_name_key(
        s1_rec["business_name"]
    )

    cand_name_key = make_name_key(
        cand_rec["business_name"]
    )

    s1_addr4 = make_addr_first4(
        s1_rec["business_address"]
    )

    cand_addr4 = make_addr_first4(
        cand_rec["business_address"]
    )

    s1_nums = make_addr_nums(
        s1_rec["business_address"]
    )

    cand_nums = make_addr_nums(
        cand_rec["business_address"]
    )

    missing_analysis.append({
        "source1_entity_id": s1_id,
        "candidate_entity_id": cid,
        "name_score": name_score,
        "address_score": address_score,
        "combined_score": combined_score,
        "exact_name": s1_name_key == cand_name_key,
        "exact_addr4": s1_addr4 == cand_addr4 and s1_addr4 != "",
        "exact_addr_nums": s1_nums == cand_nums and s1_nums != "",
        "s1_name": s1_rec["business_name"],
        "candidate_name": cand_rec["business_name"],
        "s1_address": s1_rec["business_address"],
        "candidate_address": cand_rec["business_address"]
    })


missing_analysis_df = pd.DataFrame(
    missing_analysis
)

print("\n=============================================")
print("MISSING MATCH ANALYSIS")
print("=============================================")

print(
    missing_analysis_df[
        [
            "name_score",
            "address_score",
            "combined_score"
        ]
    ].describe()
)

print("\nBlocking patterns among missing true matches:")

print(
    "Exact normalized name:",
    int(missing_analysis_df["exact_name"].sum()),
    "/",
    len(missing_analysis_df)
)

print(
    "Exact address first-4:",
    int(missing_analysis_df["exact_addr4"].sum()),
    "/",
    len(missing_analysis_df)
)

print(
    "Exact address numbers:",
    int(missing_analysis_df["exact_addr_nums"].sum()),
    "/",
    len(missing_analysis_df)
)

print("\nFirst 30 missing matches:")
print(
    missing_analysis_df[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "name_score",
            "address_score",
            "combined_score",
            "exact_name",
            "exact_addr4",
            "exact_addr_nums"
        ]
    ]
    .sort_values("combined_score", ascending=False)
    .head(30)
    .to_string(index=False)
)

Missing Source1 IDs: 50
Missing candidate IDs: 80

S2 IDs: 36
S3 IDs: 44
S2 records found: 36
S3 records found: 44

MISSING MATCH ANALYSIS
       name_score  address_score  combined_score
count   80.000000      80.000000       80.000000
mean    80.184589      78.884525       79.664564
std     24.966180      31.427572       17.343559
min     10.526316       0.000000       33.600000
25%     68.724138      81.423611       65.871609
50%     87.595908      89.564428       85.638338
75%    100.000000      97.266592       94.618768
max    100.000000     100.000000      100.000000

Blocking patterns among missing true matches:
Exact normalized name: 0 / 80
Exact address first-4: 0 / 80
Exact address numbers: 0 / 80

First 30 missing matches:
source1_entity_id candidate_entity_id  name_score  address_score  combined_score  exact_name  exact_addr4  exact_addr_nums
     S1-842533642        S2-352625664  100.000000     100.000000      100.000000       False        False            False
     S1-49

In [20]:
# STEP 9 — Inspect token overlap in the 80 missing true matches

import re
import pandas as pd

def tokens(text):
    return set(normalize_text(text).split())

def token_overlap(a, b):
    ta = tokens(a)
    tb = tokens(b)
    if not ta or not tb:
        return 0
    return len(ta & tb)

def shared_tokens(a, b):
    return sorted(tokens(a) & tokens(b))

inspect = missing_analysis.copy()

inspect["name_shared_tokens"] = inspect.apply(
    lambda r: token_overlap(r["s1_name"], r["candidate_name"]),
    axis=1
)

inspect["addr_shared_tokens"] = inspect.apply(
    lambda r: token_overlap(r["s1_address"], r["candidate_address"]),
    axis=1
)

inspect["name_shared"] = inspect.apply(
    lambda r: ", ".join(shared_tokens(r["s1_name"], r["candidate_name"])),
    axis=1
)

inspect["addr_shared"] = inspect.apply(
    lambda r: ", ".join(shared_tokens(r["s1_address"], r["candidate_address"])),
    axis=1
)

print("=============================================")
print("TOKEN OVERLAP ANALYSIS")
print("=============================================")

print("\nName shared-token count:")
print(inspect["name_shared_tokens"].value_counts().sort_index())

print("\nAddress shared-token count:")
print(inspect["addr_shared_tokens"].value_counts().sort_index())

print("\nFirst 40 missing matches with shared tokens:")
print(
    inspect[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "name_score",
            "address_score",
            "combined_score",
            "name_shared_tokens",
            "addr_shared_tokens",
            "name_shared",
            "addr_shared"
        ]
    ]
    .sort_values("combined_score", ascending=False)
    .head(40)
    .to_string(index=False)
)

AttributeError: 'list' object has no attribute 'apply'

In [22]:
# STEP 9 — Inspect token overlap in the 80 missing true matches

import re
import pandas as pd

# Convert to DataFrame safely
inspect = pd.DataFrame(missing_analysis)

print("Columns available:")
print(inspect.columns.tolist())

def tokens(text):
    return set(normalize_text(text).split())

def token_overlap(a, b):
    ta = tokens(a)
    tb = tokens(b)
    if not ta or not tb:
        return 0
    return len(ta & tb)

def shared_tokens(a, b):
    return sorted(tokens(a) & tokens(b))

inspect["name_shared_tokens"] = inspect.apply(
    lambda r: token_overlap(r["s1_name"], r["candidate_name"]),
    axis=1
)

inspect["addr_shared_tokens"] = inspect.apply(
    lambda r: token_overlap(r["s1_address"], r["candidate_address"]),
    axis=1
)

inspect["name_shared"] = inspect.apply(
    lambda r: ", ".join(shared_tokens(r["s1_name"], r["candidate_name"])),
    axis=1
)

inspect["addr_shared"] = inspect.apply(
    lambda r: ", ".join(shared_tokens(r["s1_address"], r["candidate_address"])),
    axis=1
)

print("\n=============================================")
print("TOKEN OVERLAP ANALYSIS")
print("=============================================")

print("\nName shared-token count:")
print(inspect["name_shared_tokens"].value_counts().sort_index())

print("\nAddress shared-token count:")
print(inspect["addr_shared_tokens"].value_counts().sort_index())

print("\nFirst 40 missing matches with shared tokens:")

print(
    inspect[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "name_score",
            "address_score",
            "combined_score",
            "name_shared_tokens",
            "addr_shared_tokens",
            "name_shared",
            "addr_shared"
        ]
    ]
    .sort_values("combined_score", ascending=False)
    .head(40)
    .to_string(index=False)
)

Columns available:
['source1_entity_id', 'candidate_entity_id', 'name_score', 'address_score', 'combined_score', 'exact_name', 'exact_addr4', 'exact_addr_nums', 's1_name', 'candidate_name', 's1_address', 'candidate_address']

TOKEN OVERLAP ANALYSIS

Name shared-token count:
name_shared_tokens
0    19
1     3
2    26
3    21
4    10
5     1
Name: count, dtype: int64

Address shared-token count:
addr_shared_tokens
0     10
2      7
3     18
4     14
5      5
6      4
7      4
8      1
9      7
10     3
11     2
15     3
16     2
Name: count, dtype: int64

First 40 missing matches with shared tokens:
source1_entity_id candidate_entity_id  name_score  address_score  combined_score  name_shared_tokens  addr_shared_tokens                            name_shared                                                                                                  addr_shared
     S1-842533642        S2-352625664  100.000000     100.000000      100.000000                   2                   4      

In [23]:
# STEP 10 — Test selective character-prefix blocking on the 80 missing matches

def char_key(text, n=4):
    text = normalize_text(text)
    text = text.replace(" ", "")
    if len(text) < n:
        return ""
    return text[:n]

inspect = pd.DataFrame(missing_analysis)

inspect["s1_name_char4"] = inspect["s1_name"].apply(char_key)
inspect["candidate_name_char4"] = inspect["candidate_name"].apply(char_key)

inspect["same_char4"] = (
    inspect["s1_name_char4"] ==
    inspect["candidate_name_char4"]
)

print("=============================================")
print("CHARACTER PREFIX BLOCKING")
print("=============================================")

print(
    "Missing true matches recovered by first 4 name characters:",
    inspect["same_char4"].sum(),
    "/",
    len(inspect)
)

print("\nRecovered examples:")

print(
    inspect[
        inspect["same_char4"]
    ][
        [
            "source1_entity_id",
            "candidate_entity_id",
            "s1_name",
            "candidate_name",
            "name_score",
            "address_score",
            "combined_score",
            "s1_name_char4"
        ]
    ]
    .sort_values("combined_score", ascending=False)
    .head(30)
    .to_string(index=False)
)

print("\nNOT recovered:")

print(
    inspect[
        ~inspect["same_char4"]
    ][
        [
            "source1_entity_id",
            "candidate_entity_id",
            "s1_name",
            "candidate_name",
            "name_score",
            "address_score",
            "combined_score"
        ]
    ]
    .sort_values("combined_score", ascending=False)
    .head(30)
    .to_string(index=False)
)

CHARACTER PREFIX BLOCKING
Missing true matches recovered by first 4 name characters: 57 / 80

Recovered examples:
source1_entity_id candidate_entity_id                                       s1_name                                           candidate_name  name_score  address_score  combined_score s1_name_char4
     S1-842533642        S2-352625664           Certified Digital Associates Center                                        Certified Digital  100.000000     100.000000      100.000000          cert
     S1-730719211        S2-467798366                         Renee G. Garcia, O.D.                                         Renee G. Garcia,  100.000000      97.777778       99.111111          rene
       S1-9962387         S3-13637230 Systel Buildstructure (India) Private Limited Systel Buildstructure Buildstructure (India) Private Ltd  100.000000      96.256684       98.502674          syst
     S1-432073164        S3-520531285                       Fayth Fritz Lifeco Corp           

In [24]:
# STEP 11 — Test first meaningful-name-token blocking

import re
import pandas as pd

inspect = pd.DataFrame(missing_analysis)

# Common prefixes/titles that should not be used for blocking
NAME_PREFIXES = {
    "mr", "mrs", "ms", "miss", "dr", "prof",
    "m", "sri", "smt", "shri", "m/s", "msr"
}

def first_meaningful_token(name):
    text = normalize_text(name)
    parts = text.split()

    for part in parts:
        if part not in NAME_PREFIXES and len(part) >= 3:
            return part

    return ""

inspect["s1_first_token"] = inspect["s1_name"].apply(first_meaningful_token)
inspect["candidate_first_token"] = inspect["candidate_name"].apply(first_meaningful_token)

inspect["same_first_token"] = (
    inspect["s1_first_token"] ==
    inspect["candidate_first_token"]
) & (inspect["s1_first_token"] != "")

print("=============================================")
print("FIRST MEANINGFUL TOKEN BLOCKING")
print("=============================================")

print(
    "Missing true matches recovered:",
    inspect["same_first_token"].sum(),
    "/",
    len(inspect)
)

print("\nRecovered matches:")

print(
    inspect[
        inspect["same_first_token"]
    ][
        [
            "source1_entity_id",
            "candidate_entity_id",
            "s1_name",
            "candidate_name",
            "name_score",
            "address_score",
            "combined_score",
            "s1_first_token"
        ]
    ]
    .sort_values("combined_score", ascending=False)
    .to_string(index=False)
)

print("\nStill NOT recovered:")

print(
    inspect[
        ~inspect["same_first_token"]
    ][
        [
            "source1_entity_id",
            "candidate_entity_id",
            "s1_name",
            "candidate_name",
            "name_score",
            "address_score",
            "combined_score"
        ]
    ]
    .sort_values("combined_score", ascending=False)
    .to_string(index=False)
)

FIRST MEANINGFUL TOKEN BLOCKING
Missing true matches recovered: 56 / 80

Recovered matches:
source1_entity_id candidate_entity_id                                       s1_name                                           candidate_name  name_score  address_score  combined_score s1_first_token
      S1-72310418        S2-516305116                       Burger Solution Limited                                     Sri Burger  Solution  100.000000     100.000000      100.000000         burger
     S1-491795528        S2-977672203                                Anushree Sangh                                       M/s Anushree Sangh  100.000000     100.000000      100.000000       anushree
     S1-842533642        S2-352625664           Certified Digital Associates Center                                        Certified Digital  100.000000     100.000000      100.000000      certified
     S1-730719211        S2-467798366                         Renee G. Garcia, O.D.                             

In [26]:
# STEP 12 — Self-contained blocker analysis

import pandas as pd

inspect = pd.DataFrame(missing_analysis)

# ---------- Name blockers ----------

def char_key(text, n=4):
    text = normalize_text(text).replace(" ", "")
    return text[:n] if len(text) >= n else ""

NAME_PREFIXES = {
    "mr", "mrs", "ms", "miss", "dr", "prof",
    "m", "sri", "smt", "shri", "m/s", "msr"
}

def first_meaningful_token(name):
    parts = normalize_text(name).split()

    for part in parts:
        if part not in NAME_PREFIXES and len(part) >= 3:
            return part

    return ""

def sorted_token_key(text):
    parts = [
        x for x in normalize_text(text).split()
        if len(x) >= 2
    ]
    return "|".join(sorted(set(parts)))

# Character-prefix
inspect["s1_char4"] = inspect["s1_name"].apply(char_key)
inspect["candidate_char4"] = inspect["candidate_name"].apply(char_key)

inspect["same_char4"] = (
    (inspect["s1_char4"] == inspect["candidate_char4"]) &
    (inspect["s1_char4"] != "")
)

# First meaningful token
inspect["s1_first_token"] = inspect["s1_name"].apply(
    first_meaningful_token
)

inspect["candidate_first_token"] = inspect["candidate_name"].apply(
    first_meaningful_token
)

inspect["same_first_token"] = (
    (inspect["s1_first_token"] == inspect["candidate_first_token"]) &
    (inspect["s1_first_token"] != "")
)

# Sorted tokens
inspect["s1_sorted_name"] = inspect["s1_name"].apply(
    sorted_token_key
)

inspect["candidate_sorted_name"] = inspect["candidate_name"].apply(
    sorted_token_key
)

inspect["same_sorted_name"] = (
    (inspect["s1_sorted_name"] == inspect["candidate_sorted_name"]) &
    (inspect["s1_sorted_name"] != "")
)

# ---------- Address character blocker ----------

def address_char4(address):
    text = normalize_address(address).replace(" ", "")
    return text[:4] if len(text) >= 4 else ""

inspect["s1_addr_char4"] = inspect["s1_address"].apply(
    address_char4
)

inspect["candidate_addr_char4"] = inspect["candidate_address"].apply(
    address_char4
)

inspect["same_addr_char4"] = (
    (inspect["s1_addr_char4"] == inspect["candidate_addr_char4"]) &
    (inspect["s1_addr_char4"] != "")
)

# ---------- Combined results ----------

inspect["combined_name_block"] = (
    inspect["same_char4"] |
    inspect["same_first_token"] |
    inspect["same_sorted_name"]
)

inspect["combined_all_block"] = (
    inspect["combined_name_block"] |
    inspect["same_addr_char4"]
)

print("=============================================")
print("STEP 12 BLOCKER RESULTS")
print("=============================================")

print(
    "Character-prefix name:",
    inspect["same_char4"].sum(),
    "/ 80"
)

print(
    "First meaningful token:",
    inspect["same_first_token"].sum(),
    "/ 80"
)

print(
    "Sorted-token name:",
    inspect["same_sorted_name"].sum(),
    "/ 80"
)

print(
    "Address character-prefix:",
    inspect["same_addr_char4"].sum(),
    "/ 80"
)

print(
    "Combined name blockers:",
    inspect["combined_name_block"].sum(),
    "/ 80"
)

print(
    "Combined name + address blockers:",
    inspect["combined_all_block"].sum(),
    "/ 80"
)

print("\n=============================================")
print("SORTED-TOKEN MATCHES")
print("=============================================")

sorted_matches = inspect[inspect["same_sorted_name"]]

if len(sorted_matches) == 0:
    print("No matches found.")
else:
    print(
        sorted_matches[
            [
                "source1_entity_id",
                "candidate_entity_id",
                "s1_name",
                "candidate_name",
                "name_score",
                "address_score",
                "combined_score"
            ]
        ]
        .sort_values("combined_score", ascending=False)
        .to_string(index=False)
    )

print("\n=============================================")
print("ADDRESS PREFIX MATCHES")
print("=============================================")

addr_matches = inspect[inspect["same_addr_char4"]]

if len(addr_matches) == 0:
    print("No matches found.")
else:
    print(
        addr_matches[
            [
                "source1_entity_id",
                "candidate_entity_id",
                "s1_address",
                "candidate_address",
                "name_score",
                "address_score",
                "combined_score",
                "s1_addr_char4"
            ]
        ]
        .sort_values("combined_score", ascending=False)
        .to_string(index=False)
    )

print("\n=============================================")
print("REMAINING AFTER ALL BLOCKERS")
print("=============================================")

remaining = inspect[~inspect["combined_all_block"]]

print("Remaining:", len(remaining))

print(
    remaining[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "s1_name",
            "candidate_name",
            "name_score",
            "address_score",
            "combined_score"
        ]
    ]
    .sort_values("combined_score", ascending=False)
    .to_string(index=False)
)

STEP 12 BLOCKER RESULTS
Character-prefix name: 57 / 80
First meaningful token: 56 / 80
Sorted-token name: 9 / 80
Address character-prefix: 17 / 80
Combined name blockers: 66 / 80
Combined name + address blockers: 70 / 80

SORTED-TOKEN MATCHES
source1_entity_id candidate_entity_id                             s1_name                       candidate_name  name_score  address_score  combined_score
     S1-491795528        S2-977672203                      Anushree Sangh                   M/s Anushree Sangh       100.0     100.000000      100.000000
     S1-730719211        S2-467798366               Renee G. Garcia, O.D.                     Renee G. Garcia,       100.0      97.777778       99.111111
     S1-432073164        S3-520531285             Fayth Fritz Lifeco Corp      Fayth Fritz Lifeco Lifeco Corp.       100.0      95.890411       98.356164
     S1-550488938        S3-772751614             Jaipur (india) Partners              Jaipur Partners (India)       100.0      90.740741    

In [1]:
# STEP 13 — Measure blocker-key frequency on TRAINING S2/S3
# This does NOT generate candidates yet.

import re
from collections import Counter

def name_char4_key(name):
    text = normalize_text(name).replace(" ", "")
    return text[:4] if len(text) >= 4 else ""

def first_meaningful_token_v2(name):
    # Remove common legal/title words before taking the first useful token
    ignored = {
        "mr", "mrs", "ms", "miss", "dr", "prof",
        "sri", "smt", "shri", "m/s",
        "llc", "ltd", "limited", "inc", "incorporated",
        "corp", "corporation", "company", "co", "pc",
        "private", "pvt"
    }

    parts = normalize_text(name).split()

    for part in parts:
        if part not in ignored and len(part) >= 3:
            return part

    return ""

def sorted_name_key(name):
    parts = [
        x for x in normalize_text(name).split()
        if len(x) >= 2
    ]
    return "|".join(sorted(set(parts)))

def addr_char4_key(address):
    text = normalize_address(address).replace(" ", "")
    return text[:4] if len(text) >= 4 else ""


# Count frequencies in training S2 and S3
s2_counts = {
    "name_char4": Counter(),
    "first_token": Counter(),
    "sorted_name": Counter(),
    "addr_char4": Counter()
}

s3_counts = {
    "name_char4": Counter(),
    "first_token": Counter(),
    "sorted_name": Counter(),
    "addr_char4": Counter()
}

print("Scanning TRAIN S2...")

obj = s3.get_object(
    Bucket=BUCKET,
    Key="dataset/train/train_source2.tsv"
)

for chunk in pd.read_csv(
    obj["Body"],
    sep="\t",
    chunksize=100000,
    usecols=["business_name", "business_address"]
):
    for name in chunk["business_name"].dropna():
        s2_counts["name_char4"][name_char4_key(name)] += 1
        s2_counts["first_token"][first_meaningful_token_v2(name)] += 1
        s2_counts["sorted_name"][sorted_name_key(name)] += 1

    for address in chunk["business_address"].dropna():
        s2_counts["addr_char4"][addr_char4_key(address)] += 1


print("Scanning TRAIN S3...")

obj = s3.get_object(
    Bucket=BUCKET,
    Key="dataset/train/train_source3.tsv"
)

for chunk in pd.read_csv(
    obj["Body"],
    sep="\t",
    chunksize=100000,
    usecols=["business_name", "business_address"]
):
    for name in chunk["business_name"].dropna():
        s3_counts["name_char4"][name_char4_key(name)] += 1
        s3_counts["first_token"][first_meaningful_token_v2(name)] += 1
        s3_counts["sorted_name"][sorted_name_key(name)] += 1

    for address in chunk["business_address"].dropna():
        s3_counts["addr_char4"][addr_char4_key(address)] += 1


print("\n=============================================")
print("TRAINING BLOCKER FREQUENCIES")
print("=============================================")

for source, counts in [
    ("S2", s2_counts),
    ("S3", s3_counts)
]:
    print(f"\n{source}")

    for key_type, counter in counts.items():
        values = [
            (k, v)
            for k, v in counter.items()
            if k != ""
        ]

        values.sort(key=lambda x: x[1], reverse=True)

        print(f"\n{key_type}")
        print("Distinct keys:", len(values))
        print("Largest frequency:", values[0][1] if values else 0)
        print("Top 10:")

        for k, v in values[:10]:
            print(f"  {k!r}: {v}")

Scanning TRAIN S2...


NameError: name 's3' is not defined

In [2]:
# STEP 13 — SELF-CONTAINED BLOCKER FREQUENCY CHECK
# Safe: reads TRAIN S2/S3 only. Does NOT create candidate pairs.

import re
import boto3
import pandas as pd
from collections import Counter
from unidecode import unidecode

# --------------------------------------------------
# 1. AWS / S3 setup
# --------------------------------------------------

BUCKET = "sagemaker-eu-north-1-152025327817"

s3 = boto3.client(
    "s3",
    region_name="eu-north-1"
)

# --------------------------------------------------
# 2. Normalization functions
# --------------------------------------------------

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = unidecode(text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_name(name):
    text = normalize_text(name)

    suffixes = [
        "private limited",
        "pvt ltd",
        "pvt",
        "limited",
        "ltd",
        "llc",
        "incorporated",
        "inc",
        "corporation",
        "corp",
        "company",
        "co",
        "pc"
    ]

    for suffix in suffixes:
        text = re.sub(
            rf"\b{suffix}\b",
            " ",
            text
        )

    return re.sub(r"\s+", " ", text).strip()


def normalize_address(address):
    text = normalize_text(address)

    replacements = {
        "street": "st",
        "road": "rd",
        "avenue": "ave",
        "boulevard": "blvd",
        "drive": "dr",
        "lane": "ln",
        "highway": "hwy",
        "apartment": "apt",
        "suite": "ste",
        "floor": "fl"
    }

    for full, short in replacements.items():
        text = re.sub(
            rf"\b{full}\b",
            short,
            text
        )

    return re.sub(r"\s+", " ", text).strip()


# --------------------------------------------------
# 3. Blocker keys
# --------------------------------------------------

def name_char4_key(name):
    text = normalize_text(name).replace(" ", "")

    if len(text) < 4:
        return ""

    return text[:4]


def first_meaningful_token(name):
    ignored = {
        "mr", "mrs", "ms", "miss", "dr", "prof",
        "sri", "smt", "shri", "m/s",
        "llc", "ltd", "limited",
        "inc", "incorporated",
        "corp", "corporation",
        "company", "co", "pc",
        "private", "pvt"
    }

    parts = normalize_text(name).split()

    for part in parts:
        if part not in ignored and len(part) >= 3:
            return part

    return ""


def sorted_name_key(name):
    parts = [
        x
        for x in normalize_text(name).split()
        if len(x) >= 2
    ]

    return "|".join(sorted(set(parts)))


def addr_char4_key(address):
    text = normalize_address(address).replace(" ", "")

    if len(text) < 4:
        return ""

    return text[:4]


# --------------------------------------------------
# 4. Counters
# --------------------------------------------------

s2_counts = {
    "name_char4": Counter(),
    "first_token": Counter(),
    "sorted_name": Counter(),
    "addr_char4": Counter()
}

s3_counts = {
    "name_char4": Counter(),
    "first_token": Counter(),
    "sorted_name": Counter(),
    "addr_char4": Counter()
}


# --------------------------------------------------
# 5. Scan TRAIN S2
# --------------------------------------------------

print("Scanning TRAIN S2...")

obj = s3.get_object(
    Bucket=BUCKET,
    Key="dataset/train/train_source2.tsv"
)

for chunk in pd.read_csv(
    obj["Body"],
    sep="\t",
    chunksize=100000,
    usecols=[
        "business_name",
        "business_address"
    ]
):

    for name in chunk["business_name"].dropna():
        s2_counts["name_char4"][
            name_char4_key(name)
        ] += 1

        s2_counts["first_token"][
            first_meaningful_token(name)
        ] += 1

        s2_counts["sorted_name"][
            sorted_name_key(name)
        ] += 1

    for address in chunk["business_address"].dropna():
        s2_counts["addr_char4"][
            addr_char4_key(address)
        ] += 1


print("TRAIN S2 complete.")


# --------------------------------------------------
# 6. Scan TRAIN S3
# --------------------------------------------------

print("Scanning TRAIN S3...")

obj = s3.get_object(
    Bucket=BUCKET,
    Key="dataset/train/train_source3.tsv"
)

for chunk in pd.read_csv(
    obj["Body"],
    sep="\t",
    chunksize=100000,
    usecols=[
        "business_name",
        "business_address"
    ]
):

    for name in chunk["business_name"].dropna():
        s3_counts["name_char4"][
            name_char4_key(name)
        ] += 1

        s3_counts["first_token"][
            first_meaningful_token(name)
        ] += 1

        s3_counts["sorted_name"][
            sorted_name_key(name)
        ] += 1

    for address in chunk["business_address"].dropna():
        s3_counts["addr_char4"][
            addr_char4_key(address)
        ] += 1


print("TRAIN S3 complete.")


# --------------------------------------------------
# 7. Display results
# --------------------------------------------------

print("\n=============================================")
print("TRAINING BLOCKER FREQUENCIES")
print("=============================================")

for source, counts in [
    ("S2", s2_counts),
    ("S3", s3_counts)
]:

    print(f"\n================ {source} ================")

    for key_type, counter in counts.items():

        values = [
            (k, v)
            for k, v in counter.items()
            if k != ""
        ]

        values.sort(
            key=lambda x: x[1],
            reverse=True
        )

        print(f"\n--- {key_type} ---")
        print("Distinct keys:", len(values))

        if values:
            print(
                "Largest frequency:",
                values[0][1]
            )

            print("Top 10:")

            for k, v in values[:10]:
                print(f"  {k!r}: {v}")
        else:
            print("No usable keys.")

Scanning TRAIN S2...
TRAIN S2 complete.
Scanning TRAIN S3...
TRAIN S3 complete.

TRAINING BLOCKER FREQUENCIES

================ S2 ================

--- name_char4 ---
Distinct keys: 157750
Largest frequency: 41059
Top 10:
  'priv': 41059
  'shri': 35346
  'inte': 33026
  'pedi': 30795
  'gold': 22695
  'shiv': 20940
  'prim': 19061
  'silv': 16592
  'glob': 16212
  'nort': 16121

--- first_token ---
Distinct keys: 368108
Largest frequency: 42649
Top 10:
  'the': 42649
  'pediatric': 30655
  'services': 17826
  'new': 17118
  'india': 15095
  'blue': 14620
  'global': 13807
  'family': 13453
  'vision': 13179
  'golden': 12870

--- sorted_name ---
Distinct keys: 3709989
Largest frequency: 564
Top 10:
  'physical|therapy': 564
  'care|primary': 559
  'health|womens': 517
  'care|urgent': 501
  'dental|pediatric': 501
  'internal|medicine': 499
  'behavioral|health': 498
  'dentistry|pediatric': 480
  'ankle|foot': 390
  'ear|nose|throat': 378

--- addr_char4 ---
Distinct keys: 173819
La

In [3]:
# STEP 14 — Analyze the final 10 difficult matches
# No S3/S2 scan. Uses the existing `missing_analysis`.

import pandas as pd
import re

inspect = pd.DataFrame(missing_analysis)

# Recreate blocker columns safely
def normalize_tokens(text):
    return set(
        x for x in normalize_text(text).split()
        if len(x) >= 2
    )

def name_tokens(text):
    return set(
        x for x in normalize_text(text).split()
        if len(x) >= 2
    )

def address_tokens(text):
    return set(
        x for x in normalize_address(text).split()
        if len(x) >= 2
    )

# Recreate the previous blockers
def char_key(text, n=4):
    text = normalize_text(text).replace(" ", "")
    return text[:n] if len(text) >= n else ""

NAME_PREFIXES = {
    "mr", "mrs", "ms", "miss", "dr", "prof",
    "m", "sri", "smt", "shri", "m/s", "msr"
}

def first_meaningful_token(name):
    parts = normalize_text(name).split()

    for part in parts:
        if part not in NAME_PREFIXES and len(part) >= 3:
            return part

    return ""

def sorted_token_key(text):
    parts = [
        x for x in normalize_text(text).split()
        if len(x) >= 2
    ]
    return "|".join(sorted(set(parts)))

def address_char4(address):
    text = normalize_address(address).replace(" ", "")
    return text[:4] if len(text) >= 4 else ""

inspect["same_char4"] = (
    inspect["s1_name"].apply(char_key)
    ==
    inspect["candidate_name"].apply(char_key)
)

inspect["same_first_token"] = (
    inspect["s1_name"].apply(first_meaningful_token)
    ==
    inspect["candidate_name"].apply(first_meaningful_token)
)

inspect["same_sorted_name"] = (
    inspect["s1_name"].apply(sorted_token_key)
    ==
    inspect["candidate_name"].apply(sorted_token_key)
)

inspect["same_addr_char4"] = (
    inspect["s1_address"].apply(address_char4)
    ==
    inspect["candidate_address"].apply(address_char4)
)

inspect["combined_all_block"] = (
    inspect["same_char4"]
    | inspect["same_first_token"]
    | inspect["same_sorted_name"]
    | inspect["same_addr_char4"]
)

remaining = inspect[
    ~inspect["combined_all_block"]
].copy()

# --------------------------------------------------
# Exact token overlap
# --------------------------------------------------

remaining["shared_name_tokens"] = remaining.apply(
    lambda r: sorted(
        name_tokens(r["s1_name"])
        &
        name_tokens(r["candidate_name"])
    ),
    axis=1
)

remaining["shared_address_tokens"] = remaining.apply(
    lambda r: sorted(
        address_tokens(r["s1_address"])
        &
        address_tokens(r["candidate_address"])
    ),
    axis=1
)

remaining["name_token_count"] = (
    remaining["shared_name_tokens"].apply(len)
)

remaining["address_token_count"] = (
    remaining["shared_address_tokens"].apply(len)
)

# --------------------------------------------------
# Domain / email-like / compact-name diagnostics
# --------------------------------------------------

def compact_name(text):
    return re.sub(
        r"[^a-z0-9]",
        "",
        normalize_text(text)
    )

remaining["s1_compact_name"] = (
    remaining["s1_name"].apply(compact_name)
)

remaining["candidate_compact_name"] = (
    remaining["candidate_name"].apply(compact_name)
)

def extract_domain_like(text):
    text = normalize_text(text)

    # Remove common website prefixes/suffixes
    text = re.sub(r"\bwww\b", " ", text)
    text = re.sub(r"\bhttps?\b", " ", text)
    text = re.sub(r"\bcom\b", " ", text)
    text = re.sub(r"\borg\b", " ", text)
    text = re.sub(r"\bnet\b", " ", text)

    return re.sub(r"\s+", " ", text).strip()

remaining["candidate_domain_clean"] = (
    remaining["candidate_name"]
    .apply(extract_domain_like)
)

# --------------------------------------------------
# Display
# --------------------------------------------------

print("=============================================")
print("STEP 14 — FINAL 10 DIAGNOSTIC")
print("=============================================")

print("Remaining:", len(remaining))

for _, r in remaining.iterrows():

    print("\n---------------------------------------------")

    print("S1:", r["source1_entity_id"])
    print("Candidate:", r["candidate_entity_id"])

    print("\nS1 name:")
    print(r["s1_name"])

    print("\nCandidate name:")
    print(r["candidate_name"])

    print("\nS1 address:")
    print(r["s1_address"])

    print("\nCandidate address:")
    print(r["candidate_address"])

    print("\nScores:")
    print(
        "Name:",
        round(r["name_score"], 2),
        "| Address:",
        round(r["address_score"], 2),
        "| Combined:",
        round(r["combined_score"], 2)
    )

    print("\nShared name tokens:")
    print(r["shared_name_tokens"])

    print("Shared address tokens:")
    print(r["shared_address_tokens"])

    print(
        "Name token overlap:",
        r["name_token_count"]
    )

    print(
        "Address token overlap:",
        r["address_token_count"]
    )

    print(
        "S1 compact name:",
        r["s1_compact_name"]
    )

    print(
        "Candidate compact name:",
        r["candidate_compact_name"]
    )

    print(
        "Candidate domain-clean:",
        r["candidate_domain_clean"]
    )

NameError: name 'missing_analysis' is not defined

In [4]:
# STEP 14 — SELF-CONTAINED FINAL 10 ANALYSIS
# Retrieves ONLY the 80 previously identified difficult candidate pairs.

import re
import boto3
import pandas as pd
from rapidfuzz.fuzz import token_set_ratio
from unidecode import unidecode

# ============================================================
# 1. AWS / S3 setup
# ============================================================

BUCKET = "sagemaker-eu-north-1-152025327817"

s3 = boto3.client(
    "s3",
    region_name="eu-north-1"
)

S1_KEY = "dataset/train/train_source1.tsv"
S2_KEY = "dataset/train/train_source2.tsv"
S3_KEY = "dataset/train/train_source3.tsv"


# ============================================================
# 2. Normalization
# ============================================================

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = unidecode(text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_name(name):
    text = normalize_text(name)

    suffixes = [
        "private limited",
        "pvt ltd",
        "pvt",
        "limited",
        "ltd",
        "llc",
        "incorporated",
        "inc",
        "corporation",
        "corp",
        "company",
        "co",
        "pc"
    ]

    for suffix in suffixes:
        text = re.sub(
            rf"\b{suffix}\b",
            " ",
            text
        )

    return re.sub(r"\s+", " ", text).strip()


def normalize_address(address):
    text = normalize_text(address)

    replacements = {
        "street": "st",
        "road": "rd",
        "avenue": "ave",
        "boulevard": "blvd",
        "drive": "dr",
        "lane": "ln",
        "highway": "hwy",
        "apartment": "apt",
        "suite": "ste",
        "floor": "fl"
    }

    for full, short in replacements.items():
        text = re.sub(
            rf"\b{full}\b",
            short,
            text
        )

    return re.sub(r"\s+", " ", text).strip()


# ============================================================
# 3. The 80 known difficult true pairs
# ============================================================

missing_pairs = [
    ("S1-842533642", "S2-352625664"),
    ("S1-491795528", "S2-977672203"),
    ("S1-72310418", "S2-516305116"),
    ("S1-730719211", "S2-467798366"),
    ("S1-9962387", "S3-13637230"),
    ("S1-432073164", "S3-520531285"),
    ("S1-314714647", "S3-837108914"),
    ("S1-389414274", "S3-993901020"),
    ("S1-503957000", "S3-555791452"),
    ("S1-876102895", "S2-358560681"),

    ("S1-550488938", "S3-651095723"),
    ("S1-212509374", "S2-598327313"),
    ("S1-731542669", "S2-939487122"),
    ("S1-876102895", "S2-791711694"),
    ("S1-965667", "S3-11291185"),
    ("S1-561341312", "S2-483615364"),
    ("S1-818149988", "S3-965578558"),
    ("S1-185938539", "S3-846884848"),
    ("S1-541270202", "S3-31133661"),
    ("S1-463669205", "S2-301757320"),

    ("S1-72310418", "S3-731160199"),
    ("S1-72310418", "S3-570980846"),
    ("S1-518197819", "S2-891724637"),
    ("S1-9962387", "S3-160760047"),
    ("S1-502736054", "S3-537652714"),
    ("S1-502736054", "S3-82080725"),
    ("S1-567308588", "S2-191200521"),
    ("S1-567308588", "S2-226928496"),
    ("S1-116043204", "S3-85523430"),
    ("S1-170252930", "S3-228972158"),

    ("S1-170252930", "S2-233855016"),
    ("S1-67172650", "S3-251407198"),
    ("S1-67172650", "S3-520282171"),
    ("S1-67172650", "S2-74945214"),
    ("S1-550488938", "S3-772751614"),
    ("S1-550488938", "S2-918671514"),
    ("S1-656753428", "S3-679606215"),
    ("S1-656753428", "S2-24659151"),
    ("S1-656753428", "S2-153058913"),
    ("S1-389414274", "S2-933610960"),

    ("S1-389414274", "S3-443891409"),
    ("S1-389414274", "S2-149253287"),
    ("S1-389414274", "S2-24125756"),
    ("S1-212509374", "S2-682745911"),
    ("S1-731542669", "S3-988120505"),
    ("S1-876102895", "S2-358560681"),
    ("S1-965667", "S3-860443364"),
    ("S1-965667", "S3-11291185"),
    ("S1-185938539", "S3-880695057"),
    ("S1-185938539", "S3-846884848"),

    ("S1-385608858", "S3-393804899"),
    ("S1-72444401", "S2-973800896"),
    ("S1-72444401", "S2-503949813"),
    ("S1-777597828", "S3-413669121"),
    ("S1-314714647", "S3-192802051"),
    ("S1-656753428", "S2-153058913"),
    ("S1-656753428", "S2-24659151"),
    ("S1-656753428", "S3-679606215"),
    ("S1-546142636", "S3-729771680"),
    ("S1-818149988", "S3-965578558"),

    ("S1-385608858", "S3-393804899"),
    ("S1-502736054", "S3-537652714"),
    ("S1-561341312", "S2-483615364"),
    ("S1-318373630", "S2-660036492"),
    ("S1-567308588", "S2-191200521"),
    ("S1-314714647", "S3-192802051"),
    ("S1-656753428", "S2-153058913"),
    ("S1-656753428", "S2-24659151"),
    ("S1-72444401", "S2-973800896"),
    ("S1-72444401", "S2-503949813"),

    ("S1-965667", "S3-11291185"),
    ("S1-385608858", "S3-393804899"),
    ("S1-567308588", "S2-191200521"),
    ("S1-818149988", "S3-965578558"),
    ("S1-546142636", "S3-729771680"),
    ("S1-502736054", "S3-537652714"),
    ("S1-561341312", "S2-483615364"),
    ("S1-318373630", "S2-660036492"),
    ("S1-314714647", "S3-192802051"),
    ("S1-656753428", "S2-153058913")
]

# Remove accidental duplicate pairs
missing_pairs = list(dict.fromkeys(missing_pairs))

print("Pairs to retrieve:", len(missing_pairs))


# ============================================================
# 4. IDs needed
# ============================================================

s1_ids = set(x[0] for x in missing_pairs)
s2_ids = set(x[1] for x in missing_pairs if x[1].startswith("S2-"))
s3_ids = set(x[1] for x in missing_pairs if x[1].startswith("S3-"))

print("S1 IDs:", len(s1_ids))
print("S2 IDs:", len(s2_ids))
print("S3 IDs:", len(s3_ids))


# ============================================================
# 5. Safe targeted retrieval
# ============================================================

def collect_records(key, wanted_ids):
    found = {}

    obj = s3.get_object(
        Bucket=BUCKET,
        Key=key
    )

    for chunk in pd.read_csv(
        obj["Body"],
        sep="\t",
        chunksize=100000,
        usecols=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    ):

        selected = chunk[
            chunk["entity_id"].isin(wanted_ids)
        ]

        for row in selected.itertuples(index=False):

            found[row.entity_id] = {
                "business_name": row.business_name,
                "business_address": row.business_address,
                "country": row.country
            }

        if len(found) == len(wanted_ids):
            break

    return found


print("\nRetrieving S1...")
s1_records = collect_records(
    S1_KEY,
    s1_ids
)

print("Found:", len(s1_records), "/", len(s1_ids))


print("\nRetrieving S2...")
s2_records = collect_records(
    S2_KEY,
    s2_ids
)

print("Found:", len(s2_records), "/", len(s2_ids))


print("\nRetrieving S3...")
s3_records = collect_records(
    S3_KEY,
    s3_ids
)

print("Found:", len(s3_records), "/", len(s3_ids))


# ============================================================
# 6. Build analysis
# ============================================================

rows = []

for s1_id, candidate_id in missing_pairs:

    if s1_id not in s1_records:
        continue

    if candidate_id.startswith("S2-"):
        candidate = s2_records.get(candidate_id)
    else:
        candidate = s3_records.get(candidate_id)

    if candidate is None:
        continue

    s1 = s1_records[s1_id]

    s1_name = normalize_name(
        s1["business_name"]
    )

    candidate_name = normalize_name(
        candidate["business_name"]
    )

    s1_address = normalize_address(
        s1["business_address"]
    )

    candidate_address = normalize_address(
        candidate["business_address"]
    )

    name_score = token_set_ratio(
        s1_name,
        candidate_name
    )

    address_score = token_set_ratio(
        s1_address,
        candidate_address
    )

    combined_score = (
        0.6 * name_score
        +
        0.4 * address_score
    )

    rows.append({
        "source1_entity_id": s1_id,
        "candidate_entity_id": candidate_id,
        "s1_name": s1["business_name"],
        "candidate_name": candidate["business_name"],
        "s1_address": s1["business_address"],
        "candidate_address": candidate["business_address"],
        "name_score": name_score,
        "address_score": address_score,
        "combined_score": combined_score
    })


inspect = pd.DataFrame(rows)


# ============================================================
# 7. Analyze token overlap
# ============================================================

def token_set(text):
    return set(
        x
        for x in normalize_text(text).split()
        if len(x) >= 2
    )


inspect["shared_name_tokens"] = inspect.apply(
    lambda r: sorted(
        token_set(r["s1_name"])
        &
        token_set(r["candidate_name"])
    ),
    axis=1
)

inspect["shared_address_tokens"] = inspect.apply(
    lambda r: sorted(
        token_set(r["s1_address"])
        &
        token_set(r["candidate_address"])
    ),
    axis=1
)

inspect["name_token_count"] = (
    inspect["shared_name_tokens"].apply(len)
)

inspect["address_token_count"] = (
    inspect["shared_address_tokens"].apply(len)
)


# ============================================================
# 8. Display the difficult cases
# ============================================================

print("\n=============================================")
print("STEP 14 RESULTS")
print("=============================================")

print("Retrieved pairs:", len(inspect))

for _, r in inspect.sort_values(
    "combined_score",
    ascending=False
).iterrows():

    print("\n---------------------------------------------")

    print(
        r["source1_entity_id"],
        "->",
        r["candidate_entity_id"]
    )

    print(
        "S1 NAME:",
        r["s1_name"]
    )

    print(
        "CANDIDATE NAME:",
        r["candidate_name"]
    )

    print(
        "S1 ADDRESS:",
        r["s1_address"]
    )

    print(
        "CANDIDATE ADDRESS:",
        r["candidate_address"]
    )

    print(
        "NAME SCORE:",
        round(r["name_score"], 2)
    )

    print(
        "ADDRESS SCORE:",
        round(r["address_score"], 2)
    )

    print(
        "COMBINED:",
        round(r["combined_score"], 2)
    )

    print(
        "SHARED NAME TOKENS:",
        r["shared_name_tokens"]
    )

    print(
        "SHARED ADDRESS TOKENS:",
        r["shared_address_tokens"]
    )

Pairs to retrieve: 54
S1 IDs: 31
S2 IDs: 25
S3 IDs: 29

Retrieving S1...
Found: 31 / 31

Retrieving S2...
Found: 25 / 25

Retrieving S3...
Found: 29 / 29

STEP 14 RESULTS
Retrieved pairs: 54

---------------------------------------------
S1-842533642 -> S2-352625664
S1 NAME: Certified Digital Associates Center
CANDIDATE NAME: Certified Digital
S1 ADDRESS: 6984 Vining Street, Chino, CA
CANDIDATE ADDRESS: 6984-6988 VINING ST, CHINO CIYT, CA
NAME SCORE: 100.0
ADDRESS SCORE: 100.0
COMBINED: 100.0
SHARED NAME TOKENS: ['certified', 'digital']
SHARED ADDRESS TOKENS: ['6984', 'ca', 'chino', 'vining']

---------------------------------------------
S1-491795528 -> S2-977672203
S1 NAME: Anushree Sangh
CANDIDATE NAME: M/s Anushree Sangh
S1 ADDRESS: Row House No B4, 14 /15, Hill View, Surekha Niwas Siddhivinayak Nagari, Nigdi, Pune, Maharashtra
CANDIDATE ADDRESS: HN 31 ROW HOUSE NO B4, 14 /15, HILL VIEW, SUREKHA NIWAS SIDDHIVINAYAK NAGARI, NIGDI, PUNE, Maharashtra
NAME SCORE: 100.0
ADDRESS SCORE: 1

In [ ]:
# STEP 15 — TRAINING ADDRESS-TOKEN OVERLAP BLOCKER
# Tests whether >=2 shared address tokens can recover difficult matches.
# Training only. No test candidate generation.

import re
import boto3
import pandas as pd
from collections import defaultdict

# ============================================================
# 1. Setup
# ============================================================

BUCKET = "sagemaker-eu-north-1-152025327817"

s3 = boto3.client(
    "s3",
    region_name="eu-north-1"
)

TRAIN_S1_KEY = "dataset/train/train_source1.tsv"
TRAIN_S2_KEY = "dataset/train/train_source2.tsv"
TRAIN_S3_KEY = "dataset/train/train_source3.tsv"
GROUND_TRUTH_KEY = "dataset/train/train_ground_truth.tsv"


# ============================================================
# 2. Normalization
# ============================================================

from unidecode import unidecode

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = unidecode(text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_address(address):
    text = normalize_text(address)

    replacements = {
        "street": "st",
        "road": "rd",
        "avenue": "ave",
        "boulevard": "blvd",
        "drive": "dr",
        "lane": "ln",
        "highway": "hwy",
        "apartment": "apt",
        "suite": "ste",
        "floor": "fl"
    }

    for full, short in replacements.items():
        text = re.sub(
            rf"\b{full}\b",
            short,
            text
        )

    return re.sub(r"\s+", " ", text).strip()


def address_tokens(address):
    text = normalize_address(address)

    tokens = [
        x for x in text.split()
        if len(x) >= 2
    ]

    return set(tokens)


# ============================================================
# 3. Load the SAME 100 training Source1 records
# ============================================================

print("Loading training Source1 sample...")

obj = s3.get_object(
    Bucket=BUCKET,
    Key=TRAIN_S1_KEY
)

train_s1 = pd.read_csv(
    obj["Body"],
    sep="\t",
    nrows=100,
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

print("Training Source1:", len(train_s1))


# ============================================================
# 4. Load ground truth for those 100
# ============================================================

obj = s3.get_object(
    Bucket=BUCKET,
    Key=GROUND_TRUTH_KEY
)

ground_truth = pd.read_csv(
    obj["Body"],
    sep="\t"
)

ground_truth = ground_truth[
    ground_truth["source1_entity_id"].isin(
        set(train_s1["entity_id"])
    )
].copy()

print("Ground-truth Source1:", len(ground_truth))


# ============================================================
# 5. Build true-pair set
# ============================================================

true_pairs = set()

for row in ground_truth.itertuples(index=False):

    s1_id = row.source1_entity_id

    matched = row.matched_entity_ids

    if pd.isna(matched):
        continue

    matched = str(matched).strip()

    if not matched:
        continue

    for candidate_id in matched.split(","):

        candidate_id = candidate_id.strip()

        if candidate_id:
            true_pairs.add(
                (s1_id, candidate_id)
            )

print("True pairs:", len(true_pairs))


# ============================================================
# 6. Build Source1 address-token index
# ============================================================

s1_address_tokens = {}

token_to_s1 = defaultdict(set)

for row in train_s1.itertuples(index=False):

    tokens = address_tokens(
        row.business_address
    )

    s1_address_tokens[row.entity_id] = tokens

    for token in tokens:
        token_to_s1[token].add(
            row.entity_id
        )

print(
    "Distinct address tokens:",
    len(token_to_s1)
)


# ============================================================
# 7. Scan TRAIN S2
# ============================================================

candidate_pairs = set()

print("\nScanning TRAIN S2...")

obj = s3.get_object(
    Bucket=BUCKET,
    Key=TRAIN_S2_KEY
)

for chunk in pd.read_csv(
    obj["Body"],
    sep="\t",
    chunksize=100000,
    usecols=[
        "entity_id",
        "business_address",
        "country"
    ]
):

    for row in chunk.itertuples(index=False):

        candidate_tokens = address_tokens(
            row.business_address
        )

        if not candidate_tokens:
            continue

        possible_s1 = set()

        for token in candidate_tokens:
            possible_s1.update(
                token_to_s1.get(token, ())
            )

        for s1_id in possible_s1:

            shared = (
                candidate_tokens
                &
                s1_address_tokens[s1_id]
            )

            if len(shared) >= 2:

                candidate_pairs.add(
                    (s1_id, row.entity_id)
                )


# ============================================================
# 8. Scan TRAIN S3
# ============================================================

print("Scanning TRAIN S3...")

obj = s3.get_object(
    Bucket=BUCKET,
    Key=TRAIN_S3_KEY
)

for chunk in pd.read_csv(
    obj["Body"],
    sep="\t",
    chunksize=100000,
    usecols=[
        "entity_id",
        "business_address",
        "country"
    ]
):

    for row in chunk.itertuples(index=False):

        candidate_tokens = address_tokens(
            row.business_address
        )

        if not candidate_tokens:
            continue

        possible_s1 = set()

        for token in candidate_tokens:
            possible_s1.update(
                token_to_s1.get(token, ())
            )

        for s1_id in possible_s1:

            shared = (
                candidate_tokens
                &
                s1_address_tokens[s1_id]
            )

            if len(shared) >= 2:

                candidate_pairs.add(
                    (s1_id, row.entity_id)
                )


# ============================================================
# 9. Evaluate
# ============================================================

true_recovered = (
    true_pairs
    &
    candidate_pairs
)

print("\n=============================================")
print("STEP 15 RESULTS")
print("=============================================")

print(
    "Address-token candidates:",
    len(candidate_pairs)
)

print(
    "True pairs:",
    len(true_pairs)
)

print(
    "True pairs recovered:",
    len(true_recovered)
)

print(
    "Candidate recall:",
    round(
        100 * len(true_recovered) / len(true_pairs),
        2
    ),
    "%"
)

print(
    "False candidate pairs:",
    len(candidate_pairs - true_pairs)
)


# ============================================================
# 10. Specifically inspect the previously difficult matches
# ============================================================

difficult_s1 = {
    "S1-546142636",
    "S1-818149988",
    "S1-385608858",
    "S1-502736054",
    "S1-561341312",
    "S1-318373630",
    "S1-567308588",
    "S1-314714647",
    "S1-656753428",
    "S1-777597828",
    "S1-72444401",
    "S1-965667",
    "S1-541270202",
    "S1-185938539",
    "S1-116043204"
}

difficult_true = {
    pair
    for pair in true_pairs
    if pair[0] in difficult_s1
}

difficult_recovered = (
    difficult_true
    &
    candidate_pairs
)

print("\n=============================================")
print("DIFFICULT MATCH RECOVERY")
print("=============================================")

print(
    "Difficult true pairs:",
    len(difficult_true)
)

print(
    "Recovered:",
    len(difficult_recovered)
)

print(
    "Recovery:",
    round(
        100 * len(difficult_recovered)
        / len(difficult_true),
        2
    ),
    "%"
)

print("\nRecovered difficult pairs:")

for pair in sorted(difficult_recovered):
    print(
        pair[0],
        "->",
        pair[1]
    )

In [1]:
import os
import pandas as pd

print("Checking candidate files...")

files = [
    "candidate_index/exact_name_candidates.tsv",
    "candidate_index/addr_nums_candidates.tsv",
    "candidate_index/unique_candidates.tsv"
]

for path in files:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"✅ {path} — {size_mb:.2f} MB")
    else:
        print(f"❌ MISSING: {path}")

Checking candidate files...
✅ candidate_index/exact_name_candidates.tsv — 175.05 MB
✅ candidate_index/addr_nums_candidates.tsv — 65.84 MB
✅ candidate_index/unique_candidates.tsv — 240.09 MB


In [2]:
import pandas as pd

path = "candidate_index/unique_candidates.tsv"

df_check = pd.read_csv(
    path,
    sep="\t",
    nrows=5
)

print("Columns:")
print(df_check.columns.tolist())

print("\nFirst 5 rows:")
print(df_check)

print("\nTotal candidate rows:")
print(sum(1 for _ in open(path, encoding="utf-8")) - 1)

Columns:
['source1_entity_id', 'candidate_entity_id']

First 5 rows:
  source1_entity_id candidate_entity_id
0      S1-607177411         S2-89663826
1       S1-10991820        S2-884102769
2      S1-439579149        S2-375151053
3      S1-984582949        S2-767997013
4      S1-199924207        S2-292567038

Total candidate rows:
9765660


In [3]:
import pandas as pd

path = "candidate_index/unique_candidates.tsv"

s1_with_candidates = set()
candidate_count = 0

for chunk in pd.read_csv(
    path,
    sep="\t",
    chunksize=200000
):
    candidate_count += len(chunk)
    s1_with_candidates.update(chunk["source1_entity_id"])

print("Candidate pairs:", candidate_count)
print("Unique Source 1 entities with candidates:", len(s1_with_candidates))

Candidate pairs: 9765660
Unique Source 1 entities with candidates: 1099632


In [4]:
import boto3
import pandas as pd

BUCKET = "sagemaker-eu-north-1-152025327817"
TEST_S1_KEY = "dataset/test/test_source1.tsv"

s3 = boto3.client("s3", region_name="eu-north-1")

print("Loading Test Source1 IDs and fields...")

obj = s3.get_object(Bucket=BUCKET, Key=TEST_S1_KEY)

test_s1 = pd.read_csv(
    obj["Body"],
    sep="\t",
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

print("Test Source1 rows:", len(test_s1))
print("\nColumns:", test_s1.columns.tolist())
print("\nMemory used: %.2f MB" % (test_s1.memory_usage(deep=True).sum() / 1024**2))

Loading Test Source1 IDs and fields...
Test Source1 rows: 1732544

Columns: ['entity_id', 'business_name', 'business_address', 'country']

Memory used: 485.89 MB


In [6]:
import re
import unicodedata
from rapidfuzz.fuzz import token_set_ratio

def remove_accents(text):
    text = str(text)
    return "".join(
        c for c in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(c)
    )

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = remove_accents(text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def normalize_name(name):
    text = normalize_text(name)

    suffixes = [
        "private limited", "pvt ltd", "pvt", "limited", "ltd",
        "llc", "incorporated", "inc", "corporation", "corp",
        "company", "co", "pc"
    ]

    for suffix in suffixes:
        text = re.sub(rf"\b{suffix}\b", " ", text)

    return re.sub(r"\s+", " ", text).strip()

def normalize_address(address):
    text = normalize_text(address)

    replacements = {
        "street": "st",
        "road": "rd",
        "avenue": "ave",
        "boulevard": "blvd",
        "drive": "dr",
        "lane": "ln",
        "highway": "hwy",
        "apartment": "apt",
        "suite": "ste",
        "floor": "fl"
    }

    for full, short in replacements.items():
        text = re.sub(rf"\b{full}\b", short, text)

    return re.sub(r"\s+", " ", text).strip()

print("✅ Normalization and scoring functions ready.")

ModuleNotFoundError: No module named 'rapidfuzz'

In [7]:
import sys
import importlib.util

packages = ["pandas", "numpy", "sklearn", "rapidfuzz", "unidecode"]

print("Python:", sys.version.split()[0])
print()

for package in packages:
    available = importlib.util.find_spec(package) is not None
    print(f"{package}: {'✅ available' if available else '❌ not available'}")

Python: 3.12.14

pandas: ✅ available
numpy: ✅ available
sklearn: ✅ available
rapidfuzz: ❌ not available
unidecode: ❌ not available


In [8]:
import re
import unicodedata
from difflib import SequenceMatcher

def remove_accents(text):
    text = str(text)
    return "".join(
        c for c in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(c)
    )

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = remove_accents(text)

    # Keep only English letters, numbers and spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def normalize_name(name):
    text = normalize_text(name)

    suffixes = [
        "private limited",
        "pvt ltd",
        "pvt",
        "limited",
        "ltd",
        "llc",
        "incorporated",
        "inc",
        "corporation",
        "corp",
        "company",
        "co",
        "pc"
    ]

    for suffix in suffixes:
        text = re.sub(rf"\b{suffix}\b", " ", text)

    return re.sub(r"\s+", " ", text).strip()

def normalize_address(address):
    text = normalize_text(address)

    replacements = {
        "street": "st",
        "road": "rd",
        "avenue": "ave",
        "boulevard": "blvd",
        "drive": "dr",
        "lane": "ln",
        "highway": "hwy",
        "apartment": "apt",
        "suite": "ste",
        "floor": "fl"
    }

    for full, short in replacements.items():
        text = re.sub(rf"\b{full}\b", short, text)

    return re.sub(r"\s+", " ", text).strip()

def token_set_ratio(a, b):
    """
    Pure-Python approximation of RapidFuzz token_set_ratio.
    Returns a score from 0 to 100.
    """
    a = normalize_text(a)
    b = normalize_text(b)

    if not a or not b:
        return 0.0

    tokens_a = set(a.split())
    tokens_b = set(b.split())

    common = tokens_a & tokens_b

    if not common:
        return SequenceMatcher(None, a, b).ratio() * 100

    common_text = " ".join(sorted(common))
    a_text = " ".join(sorted(tokens_a))
    b_text = " ".join(sorted(tokens_b))

    score1 = SequenceMatcher(None, common_text, a_text).ratio() * 100
    score2 = SequenceMatcher(None, common_text, b_text).ratio() * 100
    score3 = SequenceMatcher(None, a_text, b_text).ratio() * 100

    return max(score1, score2, score3)

print("✅ Normalization functions ready")
print("✅ Pure-Python similarity function ready")

✅ Normalization functions ready
✅ Pure-Python similarity function ready


In [9]:
tests = [
    ("Crystal Lending PC", "Crystal Lending PC"),
    ("Dream Construction Limited", "Dream Construction Limited"),
    ("Smart Healthcare Private Limited", "Kelonyla"),
    ("Burger Solution Limited", "Sri Burger Solution")
]

print("Similarity test:\n")

for a, b in tests:
    score = token_set_ratio(a, b)
    print(f"{a}")
    print(f"  ↔ {b}")
    print(f"  Score: {score:.2f}")
    print()

Similarity test:

Crystal Lending PC
  ↔ Crystal Lending PC
  Score: 100.00

Dream Construction Limited
  ↔ Dream Construction Limited
  Score: 100.00

Smart Healthcare Private Limited
  ↔ Kelonyla
  Score: 5.00

Burger Solution Limited
  ↔ Sri Burger Solution
  Score: 88.24



In [10]:
import pandas as pd
import re

candidate_path = "candidate_index/unique_candidates.tsv"

# Build normalized lookup for Test Source1
s1_name = {}
s1_addr_nums = {}

def make_addr_nums(address):
    text = normalize_address(address)
    nums = re.findall(r"\d+", text)

    seen = set()
    result = []

    for n in nums:
        if n not in seen:
            seen.add(n)
            result.append(n)

    return "|".join(result)

for row in test_s1.itertuples(index=False):
    s1_name[row.entity_id] = normalize_name(row.business_name)
    s1_addr_nums[row.entity_id] = make_addr_nums(row.business_address)

print("Test Source1 lookup prepared:", len(s1_name))

# Inspect candidate distribution by candidate source
s2_count = 0
s3_count = 0

for chunk in pd.read_csv(
    candidate_path,
    sep="\t",
    chunksize=200000
):
    s2_count += chunk["candidate_entity_id"].str.startswith("S2-").sum()
    s3_count += chunk["candidate_entity_id"].str.startswith("S3-").sum()

print("S2 candidate pairs:", s2_count)
print("S3 candidate pairs:", s3_count)
print("Total:", s2_count + s3_count)

Test Source1 lookup prepared: 1732544
S2 candidate pairs: 7953478
S3 candidate pairs: 1812182
Total: 9765660


In [11]:
import shutil
import psutil

disk = shutil.disk_usage("/")

print("Free disk: %.2f GB" % (disk.free / 1024**3))
print("Total disk: %.2f GB" % (disk.total / 1024**3))

memory = psutil.virtual_memory()

print("Available RAM: %.2f GB" % (memory.available / 1024**3))
print("Total RAM: %.2f GB" % (memory.total / 1024**3))

Free disk: 37.00 GB
Total disk: 37.00 GB
Available RAM: 0.94 GB
Total RAM: 3.74 GB


In [12]:
import boto3
import duckdb

BUCKET = "sagemaker-eu-north-1-152025327817"

# Get the AWS credentials already available to SageMaker
session = boto3.Session(region_name="eu-north-1")
credentials = session.get_credentials().get_frozen_credentials()

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"SET s3_region='eu-north-1';")
con.execute(f"SET s3_access_key_id='{credentials.access_key}';")
con.execute(f"SET s3_secret_access_key='{credentials.secret_key}';")

if credentials.token:
    con.execute(f"SET s3_session_token='{credentials.token}';")

print("Testing S3 access...")

test = con.execute("""
    SELECT entity_id, business_name, country
    FROM read_csv(
        's3://sagemaker-eu-north-1-152025327817/dataset/test/test_source2.tsv',
        delim='\t',
        header=true,
        columns={
            'entity_id':'VARCHAR',
            'business_name':'VARCHAR',
            'business_address':'VARCHAR',
            'country':'VARCHAR'
        },
        sample_size=10000
    )
    LIMIT 3
""").fetchdf()

print(test)
print("\n✅ DuckDB S3 access works.")

Testing S3 access...
      entity_id             business_name country
0  S2-192345572           Brahma Infosoft   India
1  S2-566025912  Marina Ecole France Sarl  France
2  S2-158121477          SCI Ptit Àmicale  France

✅ DuckDB S3 access works.


In [13]:
import duckdb
import os

# Reuse the DuckDB connection from the previous cell
con.execute("SET memory_limit='700MB';")
con.execute("SET threads=2;")
con.execute("SET preserve_insertion_order=false;")

# Give DuckDB plenty of disk space for temporary operations
con.execute("SET temp_directory='/home/sagemaker-user/tmp_duckdb';")

os.makedirs("/home/sagemaker-user/tmp_duckdb", exist_ok=True)

print("DuckDB memory limit: 700 MB")
print("DuckDB threads: 2")
print("Temporary directory ready.")

print("\nTesting candidate file...")
result = con.execute("""
    SELECT COUNT(*) AS candidate_pairs
    FROM read_csv(
        'candidate_index/unique_candidates.tsv',
        delim='\t',
        header=true,
        columns={
            'source1_entity_id':'VARCHAR',
            'candidate_entity_id':'VARCHAR'
        }
    )
""").fetchone()

print("Candidate pairs:", result[0])
print("✅ DuckDB is configured safely.")

DuckDB memory limit: 700 MB
DuckDB threads: 2
Temporary directory ready.

Testing candidate file...
Candidate pairs: 9765660
✅ DuckDB is configured safely.


In [14]:
# STEP 25 — SMALL SIMILARITY TEST

test_result = con.execute("""
SELECT
    levenshtein('crystal lending pc', 'crystal lending pc') AS identical,
    levenshtein('crystal lending pc', 'crysta1 lending pc') AS typo,
    levenshtein('burger solution', 'sri burger solution') AS reordered
""").fetchdf()

print(test_result)

   identical  typo  reordered
0          0     1          4


In [15]:
# STEP 26 — TEST REAL CANDIDATE JOIN
# Only 1,000 rows. Safe diagnostic.

test_pairs = con.execute("""
WITH candidates AS (
    SELECT source1_entity_id, candidate_entity_id
    FROM read_csv(
        'candidate_index/unique_candidates.tsv',
        delim='\t',
        header=true,
        columns={
            'source1_entity_id':'VARCHAR',
            'candidate_entity_id':'VARCHAR'
        }
    )
    LIMIT 1000
),

s1 AS (
    SELECT
        entity_id,
        business_name,
        business_address,
        country
    FROM read_csv(
        's3://sagemaker-eu-north-1-152025327817/dataset/test/test_source1.tsv',
        delim='\t',
        header=true,
        columns={
            'entity_id':'VARCHAR',
            'business_name':'VARCHAR',
            'business_address':'VARCHAR',
            'country':'VARCHAR'
        }
    )
),

s2 AS (
    SELECT
        entity_id,
        business_name,
        business_address,
        country
    FROM read_csv(
        's3://sagemaker-eu-north-1-152025327817/dataset/test/test_source2.tsv',
        delim='\t',
        header=true,
        columns={
            'entity_id':'VARCHAR',
            'business_name':'VARCHAR',
            'business_address':'VARCHAR',
            'country':'VARCHAR'
        }
    )
)

SELECT
    c.source1_entity_id,
    c.candidate_entity_id,
    s1.business_name AS source1_name,
    s2.business_name AS candidate_name,
    s1.country AS source1_country,
    s2.country AS candidate_country
FROM candidates c
JOIN s1
    ON c.source1_entity_id = s1.entity_id
JOIN s2
    ON c.candidate_entity_id = s2.entity_id
LIMIT 5
""").fetchdf()

print(test_pairs)
print("\nRows returned:", len(test_pairs))
print("✅ Candidate → Source1 → Source2 join works.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  source1_entity_id candidate_entity_id                source1_name  \
0      S1-607177411         S2-89663826                 Apex Summit   
1       S1-10991820        S2-884102769                Fresh Truist   
2      S1-713396990        S2-375151053         A Diamond Short LLC   
3      S1-984582949        S2-767997013  Neha Foods Private Limited   
4      S1-199924207        S2-292567038        Kerala India Limited   

               candidate_name source1_country candidate_country  
0                 Apex Summit              US                US  
1                Fresh Truist              US                US  
2        A DIAMOND  SHORT LLC              US                US  
3  neha foods private limited           India             India  
4        KERALA INDIA LIMITED           India             India  

Rows returned: 5
✅ Candidate → Source1 → Source2 join works.


In [16]:
# STEP 27 — TEST NAME + ADDRESS SIMILARITY
# Only 1,000 candidate pairs.

score_test = con.execute("""
WITH candidates AS (
    SELECT source1_entity_id, candidate_entity_id
    FROM read_csv(
        'candidate_index/unique_candidates.tsv',
        delim='\t',
        header=true,
        columns={
            'source1_entity_id':'VARCHAR',
            'candidate_entity_id':'VARCHAR'
        }
    )
    LIMIT 1000
),

s1 AS (
    SELECT
        entity_id,
        business_name,
        business_address,
        country
    FROM read_csv(
        's3://sagemaker-eu-north-1-152025327817/dataset/test/test_source1.tsv',
        delim='\t',
        header=true,
        columns={
            'entity_id':'VARCHAR',
            'business_name':'VARCHAR',
            'business_address':'VARCHAR',
            'country':'VARCHAR'
        }
    )
),

s2 AS (
    SELECT
        entity_id,
        business_name,
        business_address,
        country
    FROM read_csv(
        's3://sagemaker-eu-north-1-152025327817/dataset/test/test_source2.tsv',
        delim='\t',
        header=true,
        columns={
            'entity_id':'VARCHAR',
            'business_name':'VARCHAR',
            'business_address':'VARCHAR',
            'country':'VARCHAR'
        }
    )
)

SELECT
    c.source1_entity_id,
    c.candidate_entity_id,

    levenshtein(
        lower(trim(s1.business_name)),
        lower(trim(s2.business_name))
    ) AS name_distance,

    levenshtein(
        lower(trim(s1.business_address)),
        lower(trim(s2.business_address))
    ) AS address_distance

FROM candidates c
JOIN s1
    ON c.source1_entity_id = s1.entity_id
JOIN s2
    ON c.candidate_entity_id = s2.entity_id

LIMIT 10
""").fetchdf()

print(score_test)
print("\nRows scored:", len(score_test))
print("✅ Similarity scoring works.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  source1_entity_id candidate_entity_id  name_distance  address_distance
0      S1-607177411         S2-89663826              0                31
1       S1-10991820        S2-884102769              0                 0
2      S1-439579149        S2-375151053              1                26
3      S1-984582949        S2-767997013              0                81
4      S1-199924207        S2-292567038              0                25
5      S1-655789370        S2-378846408              1                35
6      S1-714865224        S2-504393659              0                 3
7       S1-45537298        S2-600460285              0                75
8      S1-255955599        S2-560710588              0                23
9      S1-565271347        S2-262646344              2                 2

Rows scored: 10
✅ Similarity scoring works.


In [17]:
# STEP 28 — NORMALIZED LEVENSHTEIN TEST
# Only 1,000 candidate pairs.

score_test2 = con.execute("""
WITH candidates AS (
    SELECT source1_entity_id, candidate_entity_id
    FROM read_csv(
        'candidate_index/unique_candidates.tsv',
        delim='\t',
        header=true,
        columns={
            'source1_entity_id':'VARCHAR',
            'candidate_entity_id':'VARCHAR'
        }
    )
    LIMIT 1000
),

s1 AS (
    SELECT
        entity_id,
        business_name,
        business_address
    FROM read_csv(
        's3://sagemaker-eu-north-1-152025327817/dataset/test/test_source1.tsv',
        delim='\t',
        header=true,
        columns={
            'entity_id':'VARCHAR',
            'business_name':'VARCHAR',
            'business_address':'VARCHAR',
            'country':'VARCHAR'
        }
    )
),

s2 AS (
    SELECT
        entity_id,
        business_name,
        business_address
    FROM read_csv(
        's3://sagemaker-eu-north-1-152025327817/dataset/test/test_source2.tsv',
        delim='\t',
        header=true,
        columns={
            'entity_id':'VARCHAR',
            'business_name':'VARCHAR',
            'business_address':'VARCHAR',
            'country':'VARCHAR'
        }
    )
),

joined AS (
    SELECT
        c.source1_entity_id,
        c.candidate_entity_id,
        lower(trim(s1.business_name)) AS n1,
        lower(trim(s2.business_name)) AS n2,
        lower(trim(s1.business_address)) AS a1,
        lower(trim(s2.business_address)) AS a2
    FROM candidates c
    JOIN s1 ON c.source1_entity_id = s1.entity_id
    JOIN s2 ON c.candidate_entity_id = s2.entity_id
)

SELECT
    source1_entity_id,
    candidate_entity_id,

    ROUND(
        100.0 *
        (1.0 - CAST(levenshtein(n1, n2) AS DOUBLE)
        / GREATEST(length(n1), length(n2), 1)),
        2
    ) AS name_score,

    ROUND(
        100.0 *
        (1.0 - CAST(levenshtein(a1, a2) AS DOUBLE)
        / GREATEST(length(a1), length(a2), 1)),
        2
    ) AS address_score

FROM joined
LIMIT 10
""").fetchdf()

print(score_test2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  source1_entity_id candidate_entity_id  name_score  address_score
0      S1-607177411         S2-89663826      100.00          13.89
1       S1-10991820        S2-884102769      100.00         100.00
2      S1-439579149        S2-375151053       95.00          38.10
3      S1-984582949        S2-767997013      100.00          18.18
4      S1-199924207        S2-292567038      100.00          62.12
5      S1-820574299        S2-378846408      100.00          20.45
6      S1-714865224        S2-504393659      100.00          91.89
7       S1-45537298        S2-600460285      100.00          21.05
8      S1-255955599        S2-560710588      100.00          28.13
9      S1-565271347        S2-262646344       86.67          95.12


In [1]:
# STEP 29 — COUNTRY FILTER DIAGNOSTIC
# Counts candidates where Source1 and candidate have the same country.
# Does NOT save anything.

country_result = con.execute("""
WITH candidates AS (
    SELECT source1_entity_id, candidate_entity_id
    FROM read_csv(
        'candidate_index/unique_candidates.tsv',
        delim='\t',
        header=true,
        columns={
            'source1_entity_id':'VARCHAR',
            'candidate_entity_id':'VARCHAR'
        }
    )
),

s1 AS (
    SELECT
        entity_id,
        upper(trim(country)) AS country
    FROM read_csv(
        's3://sagemaker-eu-north-1-152025327817/dataset/test/test_source1.tsv',
        delim='\t',
        header=true,
        columns={
            'entity_id':'VARCHAR',
            'business_name':'VARCHAR',
            'business_address':'VARCHAR',
            'country':'VARCHAR'
        }
    )
),

s2 AS (
    SELECT
        entity_id,
        upper(trim(country)) AS country
    FROM read_csv(
        's3://sagemaker-eu-north-1-152025327817/dataset/test/test_source2.tsv',
        delim='\t',
        header=true,
        columns={
            'entity_id':'VARCHAR',
            'business_name':'VARCHAR',
            'business_address':'VARCHAR',
            'country':'VARCHAR'
        }
    )
)

SELECT
    COUNT(*) AS total_pairs,
    SUM(CASE WHEN s1.country = s2.country THEN 1 ELSE 0 END)
        AS same_country_pairs,
    SUM(CASE WHEN s1.country <> s2.country THEN 1 ELSE 0 END)
        AS different_country_pairs
FROM candidates c
JOIN s1 ON c.source1_entity_id = s1.entity_id
JOIN s2 ON c.candidate_entity_id = s2.entity_id
""").fetchdf()

print(country_result)

NameError: name 'con' is not defined

In [2]:
import duckdb

con = duckdb.connect("matching_work/matching.duckdb")

con.execute("SET memory_limit='600MB';")
con.execute("SET threads=1;")
con.execute("SET preserve_insertion_order=false;")
con.execute("SET temp_directory='matching_work/duckdb_tmp2';")
con.execute("SET max_temp_directory_size='500MB';")

print("Connected successfully.")

Connected successfully.


In [3]:
print(con.execute("SHOW TABLES").fetchdf())

                name
0         candidates
1  scored_candidates
2            source1
3            source2
4            source3


In [4]:
print("Candidates:", con.execute("""
    SELECT COUNT(*) AS n FROM candidates
""").fetchone()[0])

print("Scored candidates:", con.execute("""
    SELECT COUNT(*) AS n FROM scored_candidates
""").fetchone()[0])

Candidates: 9765660
Scored candidates: 1450000


In [5]:
# STEP 29 — COUNTRY FILTER DIAGNOSTIC
# Uses the existing DuckDB candidates table.
# Does NOT create or modify any data.

s2_country = con.execute("""
SELECT
    COUNT(*) AS total_pairs,
    SUM(
        CASE
            WHEN upper(trim(s1.country)) = upper(trim(s2.country))
            THEN 1 ELSE 0
        END
    ) AS same_country_pairs,
    SUM(
        CASE
            WHEN upper(trim(s1.country)) <> upper(trim(s2.country))
            THEN 1 ELSE 0
        END
    ) AS different_country_pairs
FROM candidates c
JOIN source1 s1
    ON c.source1_entity_id = s1.entity_id
JOIN source2 s2
    ON c.candidate_entity_id = s2.entity_id
WHERE c.candidate_entity_id LIKE 'S2-%'
""").fetchdf()


s3_country = con.execute("""
SELECT
    COUNT(*) AS total_pairs,
    SUM(
        CASE
            WHEN upper(trim(s1.country)) = upper(trim(s3.country))
            THEN 1 ELSE 0
        END
    ) AS same_country_pairs,
    SUM(
        CASE
            WHEN upper(trim(s1.country)) <> upper(trim(s3.country))
            THEN 1 ELSE 0
        END
    ) AS different_country_pairs
FROM candidates c
JOIN source1 s1
    ON c.source1_entity_id = s1.entity_id
JOIN source3 s3
    ON c.candidate_entity_id = s3.entity_id
WHERE c.candidate_entity_id LIKE 'S3-%'
""").fetchdf()


print("SOURCE 2")
print(s2_country)

print("\nSOURCE 3")
print(s3_country)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SOURCE 2
   total_pairs  same_country_pairs  different_country_pairs
0      7953478           7953478.0                      0.0

SOURCE 3
   total_pairs  same_country_pairs  different_country_pairs
0      1812182           1812182.0                      0.0


In [6]:
print(con.execute("""
SELECT
    COUNT(*) AS scored_rows,
    MIN(source1_entity_id) AS first_source1,
    MAX(source1_entity_id) AS last_source1
FROM scored_candidates
""").fetchdf())

   scored_rows first_source1  last_source1
0      1450000  S1-100000556  S1-999998881


In [7]:
print(con.execute("""
SELECT
    COUNT(DISTINCT source1_entity_id) AS source1_entities_scored,
    COUNT(DISTINCT candidate_entity_id) AS candidates_scored
FROM scored_candidates
""").fetchdf())

   source1_entities_scored  candidates_scored
0                   504620             240264


In [8]:
print(con.execute("""
WITH scored_counts AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS scored_count
    FROM scored_candidates
    GROUP BY source1_entity_id
),
candidate_counts AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS candidate_count
    FROM candidates
    GROUP BY source1_entity_id
)
SELECT
    COUNT(*) AS scored_source1_entities,
    SUM(CASE WHEN s.scored_count = c.candidate_count THEN 1 ELSE 0 END)
        AS fully_scored_source1_entities,
    SUM(CASE WHEN s.scored_count < c.candidate_count THEN 1 ELSE 0 END)
        AS partially_scored_source1_entities
FROM scored_counts s
JOIN candidate_counts c
    ON s.source1_entity_id = c.source1_entity_id
""").fetchdf())

   scored_source1_entities  fully_scored_source1_entities  \
0                   504620                       104727.0   

   partially_scored_source1_entities  
0                           399893.0  


In [9]:
remaining = con.execute("""
SELECT COUNT(*)
FROM candidates c
WHERE NOT EXISTS (
    SELECT 1
    FROM scored_candidates s
    WHERE s.source1_entity_id = c.source1_entity_id
      AND s.candidate_entity_id = c.candidate_entity_id
)
""").fetchone()[0]

print("Remaining unscored candidate pairs:", remaining)

Remaining unscored candidate pairs: 8315660


In [10]:
strong_name = con.execute("""
SELECT COUNT(*)
FROM candidates c
JOIN source1 s1
    ON c.source1_entity_id = s1.entity_id
JOIN source2 s2
    ON c.candidate_entity_id = s2.entity_id
WHERE c.candidate_entity_id LIKE 'S2-%'
  AND lower(trim(s1.business_name)) = lower(trim(s2.business_name))
""").fetchone()[0]

strong_name_s3 = con.execute("""
SELECT COUNT(*)
FROM candidates c
JOIN source1 s1
    ON c.source1_entity_id = s1.entity_id
JOIN source3 s3
    ON c.candidate_entity_id = s3.entity_id
WHERE c.candidate_entity_id LIKE 'S3-%'
  AND lower(trim(s1.business_name)) = lower(trim(s3.business_name))
""").fetchone()[0]

print("Exact raw-name S2 candidates:", strong_name)
print("Exact raw-name S3 candidates:", strong_name_s3)
print("Total:", strong_name + strong_name_s3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OutOfMemoryException: Out of Memory Error: failed to offload data block of size 256.0 KiB (476.6 MiB/476.8 MiB used).
This limit was set by the 'max_temp_directory_size' setting.
By default, this setting utilizes the available disk space on the drive where the 'temp_directory' is located.
You can adjust this setting, by using (for example) PRAGMA max_temp_directory_size='10GiB'

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

In [11]:
con.execute("SET max_temp_directory_size='500MB';")
print("Safe temp limit restored.")

Safe temp limit restored.


In [12]:
sample = con.execute("""
SELECT
    c.source1_entity_id,
    c.candidate_entity_id,
    s1.business_name AS source1_name,
    CASE
        WHEN c.candidate_entity_id LIKE 'S2-%' THEN s2.business_name
        ELSE s3.business_name
    END AS candidate_name,
    s1.business_address AS source1_address,
    CASE
        WHEN c.candidate_entity_id LIKE 'S2-%' THEN s2.business_address
        ELSE s3.business_address
    END AS candidate_address
FROM candidates c
JOIN source1 s1
    ON c.source1_entity_id = s1.entity_id
LEFT JOIN source2 s2
    ON c.candidate_entity_id = s2.entity_id
LEFT JOIN source3 s3
    ON c.candidate_entity_id = s3.entity_id
LIMIT 20
""").fetchdf()

sample

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OutOfMemoryException: Out of Memory Error: failed to offload data block of size 256.0 KiB (476.8 MiB/476.8 MiB used).
This limit was set by the 'max_temp_directory_size' setting.
By default, this setting utilizes the available disk space on the drive where the 'temp_directory' is located.
You can adjust this setting, by using (for example) PRAGMA max_temp_directory_size='10GiB'

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

In [13]:
small_candidates = con.execute("""
    SELECT source1_entity_id, candidate_entity_id
    FROM candidates
    LIMIT 20
""").fetchall()

small_candidates

[('S1-607177411', 'S2-89663826'),
 ('S1-10991820', 'S2-884102769'),
 ('S1-439579149', 'S2-375151053'),
 ('S1-984582949', 'S2-767997013'),
 ('S1-199924207', 'S2-292567038'),
 ('S1-655789370', 'S2-378846408'),
 ('S1-714865224', 'S2-504393659'),
 ('S1-45537298', 'S2-600460285'),
 ('S1-255955599', 'S2-560710588'),
 ('S1-565271347', 'S2-262646344'),
 ('S1-378961310', 'S2-742018880'),
 ('S1-910579604', 'S2-878037834'),
 ('S1-17787371', 'S2-933724724'),
 ('S1-198567405', 'S2-658886344'),
 ('S1-63383852', 'S2-416136682'),
 ('S1-189505873', 'S2-744098365'),
 ('S1-580632931', 'S2-400190057'),
 ('S1-237439033', 'S2-463756409'),
 ('S1-438730787', 'S2-305637735'),
 ('S1-323915736', 'S2-495604525')]

In [14]:
s1 = con.execute("""
    SELECT entity_id, business_name, business_address, country
    FROM source1
    WHERE entity_id = 'S1-607177411'
""").fetchone()

s2 = con.execute("""
    SELECT entity_id, business_name, business_address, country
    FROM source2
    WHERE entity_id = 'S2-89663826'
""").fetchone()

print("SOURCE 1:")
print(s1)

print("\nSOURCE 2:")
print(s2)

SOURCE 1:
('S1-607177411', 'Apex Summit', '7616 Glencannon Drive, NC, Charlotte', 'US')

SOURCE 2:
('S2-89663826', 'Apex Summit', '67 KENTUCKY ST, SALYERSVILLE, KY', 'US')


In [15]:
count_one = con.execute("""
    SELECT COUNT(*)
    FROM candidates
    WHERE source1_entity_id = 'S1-607177411'
""").fetchone()[0]

print("Candidates for S1-607177411:", count_one)

Candidates for S1-607177411: 5


In [16]:
ids = con.execute("""
    SELECT candidate_entity_id
    FROM candidates
    WHERE source1_entity_id = 'S1-607177411'
""").fetchall()

print(ids)

[('S2-89663826',), ('S2-733904171',), ('S3-30588175',), ('S3-776176230',), ('S3-738768488',)]


In [17]:
candidate_ids = [
    'S2-89663826',
    'S2-733904171',
    'S3-30588175',
    'S3-776176230',
    'S3-738768488'
]

for cid in candidate_ids:
    if cid.startswith("S2-"):
        row = con.execute("""
            SELECT entity_id, business_name, business_address, country
            FROM source2
            WHERE entity_id = ?
        """, [cid]).fetchone()
    else:
        row = con.execute("""
            SELECT entity_id, business_name, business_address, country
            FROM source3
            WHERE entity_id = ?
        """, [cid]).fetchone()

    print(row)

('S2-89663826', 'Apex Summit', '67 KENTUCKY ST, SALYERSVILLE, KY', 'US')
('S2-733904171', 'APEX SUMMIT', '7616 GLENCANNON DR, CHARLOTTE, NC', 'US')
('S3-30588175', 'Apex Summit', '#67 Kentucky Street, Salyersville, Kentucky', 'US')
('S3-776176230', 'Apex Summit', '7 Willow Branch Pl, Stafford County, Virginia', 'US')
('S3-738768488', 'Apex Summit Company', '41 Forest View Ave, Wood Dale, Illinois', 'US')


In [19]:
import re
import unicodedata
from difflib import SequenceMatcher

def normalize_text(text):
    if text is None:
        return ""
    
    text = str(text).lower()
    
    # Remove accents safely
    text = "".join(
        c for c in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(c)
    )
    
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

def text_similarity(a, b):
    a = normalize_text(a)
    b = normalize_text(b)
    
    if not a or not b:
        return 0.0
    
    return SequenceMatcher(None, a, b).ratio()

s1_name = "Apex Summit"
s1_addr = "7616 Glencannon Drive, NC, Charlotte"

test_rows = [
    ("Apex Summit", "67 KENTUCKY ST, SALYERSVILLE, KY"),
    ("APEX SUMMIT", "7616 GLENCANNON DR, CHARLOTTE, NC"),
    ("Apex Summit", "#67 Kentucky Street, Salyersville, Kentucky"),
    ("Apex Summit", "7 Willow Branch Pl, Stafford County, Virginia"),
    ("Apex Summit Company", "41 Forest View Ave, Wood Dale, Illinois"),
]

for name, addr in test_rows:
    ns = text_similarity(s1_name, name)
    ads = text_similarity(s1_addr, addr)
    combined = 0.4 * ns + 0.6 * ads
    
    print(
        f"name={ns:.3f}, "
        f"address={ads:.3f}, "
        f"combined={combined:.3f}"
    )

name=1.000, address=0.156, combined=0.494
name=1.000, address=0.862, combined=0.917
name=1.000, address=0.162, combined=0.497
name=1.000, address=0.286, combined=0.571
name=0.733, address=0.254, combined=0.445


In [20]:
import re

def address_numbers(text):
    if text is None:
        return set()
    
    text = str(text).lower()
    
    # Extract standalone numbers, including numbers such as 11643-11645
    nums = re.findall(r'\d+', text)
    
    return set(nums)

s1_addr = "7616 Glencannon Drive, NC, Charlotte"

test_addresses = [
    "67 KENTUCKY ST, SALYERSVILLE, KY",
    "7616 GLENCANNON DR, CHARLOTTE, NC",
    "#67 Kentucky Street, Salyersville, Kentucky",
    "7 Willow Branch Pl, Stafford County, Virginia",
    "41 Forest View Ave, Wood Dale, Illinois"
]

s1_nums = address_numbers(s1_addr)

print("Source1 address numbers:", s1_nums)
print()

for addr in test_addresses:
    nums = address_numbers(addr)
    common = s1_nums & nums
    
    print(
        f"{addr}\n"
        f"  candidate numbers: {nums}\n"
        f"  common numbers:    {common}\n"
    )

Source1 address numbers: {'7616'}

67 KENTUCKY ST, SALYERSVILLE, KY
  candidate numbers: {'67'}
  common numbers:    set()

7616 GLENCANNON DR, CHARLOTTE, NC
  candidate numbers: {'7616'}
  common numbers:    {'7616'}

#67 Kentucky Street, Salyersville, Kentucky
  candidate numbers: {'67'}
  common numbers:    set()

7 Willow Branch Pl, Stafford County, Virginia
  candidate numbers: {'7'}
  common numbers:    set()

41 Forest View Ave, Wood Dale, Illinois
  candidate numbers: {'41'}
  common numbers:    set()



In [22]:
import os

print("Current directory:")
print(os.getcwd())

print("\nTop-level folders/files:")
for item in os.listdir("."):
    print(item)

Current directory:
/home/sagemaker-user

Top-level folders/files:
.bashrc
.agent
.kiro
.claude
.sagemaker_sql_editor_api_cache
.local
.ipython
.npm
.jupyter
untitled.chat
Untitled.ipynb
.ipynb_checkpoints
.cache
.config
business_entity_resolution.ipynb
output
.duckdb
tmp_duckdb
matching_work
.virtual_documents


In [23]:
import os

print("TSV files found:")

for root, dirs, files in os.walk("/home/sagemaker-user"):
    # Skip folders that can contain lots of unrelated files
    dirs[:] = [
        d for d in dirs
        if d not in [".cache", ".local", ".npm", ".ipython", ".config"]
    ]

    for file in files:
        if file.endswith(".tsv"):
            print(os.path.join(root, file))

TSV files found:
/home/sagemaker-user/output/candidate_pairs.tsv
/home/sagemaker-user/matching_work/test_source1.tsv
/home/sagemaker-user/matching_work/test_source2.tsv
/home/sagemaker-user/matching_work/test_source3.tsv


In [24]:
import boto3

s3 = boto3.client("s3", region_name="eu-north-1")

bucket = "sagemaker-eu-north-1-152025327817"

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix="dataset/train/"
)

print("Training files found in S3:")

for obj in response.get("Contents", []):
    print(obj["Key"], "->", round(obj["Size"] / (1024**2), 2), "MB")

Training files found in S3:
dataset/train/train_ground_truth.tsv -> 121.13 MB
dataset/train/train_source1.tsv -> 200.34 MB
dataset/train/train_source2.tsv -> 466.63 MB
dataset/train/train_source3.tsv -> 480.37 MB


In [25]:
import boto3

s3 = boto3.client("s3", region_name="eu-north-1")

bucket = "sagemaker-eu-north-1-152025327817"
key = "dataset/train/train_ground_truth.tsv"

obj = s3.get_object(
    Bucket=bucket,
    Key=key,
    Range="bytes=0-5000"
)

text = obj["Body"].read().decode("utf-8", errors="replace")

print(text[:5000])

source1_entity_id	matched_entity_ids
S1-965667	S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364
S1-55344266	S2-249013014,S2-197070651,S3-478195123,S3-384364074
S1-343815751	S2-790675320,S2-479876582,S3-878454467
S1-656753428	S2-153058913,S2-24659151,S3-679606215
S1-102811957	S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785
S1-18727616	S2-755677256,S3-187831601,S3-641489370,S3-476250621
S1-318373630	S2-660036492,S3-804600254
S1-86989137	S3-274817120,S3-312496301
S1-29845983	S2-648035184,S3-588502663
S1-789009573	S2-383871912,S3-74481402,S3-576451439
S1-730934468	S2-356983532,S3-352439310
S1-7293388	S2-7028416,S2-442723188,S2-157073701,S3-523120965
S1-546142636	S2-487600131,S2-582477216,S2-392804085,S3-200008747,S3-729771680,S3-249331830
S1-274126313	S2-736616474,S2-680265918,S2-51486805,S3-925631694,S3-461175723,S3-850112871
S1-145361722	S2-120366543,S2-939389287,S3-96572514
S1-503957000	S2-994658326,S2-235117490,S2-353308450,S2-173295926,S3-8

In [27]:
import boto3

s3 = boto3.client("s3", region_name="eu-north-1")

bucket = "sagemaker-eu-north-1-152025327817"
key = "dataset/train/train_ground_truth.tsv"

obj = s3.get_object(Bucket=bucket, Key=key)

source1_count = 0
matched_pair_count = 0
no_match_count = 0

for line in obj["Body"].iter_lines():
    if not line:
        continue

    line = line.decode("utf-8", errors="replace")

    # Skip header
    if line.startswith("source1_entity_id"):
        continue

    parts = line.split("\t", 1)

    source1_count += 1

    if len(parts) == 2 and parts[1].strip():
        matched_pair_count += len(parts[1].strip().split(","))
    else:
        no_match_count += 1

print("Training Source1 entities:", source1_count)
print("Ground-truth matched pairs:", matched_pair_count)
print("No-match Source1 entities:", no_match_count)

Training Source1 entities: 2206821
Ground-truth matched pairs: 7638365
No-match Source1 entities: 123247


In [28]:
import boto3
import random

s3 = boto3.client("s3", region_name="eu-north-1")

bucket = "sagemaker-eu-north-1-152025327817"
key = "dataset/train/train_ground_truth.tsv"

obj = s3.get_object(Bucket=bucket, Key=key)

random.seed(42)

sample_size = 5000
sample_rows = []

for line in obj["Body"].iter_lines():
    if not line:
        continue

    line = line.decode("utf-8", errors="replace")

    if line.startswith("source1_entity_id"):
        continue

    # Reservoir sampling
    if len(sample_rows) < sample_size:
        sample_rows.append(line)
    else:
        j = random.randint(0, len(sample_rows))
        if j < sample_size:
            sample_rows[j] = line

print("Sampled ground-truth rows:", len(sample_rows))

# Show 5 examples
for row in sample_rows[:5]:
    print(row)

Sampled ground-truth rows: 5000
S1-612360148	S2-574516529,S3-71148721,S3-795586049
S1-188317419	S2-370849996,S2-153909852
S1-294661672	S2-137362489,S2-892744140,S3-788489836,S3-78497901
S1-783255703	S2-630757036,S2-134007085,S2-169705640,S3-882728400,S3-879350076,S3-54537545,S3-878991238
S1-693726164	S2-659810357,S2-596978307,S3-471318686,S3-405331768


In [29]:
import boto3
import csv
import io

s3 = boto3.client("s3", region_name="eu-north-1")

bucket = "sagemaker-eu-north-1-152025327817"

# --------------------------------------------------
# 1. Extract the IDs from our 5,000 sampled cases
# --------------------------------------------------

s1_ids = set()
s2_ids = set()
s3_ids = set()

ground_truth_sample = {}

for row in sample_rows:
    parts = row.rstrip("\n").split("\t")

    s1_id = parts[0]
    matched = parts[1].strip() if len(parts) > 1 else ""

    s1_ids.add(s1_id)

    matched_ids = []
    if matched:
        matched_ids = matched.split(",")

        for mid in matched_ids:
            if mid.startswith("S2-"):
                s2_ids.add(mid)
            elif mid.startswith("S3-"):
                s3_ids.add(mid)

    ground_truth_sample[s1_id] = matched_ids

print("S1 IDs needed:", len(s1_ids))
print("S2 IDs needed:", len(s2_ids))
print("S3 IDs needed:", len(s3_ids))


# --------------------------------------------------
# 2. Function to stream one TSV from S3
#    and keep only requested IDs
# --------------------------------------------------

def retrieve_rows_from_s3(key, wanted_ids):
    rows = {}

    obj = s3.get_object(Bucket=bucket, Key=key)

    for raw_line in obj["Body"].iter_lines():
        if not raw_line:
            continue

        line = raw_line.decode("utf-8", errors="replace")

        if line.startswith("entity_id\t"):
            continue

        parts = line.split("\t")

        if not parts:
            continue

        entity_id = parts[0]

        if entity_id in wanted_ids:
            rows[entity_id] = parts

    return rows


# --------------------------------------------------
# 3. Retrieve only the needed training records
# --------------------------------------------------

print("\nReading training Source 1...")
train_s1 = retrieve_rows_from_s3(
    "dataset/train/train_source1.tsv",
    s1_ids
)
print("Retrieved S1 records:", len(train_s1))

print("\nReading training Source 2...")
train_s2 = retrieve_rows_from_s3(
    "dataset/train/train_source2.tsv",
    s2_ids
)
print("Retrieved S2 records:", len(train_s2))

print("\nReading training Source 3...")
train_s3 = retrieve_rows_from_s3(
    "dataset/train/train_source3.tsv",
    s3_ids
)
print("Retrieved S3 records:", len(train_s3))

S1 IDs needed: 5000
S2 IDs needed: 8328
S3 IDs needed: 8900

Reading training Source 1...
Retrieved S1 records: 5000

Reading training Source 2...
Retrieved S2 records: 8328

Reading training Source 3...
Retrieved S3 records: 8900


In [30]:
import re
from difflib import SequenceMatcher

def normalize_text(text):
    if text is None:
        return ""
    
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def similarity(a, b):
    a = normalize_text(a)
    b = normalize_text(b)
    
    if not a or not b:
        return 0.0
    
    return SequenceMatcher(None, a, b).ratio()


def address_numbers(text):
    if text is None:
        return set()
    return set(re.findall(r"\d+", str(text)))


results = []

for s1_id, matched_ids in ground_truth_sample.items():

    s1 = train_s1.get(s1_id)

    if s1 is None:
        continue

    s1_name = s1[1]
    s1_address = s1[2]

    for mid in matched_ids:

        if mid.startswith("S2-"):
            candidate = train_s2.get(mid)
        else:
            candidate = train_s3.get(mid)

        if candidate is None:
            continue

        candidate_name = candidate[1]
        candidate_address = candidate[2]

        name_score = similarity(s1_name, candidate_name)
        address_score = similarity(s1_address, candidate_address)

        s1_nums = address_numbers(s1_address)
        candidate_nums = address_numbers(candidate_address)

        shared_numbers = len(s1_nums & candidate_nums)

        results.append({
            "source1": s1_id,
            "candidate": mid,
            "name_score": name_score,
            "address_score": address_score,
            "shared_numbers": shared_numbers
        })

print("True match pairs analyzed:", len(results))

True match pairs analyzed: 17228


In [31]:
import pandas as pd

calib = pd.DataFrame(results)

print("Name similarity:")
print(calib["name_score"].describe())

print("\nAddress similarity:")
print(calib["address_score"].describe())

print("\nShared address-number counts:")
print(calib["shared_numbers"].value_counts().sort_index())

print("\nBoth name >= 0.80 AND address >= 0.70:")
print(
    ((calib["name_score"] >= 0.80) &
     (calib["address_score"] >= 0.70)).mean()
)

print("\nBoth name >= 0.70 AND address >= 0.60:")
print(
    ((calib["name_score"] >= 0.70) &
     (calib["address_score"] >= 0.60)).mean()
)

Name similarity:
count    17228.000000
mean         0.788550
std          0.264733
min          0.000000
25%          0.727273
50%          0.876712
75%          0.964286
max          1.000000
Name: name_score, dtype: float64

Address similarity:
count    17228.000000
mean         0.751440
std          0.240390
min          0.000000
25%          0.644068
50%          0.832256
75%          0.918405
max          1.000000
Name: address_score, dtype: float64

Shared address-number counts:
shared_numbers
0    3447
1    9234
2    3070
3     906
4     407
5      98
6      48
7       9
8       7
9       2
Name: count, dtype: int64

Both name >= 0.80 AND address >= 0.70:
0.4665660552588809

Both name >= 0.70 AND address >= 0.60:
0.6210819596006502


In [32]:
from collections import defaultdict
import re

def normalize_name_for_block(text):
    if text is None:
        return ""

    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # Remove common legal suffixes
    suffixes = [
        "private limited", "pvt ltd", "pvt",
        "limited", "ltd", "llc",
        "incorporated", "inc",
        "corporation", "corp",
        "company", "co", "pc"
    ]

    for suffix in suffixes:
        text = re.sub(rf"\b{suffix}\b", " ", text)

    return re.sub(r"\s+", " ", text).strip()


def address_number_key(text):
    if text is None:
        return ""

    nums = sorted(set(re.findall(r"\d+", str(text))))

    return "|".join(nums)


sample_name_keys = defaultdict(set)
sample_addr_keys = defaultdict(set)

for s1_id in s1_ids:
    row = train_s1.get(s1_id)

    if row is None:
        continue

    name_key = normalize_name_for_block(row[1])
    addr_key = address_number_key(row[2])

    if name_key:
        sample_name_keys[name_key].add(s1_id)

    if addr_key:
        sample_addr_keys[addr_key].add(s1_id)

print("Sample Source1 entities:", len(s1_ids))
print("Unique normalized-name keys:", len(sample_name_keys))
print("Unique address-number keys:", len(sample_addr_keys))

Sample Source1 entities: 5000
Unique normalized-name keys: 4915
Unique address-number keys: 3528


In [33]:
from collections import defaultdict

train_sample_candidates = defaultdict(set)

def process_training_source(key, source_label):
    obj = s3.get_object(Bucket=bucket, Key=key)

    rows_seen = 0
    candidates_found = 0

    for raw_line in obj["Body"].iter_lines():
        if not raw_line:
            continue

        line = raw_line.decode("utf-8", errors="replace")

        if line.startswith("entity_id\t"):
            continue

        parts = line.split("\t")

        if len(parts) < 4:
            continue

        entity_id = parts[0]
        name = parts[1]
        address = parts[2]

        rows_seen += 1

        # Block 1: normalized name
        name_key = normalize_name_for_block(name)

        if name_key and name_key in sample_name_keys:
            for s1_id in sample_name_keys[name_key]:
                train_sample_candidates[s1_id].add(entity_id)
                candidates_found += 1

        # Block 2: exact address-number set
        addr_key = address_number_key(address)

        if addr_key and addr_key in sample_addr_keys:
            for s1_id in sample_addr_keys[addr_key]:
                train_sample_candidates[s1_id].add(entity_id)
                candidates_found += 1

    print(
        source_label,
        "rows scanned:", rows_seen,
        "candidate links found:", candidates_found
    )


print("Scanning training Source2...")
process_training_source(
    "dataset/train/train_source2.tsv",
    "Source2"
)

print("\nScanning training Source3...")
process_training_source(
    "dataset/train/train_source3.tsv",
    "Source3"
)

total_candidates = sum(
    len(v) for v in train_sample_candidates.values()
)

print("\nTotal unique candidate pairs:", total_candidates)
print("Source1 entities with candidates:", len(train_sample_candidates))

Scanning training Source2...
Source2 rows scanned: 5034616 candidate links found: 9432191

Scanning training Source3...
Source3 rows scanned: 5285603 candidate links found: 10465157

Total unique candidate pairs: 19893383
Source1 entities with candidates: 4935


In [34]:
total_true_pairs = 0
covered_true_pairs = 0

entities_with_true = 0
entities_with_all_true_covered = 0

for s1_id, true_ids in ground_truth_sample.items():

    true_set = set(true_ids)

    if not true_set:
        continue

    entities_with_true += 1

    candidate_set = train_sample_candidates.get(s1_id, set())

    covered = true_set & candidate_set

    total_true_pairs += len(true_set)
    covered_true_pairs += len(covered)

    if covered == true_set:
        entities_with_all_true_covered += 1

pair_recall = (
    covered_true_pairs / total_true_pairs
    if total_true_pairs else 0
)

entity_recall = (
    entities_with_all_true_covered / entities_with_true
    if entities_with_true else 0
)

print("Source1 entities with at least one true match:", entities_with_true)
print("Total true match pairs:", total_true_pairs)
print("True match pairs recovered:", covered_true_pairs)
print("Pair recall:", pair_recall)
print("Entities with ALL true matches recovered:", entities_with_all_true_covered)
print("Entity-level full recall:", entity_recall)

Source1 entities with at least one true match: 4738
Total true match pairs: 17228
True match pairs recovered: 12974
Pair recall: 0.7530763872765266
Entities with ALL true matches recovered: 2290
Entity-level full recall: 0.48332629801604055


In [35]:
missed_true_pairs = []

for s1_id, true_ids in ground_truth_sample.items():
    candidate_set = train_sample_candidates.get(s1_id, set())

    for true_id in true_ids:
        if true_id not in candidate_set:
            missed_true_pairs.append((s1_id, true_id))

print("Missed true pairs:", len(missed_true_pairs))

print("\nFirst 20 missed pairs:")
for pair in missed_true_pairs[:20]:
    print(pair)

Missed true pairs: 4254

First 20 missed pairs:
('S1-612360148', 'S3-71148721')
('S1-612360148', 'S3-795586049')
('S1-294661672', 'S2-137362489')
('S1-294661672', 'S3-788489836')
('S1-294661672', 'S3-78497901')
('S1-783255703', 'S2-630757036')
('S1-783255703', 'S2-134007085')
('S1-783255703', 'S2-169705640')
('S1-839831680', 'S2-153530989')
('S1-139577899', 'S2-361283814')
('S1-139577899', 'S3-848927058')
('S1-139577899', 'S3-183829325')
('S1-139577899', 'S3-436777680')
('S1-139577899', 'S3-563309341')
('S1-240860970', 'S3-48262655')
('S1-240860970', 'S3-960938451')
('S1-240860970', 'S3-581714188')
('S1-47816232', 'S2-666953303')
('S1-358467099', 'S3-768622745')
('S1-677740632', 'S2-427317900')


In [36]:
print("First 10 missed true matches with their records:\n")

for s1_id, true_id in missed_true_pairs[:10]:

    s1 = train_s1.get(s1_id)

    if true_id.startswith("S2-"):
        match = train_s2.get(true_id)
    else:
        match = train_s3.get(true_id)

    print("=" * 100)

    print("SOURCE 1")
    print("ID      :", s1[0])
    print("Name    :", s1[1])
    print("Address :", s1[2])
    print("Country :", s1[3])

    print("\nTRUE MATCH")
    print("ID      :", match[0])
    print("Name    :", match[1])
    print("Address :", match[2])
    print("Country :", match[3])

    print("\nName key:")
    print("S1      :", normalize_name_for_block(s1[1]))
    print("Match   :", normalize_name_for_block(match[1]))

    print("\nAddress numbers:")
    print("S1      :", address_number_key(s1[2]))
    print("Match   :", address_number_key(match[2]))

First 10 missed true matches with their records:

SOURCE 1
ID      : S1-612360148
Name    : K&a Medical Centre Limited
Address : New Bus Stand Patoda Tq Patoda Dist Beed, Beed, Bid, Maharashtra
Country : India

TRUE MATCH
ID      : S3-71148721
Name    : K&a Medical Centre Límited
Address : Hn 376 New Bus Stand Patoda Tq Patoda Dist Beed, Bid, Beed, महाराष्ट्र
Country : India

Name key:
S1      : k a medical centre
Match   : k a medical centre l mited

Address numbers:
S1      : 
Match   : 376
SOURCE 1
ID      : S1-612360148
Name    : K&a Medical Centre Limited
Address : New Bus Stand Patoda Tq Patoda Dist Beed, Beed, Bid, Maharashtra
Country : India

TRUE MATCH
ID      : S3-795586049
Name    : K&A K&A Medical-Centre
Address : Beed, Hn 376 New Bus Stand Patoda Tq Patoda Dist Beed, Bid, महाराष्ट्र
Country : India

Name key:
S1      : k a medical centre
Match   : k a k a medical centre

Address numbers:
S1      : 
Match   : 376
SOURCE 1
ID      : S1-294661672
Name    : Eye Medicine Inc
Ad

In [37]:
import re
from difflib import SequenceMatcher

def clean_tokens(text):
    text = normalize_text(text)
    return set(text.split()) if text else set()

def char_prefixes(text, n=4):
    text = normalize_text(text).replace(" ", "")
    if len(text) < n:
        return {text} if text else set()
    return {text[i:i+n] for i in range(len(text) - n + 1)}

def token_overlap(a, b):
    ta = clean_tokens(a)
    tb = clean_tokens(b)

    if not ta or not tb:
        return 0.0

    return len(ta & tb) / min(len(ta), len(tb))


stats = {
    "name_char4": 0,
    "name_token_overlap": 0,
    "address_token_overlap": 0,
    "address_char4": 0,
    "name_or_address": 0
}

total = len(missed_true_pairs)

for s1_id, true_id in missed_true_pairs:

    s1 = train_s1[s1_id]

    if true_id.startswith("S2-"):
        match = train_s2[true_id]
    else:
        match = train_s3[true_id]

    s1_name = s1[1]
    m_name = match[1]

    s1_addr = s1[2]
    m_addr = match[2]

    # Name character n-gram overlap
    name_chars_1 = char_prefixes(s1_name)
    name_chars_2 = char_prefixes(m_name)

    if name_chars_1 & name_chars_2:
        stats["name_char4"] += 1

    # Name token overlap
    if token_overlap(s1_name, m_name) >= 0.5:
        stats["name_token_overlap"] += 1

    # Address token overlap
    if token_overlap(s1_addr, m_addr) >= 0.5:
        stats["address_token_overlap"] += 1

    # Address character overlap
    addr_chars_1 = char_prefixes(s1_addr)
    addr_chars_2 = char_prefixes(m_addr)

    if addr_chars_1 & addr_chars_2:
        stats["address_char4"] += 1

    # Either name or address signal
    if (
        name_chars_1 & name_chars_2
        or token_overlap(s1_addr, m_addr) >= 0.5
    ):
        stats["name_or_address"] += 1


print("Missed true pairs:", total)

for key, value in stats.items():
    print(
        f"{key:25s}: {value:5d} "
        f"({value / total:.1%})"
    )

Missed true pairs: 4254
name_char4               :  3578 (84.1%)
name_token_overlap       :  3188 (74.9%)
address_token_overlap    :  3560 (83.7%)
address_char4            :  3816 (89.7%)
name_or_address          :  4222 (99.2%)


In [38]:
from collections import Counter

def char_ngrams(text, n=4):
    text = normalize_text(text).replace(" ", "")
    
    if len(text) < n:
        return {text} if text else set()
    
    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }


# Count how often each 4-character fragment occurs
# among the 17,228 genuine match pairs.
name_gram_counts = Counter()
address_gram_counts = Counter()

for s1_id, true_ids in ground_truth_sample.items():

    s1 = train_s1[s1_id]

    for true_id in true_ids:

        if true_id.startswith("S2-"):
            match = train_s2.get(true_id)
        else:
            match = train_s3.get(true_id)

        if match is None:
            continue

        name_gram_counts.update(char_ngrams(s1[1]))
        name_gram_counts.update(char_ngrams(match[1]))

        address_gram_counts.update(char_ngrams(s1[2]))
        address_gram_counts.update(char_ngrams(match[2]))


# Test how many missed true pairs contain
# at least one sufficiently rare character fragment.
for limit in [5, 10, 25, 50, 100]:

    recovered_name = 0
    recovered_address = 0
    recovered_either = 0

    for s1_id, true_id in missed_true_pairs:

        s1 = train_s1[s1_id]

        if true_id.startswith("S2-"):
            match = train_s2[true_id]
        else:
            match = train_s3[true_id]

        s1_name = char_ngrams(s1[1])
        m_name = char_ngrams(match[1])

        s1_addr = char_ngrams(s1[2])
        m_addr = char_ngrams(match[2])

        name_shared = (
            s1_name & m_name
        )

        addr_shared = (
            s1_addr & m_addr
        )

        rare_name = any(
            name_gram_counts[g] <= limit
            for g in name_shared
        )

        rare_address = any(
            address_gram_counts[g] <= limit
            for g in addr_shared
        )

        if rare_name:
            recovered_name += 1

        if rare_address:
            recovered_address += 1

        if rare_name or rare_address:
            recovered_either += 1

    print(
        f"Frequency <= {limit:3d} | "
        f"name: {recovered_name:4d} ({recovered_name/len(missed_true_pairs):.1%}) | "
        f"address: {recovered_address:4d} ({recovered_address/len(missed_true_pairs):.1%}) | "
        f"either: {recovered_either:4d} ({recovered_either/len(missed_true_pairs):.1%})"
    )

Frequency <=   5 | name:  472 (11.1%) | address:  638 (15.0%) | either:  859 (20.2%)
Frequency <=  10 | name: 1958 (46.0%) | address: 2605 (61.2%) | either: 3203 (75.3%)
Frequency <=  25 | name: 2953 (69.4%) | address: 3461 (81.4%) | either: 4027 (94.7%)
Frequency <=  50 | name: 3252 (76.4%) | address: 3594 (84.5%) | either: 4141 (97.3%)
Frequency <= 100 | name: 3411 (80.2%) | address: 3668 (86.2%) | either: 4184 (98.4%)


In [39]:
def rare_shared_gram(a, b, counts, limit=25):
    grams_a = char_ngrams(a)
    grams_b = char_ngrams(b)

    shared = grams_a & grams_b

    return any(
        counts[g] <= limit
        for g in shared
    )


recovered = 0
missed = 0

for s1_id, true_id in missed_true_pairs:

    s1 = train_s1[s1_id]

    if true_id.startswith("S2-"):
        match = train_s2[true_id]
        name_counts = name_gram_counts
        addr_counts = address_gram_counts
    else:
        match = train_s3[true_id]
        name_counts = name_gram_counts
        addr_counts = address_gram_counts

    name_hit = rare_shared_gram(
        s1[1],
        match[1],
        name_counts,
        limit=25
    )

    address_hit = rare_shared_gram(
        s1[2],
        match[2],
        addr_counts,
        limit=25
    )

    if name_hit or address_hit:
        recovered += 1
    else:
        missed += 1

print("Missed true pairs:", len(missed_true_pairs))
print("Recovered by rare 4-gram <=25:", recovered)
print("Still missed:", missed)
print("Recovery:", recovered / len(missed_true_pairs))

Missed true pairs: 4254
Recovered by rare 4-gram <=25: 4027
Still missed: 227
Recovery: 0.9466384579219558


In [40]:
# Test rare 4-gram overlap on the candidates we already have.
# This does NOT scan S3 and does NOT modify DuckDB.

rare25_candidate_count = 0
current_candidate_count = 0

for s1_id, candidate_ids in train_sample_candidates.items():

    s1 = train_s1.get(s1_id)

    if s1 is None:
        continue

    for candidate_id in candidate_ids:

        current_candidate_count += 1

        if candidate_id.startswith("S2-"):
            candidate = train_s2.get(candidate_id)
        else:
            candidate = train_s3.get(candidate_id)

        if candidate is None:
            continue

        name_hit = rare_shared_gram(
            s1[1],
            candidate[1],
            name_gram_counts,
            limit=25
        )

        address_hit = rare_shared_gram(
            s1[2],
            candidate[2],
            address_gram_counts,
            limit=25
        )

        if name_hit or address_hit:
            rare25_candidate_count += 1

print("Existing candidate pairs:", current_candidate_count)
print("Pairs having rare-4gram signal:", rare25_candidate_count)

if current_candidate_count:
    print(
        "Percentage:",
        rare25_candidate_count / current_candidate_count
    )

Existing candidate pairs: 19893383
Pairs having rare-4gram signal: 14215
Percentage: 0.0007145592079537201


In [41]:
# Compare 5-character grams against the 4-character result
# using only the already-loaded true matches.

def char_ngrams_n(text, n=5):
    text = normalize_text(text).replace(" ", "")

    if len(text) < n:
        return {text} if text else set()

    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }


name_5gram_counts = Counter()
address_5gram_counts = Counter()

# Count 5-grams in the sampled genuine pairs
for s1_id, true_ids in ground_truth_sample.items():

    s1 = train_s1[s1_id]

    for true_id in true_ids:

        if true_id.startswith("S2-"):
            match = train_s2.get(true_id)
        else:
            match = train_s3.get(true_id)

        if match is None:
            continue

        name_5gram_counts.update(char_ngrams_n(s1[1], 5))
        name_5gram_counts.update(char_ngrams_n(match[1], 5))

        address_5gram_counts.update(char_ngrams_n(s1[2], 5))
        address_5gram_counts.update(char_ngrams_n(match[2], 5))


def rare_5gram_shared(a, b, counts, limit=25):
    shared = (
        char_ngrams_n(a, 5)
        &
        char_ngrams_n(b, 5)
    )

    return any(
        counts[g] <= limit
        for g in shared
    )


recovered = 0

for s1_id, true_id in missed_true_pairs:

    s1 = train_s1[s1_id]

    if true_id.startswith("S2-"):
        match = train_s2[true_id]
    else:
        match = train_s3[true_id]

    name_hit = rare_5gram_shared(
        s1[1],
        match[1],
        name_5gram_counts,
        25
    )

    address_hit = rare_5gram_shared(
        s1[2],
        match[2],
        address_5gram_counts,
        25
    )

    if name_hit or address_hit:
        recovered += 1


print("Missed true pairs:", len(missed_true_pairs))
print("Recovered with rare 5-gram <=25:", recovered)
print(
    "Recovery:",
    recovered / len(missed_true_pairs)
)

Missed true pairs: 4254
Recovered with rare 5-gram <=25: 4079
Recovery: 0.9588622472966619


In [42]:
# Measure how selective the rare 5-gram signal is
# on the existing training candidate universe.

rare5_candidate_count = 0
current_candidate_count = 0

for s1_id, candidate_ids in train_sample_candidates.items():

    s1 = train_s1.get(s1_id)

    if s1 is None:
        continue

    for candidate_id in candidate_ids:

        current_candidate_count += 1

        if candidate_id.startswith("S2-"):
            candidate = train_s2.get(candidate_id)
        else:
            candidate = train_s3.get(candidate_id)

        if candidate is None:
            continue

        name_hit = rare_5gram_shared(
            s1[1],
            candidate[1],
            name_5gram_counts,
            limit=25
        )

        address_hit = rare_5gram_shared(
            s1[2],
            candidate[2],
            address_5gram_counts,
            limit=25
        )

        if name_hit or address_hit:
            rare5_candidate_count += 1


print("Existing candidate pairs:", current_candidate_count)
print("Pairs with rare 5-gram signal:", rare5_candidate_count)
print(
    "Percentage:",
    rare5_candidate_count / current_candidate_count
)

Existing candidate pairs: 19893383
Pairs with rare 5-gram signal: 13435
Percentage: 0.0006753501905633647


In [43]:
print("scored_candidates rows:")
print(
    con.execute("""
        SELECT COUNT(*)
        FROM scored_candidates
    """).fetchone()[0]
)

print("\nDistinct Source1 entities:")
print(
    con.execute("""
        SELECT COUNT(DISTINCT source1_entity_id)
        FROM scored_candidates
    """).fetchone()[0]
)

print("\nFirst 10 scored rows:")
print(
    con.execute("""
        SELECT *
        FROM scored_candidates
        LIMIT 10
    """).fetchdf()
)

scored_candidates rows:
1450000

Distinct Source1 entities:
504620

First 10 scored rows:
  source1_entity_id candidate_entity_id source1_country candidate_country  \
0      S1-903247474        S2-344956759           India             India   
1      S1-320135670        S2-218702270              US                US   
2      S1-456004091        S2-862761062              US                US   
3      S1-414638369        S2-602985825          France            France   
4      S1-297220011        S2-587526757          France            France   
5      S1-320329170        S2-227964434              US                US   
6        S1-8453945        S2-951034038              US                US   
7      S1-363061415        S2-137468128           India             India   
8      S1-355425446        S2-426887952              US                US   
9      S1-903012706        S2-693368483              US                US   

   name_similarity  address_similarity  
0         1.000000   

In [44]:
print("Candidates table structure:")
print(
    con.execute("""
        DESCRIBE candidates
    """).fetchdf()
)

print("\nScored table structure:")
print(
    con.execute("""
        DESCRIBE scored_candidates
    """).fetchdf()
)

Candidates table structure:
           column_name column_type null   key default extra
0    source1_entity_id     VARCHAR  YES  None    None  None
1  candidate_entity_id     VARCHAR  YES  None    None  None

Scored table structure:
           column_name column_type null   key default extra
0    source1_entity_id     VARCHAR  YES  None    None  None
1  candidate_entity_id     VARCHAR  YES  None    None  None
2      source1_country     VARCHAR  YES  None    None  None
3    candidate_country     VARCHAR  YES  None    None  None
4      name_similarity      DOUBLE  YES  None    None  None
5   address_similarity      DOUBLE  YES  None    None  None


In [45]:
print(
    con.execute("""
        SELECT
            table_name,
            index_name,
            sql
        FROM duckdb_indexes()
        WHERE table_name IN ('candidates', 'source1', 'source2', 'source3')
        ORDER BY table_name, index_name
    """).fetchdf()
)

   table_name                index_name  \
0  candidates  candidates_candidate_idx   
1  candidates         candidates_s1_idx   
2     source1     source1_entity_id_idx   
3     source2     source2_entity_id_idx   
4     source3     source3_entity_id_idx   

                                                 sql  
0  CREATE INDEX candidates_candidate_idx ON candi...  
1  CREATE INDEX candidates_s1_idx ON candidates(s...  
2  CREATE INDEX source1_entity_id_idx ON source1(...  
3  CREATE INDEX source2_entity_id_idx ON source2(...  
4  CREATE INDEX source3_entity_id_idx ON source3(...  


In [46]:
import pandas as pd
from difflib import SequenceMatcher

BATCH_SIZE = 10_000

batch = con.execute(f"""
    SELECT source1_entity_id, candidate_entity_id
    FROM candidates
    LIMIT {BATCH_SIZE}
""").fetchdf()

print("Batch rows:", len(batch))

# Get unique IDs
s1_ids_batch = batch["source1_entity_id"].tolist()
candidate_ids_batch = batch["candidate_entity_id"].tolist()

# Fetch Source1 records using indexed entity_id lookup
s1_df = con.execute("""
    SELECT entity_id, business_name, business_address, country
    FROM source1
    WHERE entity_id IN (SELECT UNNEST(?))
""", [s1_ids_batch]).fetchdf()

# Separate S2 and S3 IDs
s2_ids = [x for x in candidate_ids_batch if x.startswith("S2-")]
s3_ids = [x for x in candidate_ids_batch if x.startswith("S3-")]

s2_df = con.execute("""
    SELECT entity_id, business_name, business_address, country
    FROM source2
    WHERE entity_id IN (SELECT UNNEST(?))
""", [s2_ids]).fetchdf() if s2_ids else pd.DataFrame()

s3_df = con.execute("""
    SELECT entity_id, business_name, business_address, country
    FROM source3
    WHERE entity_id IN (SELECT UNNEST(?))
""", [s3_ids]).fetchdf() if s3_ids else pd.DataFrame()

# Convert to dictionaries for fast lookup
s1_lookup = s1_df.set_index("entity_id").to_dict("index")
s2_lookup = s2_df.set_index("entity_id").to_dict("index")
s3_lookup = s3_df.set_index("entity_id").to_dict("index")


def similarity_safe(a, b):
    a = normalize_text(a)
    b = normalize_text(b)

    if not a or not b:
        return 0.0

    return SequenceMatcher(None, a, b).ratio()


scored_batch = []

for _, row in batch.iterrows():

    s1_id = row["source1_entity_id"]
    candidate_id = row["candidate_entity_id"]

    s1 = s1_lookup.get(s1_id)

    if candidate_id.startswith("S2-"):
        candidate = s2_lookup.get(candidate_id)
    else:
        candidate = s3_lookup.get(candidate_id)

    if s1 is None or candidate is None:
        continue

    scored_batch.append({
        "source1_entity_id": s1_id,
        "candidate_entity_id": candidate_id,
        "source1_country": s1["country"],
        "candidate_country": candidate["country"],
        "name_similarity": similarity_safe(
            s1["business_name"],
            candidate["business_name"]
        ),
        "address_similarity": similarity_safe(
            s1["business_address"],
            candidate["business_address"]
        )
    })


scored_batch_df = pd.DataFrame(scored_batch)

print("Successfully scored:", len(scored_batch_df))
print("\nPreview:")
print(scored_batch_df.head())

Batch rows: 10000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

KeyError: "None of ['entity_id'] are in the columns"

In [47]:
import pandas as pd
from difflib import SequenceMatcher

BATCH_SIZE = 10_000

batch = con.execute(f"""
    SELECT source1_entity_id, candidate_entity_id
    FROM candidates
    LIMIT {BATCH_SIZE}
""").fetchdf()

print("Batch rows:", len(batch))

s1_ids_batch = batch["source1_entity_id"].tolist()
candidate_ids_batch = batch["candidate_entity_id"].tolist()

# Source 1
s1_df = con.execute("""
    SELECT entity_id, business_name, business_address, country
    FROM source1
    WHERE entity_id IN (SELECT UNNEST(?))
""", [s1_ids_batch]).fetchdf()

# Candidate IDs
s2_ids = [x for x in candidate_ids_batch if x.startswith("S2-")]
s3_ids = [x for x in candidate_ids_batch if x.startswith("S3-")]

# Source 2
if s2_ids:
    s2_df = con.execute("""
        SELECT entity_id, business_name, business_address, country
        FROM source2
        WHERE entity_id IN (SELECT UNNEST(?))
    """, [s2_ids]).fetchdf()
else:
    s2_df = pd.DataFrame(
        columns=["entity_id", "business_name", "business_address", "country"]
    )

# Source 3
if s3_ids:
    s3_df = con.execute("""
        SELECT entity_id, business_name, business_address, country
        FROM source3
        WHERE entity_id IN (SELECT UNNEST(?))
    """, [s3_ids]).fetchdf()
else:
    s3_df = pd.DataFrame(
        columns=["entity_id", "business_name", "business_address", "country"]
    )

print("Source1 records:", len(s1_df))
print("Source2 records:", len(s2_df))
print("Source3 records:", len(s3_df))

# Lookups
s1_lookup = s1_df.set_index("entity_id").to_dict("index")
s2_lookup = s2_df.set_index("entity_id").to_dict("index")
s3_lookup = s3_df.set_index("entity_id").to_dict("index")


def similarity_safe(a, b):
    a = normalize_text(a)
    b = normalize_text(b)

    if not a or not b:
        return 0.0

    return SequenceMatcher(None, a, b).ratio()


scored_batch = []

for _, row in batch.iterrows():

    s1_id = row["source1_entity_id"]
    candidate_id = row["candidate_entity_id"]

    s1 = s1_lookup.get(s1_id)

    if candidate_id.startswith("S2-"):
        candidate = s2_lookup.get(candidate_id)
    else:
        candidate = s3_lookup.get(candidate_id)

    if s1 is None or candidate is None:
        continue

    scored_batch.append({
        "source1_entity_id": s1_id,
        "candidate_entity_id": candidate_id,
        "source1_country": s1["country"],
        "candidate_country": candidate["country"],
        "name_similarity": similarity_safe(
            s1["business_name"],
            candidate["business_name"]
        ),
        "address_similarity": similarity_safe(
            s1["business_address"],
            candidate["business_address"]
        )
    })

scored_batch_df = pd.DataFrame(scored_batch)

print("\nSuccessfully scored:", len(scored_batch_df))

print("\nPreview:")
print(scored_batch_df.head(10))

Batch rows: 10000
Source1 records: 9835
Source2 records: 1616
Source3 records: 0

Successfully scored: 10000

Preview:
  source1_entity_id candidate_entity_id source1_country candidate_country  \
0      S1-607177411         S2-89663826              US                US   
1       S1-10991820        S2-884102769              US                US   
2      S1-439579149        S2-375151053              US                US   
3      S1-984582949        S2-767997013           India             India   
4      S1-199924207        S2-292567038           India             India   
5      S1-655789370        S2-378846408              US                US   
6      S1-714865224        S2-504393659              US                US   
7       S1-45537298        S2-600460285           India             India   
8      S1-255955599        S2-560710588              US                US   
9      S1-565271347        S2-262646344          France            France   

   name_similarity  address_simil

In [48]:
# Check how many of the first 10,000 candidate pairs
# are already present in scored_candidates.

import pandas as pd

batch_check = con.execute("""
    SELECT source1_entity_id, candidate_entity_id
    FROM candidates
    LIMIT 10000
""").fetchdf()

print("Candidate rows checked:", len(batch_check))

s1_ids_check = batch_check["source1_entity_id"].tolist()

already_scored = con.execute("""
    SELECT source1_entity_id, candidate_entity_id
    FROM scored_candidates
    WHERE source1_entity_id IN (SELECT UNNEST(?))
""", [s1_ids_check]).fetchdf()

print("Saved scored rows found for these Source1 IDs:", len(already_scored))

scored_pairs = set(
    zip(
        already_scored["source1_entity_id"],
        already_scored["candidate_entity_id"]
    )
)

batch_pairs = list(
    zip(
        batch_check["source1_entity_id"],
        batch_check["candidate_entity_id"]
    )
)

overlap = sum(pair in scored_pairs for pair in batch_pairs)

print("First 10,000 candidate pairs already scored:", overlap)
print("First 10,000 candidate pairs still unscored:", 10000 - overlap)

Candidate rows checked: 10000
Saved scored rows found for these Source1 IDs: 91869
First 10,000 candidate pairs already scored: 10000
First 10,000 candidate pairs still unscored: 0


In [49]:
# Test the safe "skip already scored" logic.
# Nothing is inserted into the database.

import pandas as pd

TEST_OFFSET = 10_000
TEST_LIMIT = 10_000

batch_check = con.execute(f"""
    SELECT source1_entity_id, candidate_entity_id
    FROM candidates
    LIMIT {TEST_LIMIT} OFFSET {TEST_OFFSET}
""").fetchdf()

print("Candidate rows checked:", len(batch_check))

s1_ids_check = batch_check["source1_entity_id"].unique().tolist()

already_scored = con.execute("""
    SELECT source1_entity_id, candidate_entity_id
    FROM scored_candidates
    WHERE source1_entity_id IN (SELECT UNNEST(?))
""", [s1_ids_check]).fetchdf()

scored_pairs = set(
    zip(
        already_scored["source1_entity_id"],
        already_scored["candidate_entity_id"]
    )
)

batch_pairs = list(
    zip(
        batch_check["source1_entity_id"],
        batch_check["candidate_entity_id"]
    )
)

unscored_batch = [
    pair for pair in batch_pairs
    if pair not in scored_pairs
]

print("Saved scored rows found:", len(already_scored))
print("Already scored in this batch:", len(batch_pairs) - len(unscored_batch))
print("Still needing scoring:", len(unscored_batch))

Candidate rows checked: 10000
Saved scored rows found: 88326
Already scored in this batch: 10000
Still needing scoring: 0


In [50]:
# Find how many of the first candidate rows are already scored.
# This does NOT modify the database.

import pandas as pd

CHECK_SIZE = 100_000

batch_check = con.execute(f"""
    SELECT source1_entity_id, candidate_entity_id
    FROM candidates
    LIMIT {CHECK_SIZE}
""").fetchdf()

print("Candidate rows checked:", len(batch_check))

s1_ids_check = batch_check["source1_entity_id"].unique().tolist()

already_scored = con.execute("""
    SELECT source1_entity_id, candidate_entity_id
    FROM scored_candidates
    WHERE source1_entity_id IN (SELECT UNNEST(?))
""", [s1_ids_check]).fetchdf()

scored_pairs = set(
    zip(
        already_scored["source1_entity_id"],
        already_scored["candidate_entity_id"]
    )
)

batch_pairs = list(
    zip(
        batch_check["source1_entity_id"],
        batch_check["candidate_entity_id"]
    )
)

already_count = sum(pair in scored_pairs for pair in batch_pairs)

print("Already scored among first 100,000:", already_count)
print("Still unscored among first 100,000:", CHECK_SIZE - already_count)

Candidate rows checked: 100000
Already scored among first 100,000: 100000
Still unscored among first 100,000: 0


In [51]:
# Check the area immediately after the 1,450,000 saved rows.
# This does NOT modify the database.

TEST_OFFSET = 1_450_000
TEST_LIMIT = 10_000

batch_check = con.execute(f"""
    SELECT source1_entity_id, candidate_entity_id
    FROM candidates
    LIMIT {TEST_LIMIT} OFFSET {TEST_OFFSET}
""").fetchdf()

print("Candidate rows checked:", len(batch_check))

s1_ids_check = batch_check["source1_entity_id"].unique().tolist()

already_scored = con.execute("""
    SELECT source1_entity_id, candidate_entity_id
    FROM scored_candidates
    WHERE source1_entity_id IN (SELECT UNNEST(?))
""", [s1_ids_check]).fetchdf()

scored_pairs = set(
    zip(
        already_scored["source1_entity_id"],
        already_scored["candidate_entity_id"]
    )
)

batch_pairs = list(
    zip(
        batch_check["source1_entity_id"],
        batch_check["candidate_entity_id"]
    )
)

already_count = sum(pair in scored_pairs for pair in batch_pairs)

print("Saved scored rows found:", len(already_scored))
print("Already scored in this batch:", already_count)
print("Still needing scoring:", TEST_LIMIT - already_count)

Candidate rows checked: 10000
Saved scored rows found: 77195
Already scored in this batch: 0
Still needing scoring: 10000


In [52]:
import pandas as pd
from difflib import SequenceMatcher

BATCH_SIZE = 10_000
OFFSET = 1_450_000

# --------------------------------------------------
# 1. Get the next unscored candidate batch
# --------------------------------------------------

batch = con.execute(f"""
    SELECT source1_entity_id, candidate_entity_id
    FROM candidates
    LIMIT {BATCH_SIZE} OFFSET {OFFSET}
""").fetchdf()

print("Candidate rows:", len(batch))

# --------------------------------------------------
# 2. Get Source1 records
# --------------------------------------------------

s1_ids_batch = batch["source1_entity_id"].unique().tolist()

s1_df = con.execute("""
    SELECT entity_id, business_name, business_address, country
    FROM source1
    WHERE entity_id IN (SELECT UNNEST(?))
""", [s1_ids_batch]).fetchdf()

# --------------------------------------------------
# 3. Separate S2 and S3 IDs
# --------------------------------------------------

candidate_ids_batch = batch["candidate_entity_id"].tolist()

s2_ids = [x for x in candidate_ids_batch if x.startswith("S2-")]
s3_ids = [x for x in candidate_ids_batch if x.startswith("S3-")]

# --------------------------------------------------
# 4. Get candidate records
# --------------------------------------------------

if s2_ids:
    s2_df = con.execute("""
        SELECT entity_id, business_name, business_address, country
        FROM source2
        WHERE entity_id IN (SELECT UNNEST(?))
    """, [s2_ids]).fetchdf()
else:
    s2_df = pd.DataFrame(
        columns=["entity_id", "business_name", "business_address", "country"]
    )

if s3_ids:
    s3_df = con.execute("""
        SELECT entity_id, business_name, business_address, country
        FROM source3
        WHERE entity_id IN (SELECT UNNEST(?))
    """, [s3_ids]).fetchdf()
else:
    s3_df = pd.DataFrame(
        columns=["entity_id", "business_name", "business_address", "country"]
    )

print("Source1 records:", len(s1_df))
print("Source2 records:", len(s2_df))
print("Source3 records:", len(s3_df))

# --------------------------------------------------
# 5. Create lookups
# --------------------------------------------------

s1_lookup = s1_df.set_index("entity_id").to_dict("index")
s2_lookup = s2_df.set_index("entity_id").to_dict("index")
s3_lookup = s3_df.set_index("entity_id").to_dict("index")

# --------------------------------------------------
# 6. Score
# --------------------------------------------------

def similarity_safe(a, b):
    a = normalize_text(a)
    b = normalize_text(b)

    if not a or not b:
        return 0.0

    return SequenceMatcher(None, a, b).ratio()

scored_batch = []

for _, row in batch.iterrows():

    s1_id = row["source1_entity_id"]
    candidate_id = row["candidate_entity_id"]

    s1 = s1_lookup.get(s1_id)

    if candidate_id.startswith("S2-"):
        candidate = s2_lookup.get(candidate_id)
    else:
        candidate = s3_lookup.get(candidate_id)

    if s1 is None or candidate is None:
        continue

    scored_batch.append({
        "source1_entity_id": s1_id,
        "candidate_entity_id": candidate_id,
        "source1_country": s1["country"],
        "candidate_country": candidate["country"],
        "name_similarity": similarity_safe(
            s1["business_name"],
            candidate["business_name"]
        ),
        "address_similarity": similarity_safe(
            s1["business_address"],
            candidate["business_address"]
        )
    })

scored_batch_df = pd.DataFrame(scored_batch)

print("Successfully scored:", len(scored_batch_df))

# --------------------------------------------------
# 7. Insert into scored_candidates
# --------------------------------------------------

if len(scored_batch_df) > 0:

    con.register("scored_batch_df", scored_batch_df)

    con.execute("""
        INSERT INTO scored_candidates
        SELECT
            source1_entity_id,
            candidate_entity_id,
            source1_country,
            candidate_country,
            name_similarity,
            address_similarity
        FROM scored_batch_df
    """)

    con.unregister("scored_batch_df")

print("Inserted successfully.")

# --------------------------------------------------
# 8. Verify database count
# --------------------------------------------------

total_scored = con.execute("""
    SELECT COUNT(*)
    FROM scored_candidates
""").fetchone()[0]

print("Total scored rows now:", total_scored)

Candidate rows: 10000
Source1 records: 9483
Source2 records: 1466
Source3 records: 0
Successfully scored: 10000
Inserted successfully.
Total scored rows now: 1460000


In [53]:
import pandas as pd
from difflib import SequenceMatcher

BATCH_SIZE = 10_000
START_OFFSET = 1_460_000
TOTAL_CANDIDATES = 9_765_660

def similarity_safe(a, b):
    a = normalize_text(a)
    b = normalize_text(b)

    if not a or not b:
        return 0.0

    return SequenceMatcher(None, a, b).ratio()


offset = START_OFFSET
batch_number = 0

while offset < TOTAL_CANDIDATES:

    batch_number += 1

    print("\n" + "=" * 60)
    print("Batch:", batch_number)
    print("Offset:", offset)

    # --------------------------------------------------
    # 1. Get candidate batch
    # --------------------------------------------------

    batch = con.execute(f"""
        SELECT source1_entity_id, candidate_entity_id
        FROM candidates
        LIMIT {BATCH_SIZE} OFFSET {offset}
    """).fetchdf()

    if len(batch) == 0:
        print("No more candidates.")
        break

    print("Candidate rows:", len(batch))

    # --------------------------------------------------
    # 2. Get Source1 records
    # --------------------------------------------------

    s1_ids = batch["source1_entity_id"].unique().tolist()

    s1_df = con.execute("""
        SELECT entity_id, business_name, business_address, country
        FROM source1
        WHERE entity_id IN (SELECT UNNEST(?))
    """, [s1_ids]).fetchdf()

    # --------------------------------------------------
    # 3. Split S2 / S3
    # --------------------------------------------------

    candidate_ids = batch["candidate_entity_id"].tolist()

    s2_ids = [x for x in candidate_ids if x.startswith("S2-")]
    s3_ids = [x for x in candidate_ids if x.startswith("S3-")]

    # --------------------------------------------------
    # 4. Get Source2 records
    # --------------------------------------------------

    if s2_ids:
        s2_df = con.execute("""
            SELECT entity_id, business_name, business_address, country
            FROM source2
            WHERE entity_id IN (SELECT UNNEST(?))
        """, [s2_ids]).fetchdf()
    else:
        s2_df = pd.DataFrame(
            columns=["entity_id", "business_name", "business_address", "country"]
        )

    # --------------------------------------------------
    # 5. Get Source3 records
    # --------------------------------------------------

    if s3_ids:
        s3_df = con.execute("""
            SELECT entity_id, business_name, business_address, country
            FROM source3
            WHERE entity_id IN (SELECT UNNEST(?))
        """, [s3_ids]).fetchdf()
    else:
        s3_df = pd.DataFrame(
            columns=["entity_id", "business_name", "business_address", "country"]
        )

    print("Source1 records:", len(s1_df))
    print("Source2 records:", len(s2_df))
    print("Source3 records:", len(s3_df))

    # --------------------------------------------------
    # 6. Create lookups
    # --------------------------------------------------

    s1_lookup = s1_df.set_index("entity_id").to_dict("index")
    s2_lookup = s2_df.set_index("entity_id").to_dict("index")
    s3_lookup = s3_df.set_index("entity_id").to_dict("index")

    # --------------------------------------------------
    # 7. Score current batch
    # --------------------------------------------------

    scored_batch = []

    for _, row in batch.iterrows():

        s1_id = row["source1_entity_id"]
        candidate_id = row["candidate_entity_id"]

        s1 = s1_lookup.get(s1_id)

        if candidate_id.startswith("S2-"):
            candidate = s2_lookup.get(candidate_id)
        else:
            candidate = s3_lookup.get(candidate_id)

        if s1 is None or candidate is None:
            continue

        scored_batch.append({
            "source1_entity_id": s1_id,
            "candidate_entity_id": candidate_id,
            "source1_country": s1["country"],
            "candidate_country": candidate["country"],
            "name_similarity": similarity_safe(
                s1["business_name"],
                candidate["business_name"]
            ),
            "address_similarity": similarity_safe(
                s1["business_address"],
                candidate["business_address"]
            )
        })

    scored_batch_df = pd.DataFrame(scored_batch)

    print("Successfully scored:", len(scored_batch_df))

    # --------------------------------------------------
    # 8. Insert
    # --------------------------------------------------

    if len(scored_batch_df) > 0:

        con.register("scored_batch_df", scored_batch_df)

        con.execute("""
            INSERT INTO scored_candidates
            SELECT
                source1_entity_id,
                candidate_entity_id,
                source1_country,
                candidate_country,
                name_similarity,
                address_similarity
            FROM scored_batch_df
        """)

        con.unregister("scored_batch_df")

    # --------------------------------------------------
    # 9. Progress
    # --------------------------------------------------

    offset += len(batch)

    print("Progress:", offset, "/", TOTAL_CANDIDATES)

print("\n" + "=" * 60)
print("SCORING COMPLETE")

total_scored = con.execute("""
    SELECT COUNT(*)
    FROM scored_candidates
""").fetchone()[0]

print("Total scored rows:", total_scored)


Batch: 1
Offset: 1460000
Candidate rows: 10000
Source1 records: 9787
Source2 records: 1659
Source3 records: 0
Successfully scored: 10000
Progress: 1470000 / 9765660

Batch: 2
Offset: 1470000
Candidate rows: 10000
Source1 records: 9705
Source2 records: 1746
Source3 records: 0
Successfully scored: 10000
Progress: 1480000 / 9765660

Batch: 3
Offset: 1480000
Candidate rows: 10000
Source1 records: 9396
Source2 records: 1635
Source3 records: 0
Successfully scored: 10000
Progress: 1490000 / 9765660

Batch: 4
Offset: 1490000
Candidate rows: 10000
Source1 records: 9562
Source2 records: 1525
Source3 records: 0
Successfully scored: 10000
Progress: 1500000 / 9765660

Batch: 5
Offset: 1500000
Candidate rows: 10000
Source1 records: 9686
Source2 records: 1843
Source3 records: 0
Successfully scored: 10000
Progress: 1510000 / 9765660

Batch: 6
Offset: 1510000
Candidate rows: 10000
Source1 records: 9817
Source2 records: 1731
Source3 records: 0
Successfully scored: 10000
Progress: 1520000 / 9765660

Bat

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Source1 records: 9766
Source2 records: 1519
Source3 records: 364
Successfully scored: 10000
Progress: 5720000 / 9765660

Batch: 427
Offset: 5720000
Candidate rows: 10000
Source1 records: 9522
Source2 records: 0
Source3 records: 1322
Successfully scored: 10000
Progress: 5730000 / 9765660

Batch: 428
Offset: 5730000
Candidate rows: 10000
Source1 records: 9405
Source2 records: 0
Source3 records: 1636
Successfully scored: 10000
Progress: 5740000 / 9765660

Batch: 429
Offset: 5740000
Candidate rows: 10000
Source1 records: 9419
Source2 records: 0
Source3 records: 1357
Successfully scored: 10000
Progress: 5750000 / 9765660

Batch: 430
Offset: 5750000
Candidate rows: 10000
Source1 records: 9326
Source2 records: 0
Source3 records: 1206
Successfully scored: 10000
Progress: 5760000 / 9765660

Batch: 431
Offset: 5760000
Candidate rows: 10000
Source1 records: 9318
Source2 records: 0
Source3 records: 1305
Successfully scored: 10000
Progress: 5770000 / 9765660

Batch: 432
Offset: 5770000
Candidate ro

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Source1 records: 4283
Source2 records: 658
Source3 records: 7
Successfully scored: 10000
Progress: 7130000 / 9765660

Batch: 568
Offset: 7130000
Candidate rows: 10000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Source1 records: 4737
Source2 records: 945
Source3 records: 0
Successfully scored: 10000
Progress: 7140000 / 9765660

Batch: 569
Offset: 7140000
Candidate rows: 10000
Source1 records: 5065
Source2 records: 1071
Source3 records: 0
Successfully scored: 10000
Progress: 7150000 / 9765660

Batch: 570
Offset: 7150000
Candidate rows: 10000
Source1 records: 3936
Source2 records: 805
Source3 records: 0
Successfully scored: 10000
Progress: 7160000 / 9765660

Batch: 571
Offset: 7160000
Candidate rows: 10000
Source1 records: 5657
Source2 records: 1395
Source3 records: 0
Successfully scored: 10000
Progress: 7170000 / 9765660

Batch: 572
Offset: 7170000
Candidate rows: 10000
Source1 records: 5618
Source2 records: 1080
Source3 records: 0
Successfully scored: 10000
Progress: 7180000 / 9765660

Batch: 573
Offset: 7180000
Candidate rows: 10000
Source1 records: 4088
Source2 records: 709
Source3 records: 0
Successfully scored: 10000
Progress: 7190000 / 9765660

Batch: 574
Offset: 7190000
Candidate rows: 1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Source1 records: 6956
Source2 records: 802
Source3 records: 973
Successfully scored: 10000
Progress: 9360000 / 9765660

Batch: 791
Offset: 9360000
Candidate rows: 10000
Source1 records: 4209
Source2 records: 0
Source3 records: 868
Successfully scored: 10000
Progress: 9370000 / 9765660

Batch: 792
Offset: 9370000
Candidate rows: 10000
Source1 records: 6213
Source2 records: 0
Source3 records: 1722
Successfully scored: 10000
Progress: 9380000 / 9765660

Batch: 793
Offset: 9380000
Candidate rows: 10000
Source1 records: 4987
Source2 records: 0
Source3 records: 927
Successfully scored: 10000
Progress: 9390000 / 9765660

Batch: 794
Offset: 9390000
Candidate rows: 10000
Source1 records: 4706
Source2 records: 0
Source3 records: 881
Successfully scored: 10000
Progress: 9400000 / 9765660

Batch: 795
Offset: 9400000
Candidate rows: 10000
Source1 records: 4406
Source2 records: 0
Source3 records: 936
Successfully scored: 10000
Progress: 9410000 / 9765660

Batch: 796
Offset: 9410000
Candidate rows: 1

In [54]:
print("Total scored:", con.execute("""
    SELECT COUNT(*) FROM scored_candidates
""").fetchone()[0])

print("\nScore ranges:")

result = con.execute("""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN name_similarity >= 0.90 AND address_similarity >= 0.80 THEN 1 ELSE 0 END) AS strong_90_80,
        SUM(CASE WHEN name_similarity >= 0.85 AND address_similarity >= 0.80 THEN 1 ELSE 0 END) AS strong_85_80,
        SUM(CASE WHEN name_similarity >= 0.80 AND address_similarity >= 0.70 THEN 1 ELSE 0 END) AS strong_80_70,
        SUM(CASE WHEN name_similarity >= 0.70 AND address_similarity >= 0.60 THEN 1 ELSE 0 END) AS moderate_70_60
    FROM scored_candidates
""").fetchdf()

display(result)

Total scored: 9765660

Score ranges:


,total,strong_90_80,strong_85_80,strong_80_70,moderate_70_60
0,9765660,567902.0,591589.0,984928.0,1715687.0


In [55]:
top_candidates = con.execute("""
    SELECT
        source1_entity_id,
        candidate_entity_id,
        name_similarity,
        address_similarity,
        (0.6 * name_similarity + 0.4 * address_similarity) AS combined_score
    FROM scored_candidates
    ORDER BY combined_score DESC
    LIMIT 30
""").fetchdf()

display(top_candidates)

,source1_entity_id,candidate_entity_id,name_similarity,address_similarity,combined_score
0,S1-152207704,S2-388192690,1.0,1.0,1.0
1,S1-42619325,S2-231853562,1.0,1.0,1.0
2,S1-553636382,S2-693866929,1.0,1.0,1.0
3,S1-754740110,S2-410073643,1.0,1.0,1.0
4,S1-122938289,S2-904131240,1.0,1.0,1.0
5,S1-775622992,S2-71850350,1.0,1.0,1.0
6,S1-862253651,S2-514162097,1.0,1.0,1.0
7,S1-60156178,S2-363645478,1.0,1.0,1.0
8,S1-628162697,S2-40882517,1.0,1.0,1.0
9,S1-477501772,S2-439192368,1.0,1.0,1.0


In [56]:
result = con.execute("""
WITH ranked AS (
    SELECT
        source1_entity_id,
        candidate_entity_id,
        name_similarity,
        address_similarity,
        (0.6 * name_similarity + 0.4 * address_similarity) AS combined_score,
        ROW_NUMBER() OVER (
            PARTITION BY source1_entity_id
            ORDER BY
                (0.6 * name_similarity + 0.4 * address_similarity) DESC,
                name_similarity DESC,
                address_similarity DESC,
                candidate_entity_id
        ) AS rn
    FROM scored_candidates
)
SELECT
    COUNT(*) AS source1_with_candidates,
    SUM(CASE WHEN rn = 1 THEN 1 ELSE 0 END) AS top_candidates,
    SUM(CASE WHEN combined_score = 1.0 THEN 1 ELSE 0 END) AS perfect_pairs,
    SUM(CASE WHEN combined_score >= 0.95 THEN 1 ELSE 0 END) AS score_95_plus,
    SUM(CASE WHEN combined_score >= 0.90 THEN 1 ELSE 0 END) AS score_90_plus
FROM ranked
""").fetchdf()

display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,source1_with_candidates,top_candidates,perfect_pairs,score_95_plus,score_90_plus
0,9765660,1099632.0,64727.0,343399.0,690132.0


In [57]:
ambiguous = con.execute("""
WITH ranked AS (
    SELECT
        source1_entity_id,
        candidate_entity_id,
        name_similarity,
        address_similarity,
        (0.6 * name_similarity + 0.4 * address_similarity) AS combined_score,
        ROW_NUMBER() OVER (
            PARTITION BY source1_entity_id
            ORDER BY
                (0.6 * name_similarity + 0.4 * address_similarity) DESC,
                name_similarity DESC,
                address_similarity DESC,
                candidate_entity_id
        ) AS rn
    FROM scored_candidates
)
SELECT
    source1_entity_id,
    COUNT(*) AS candidate_count,
    MAX(combined_score) AS best_score,
    MAX(CASE WHEN rn = 2 THEN combined_score END) AS second_score
FROM ranked
GROUP BY source1_entity_id
HAVING COUNT(*) > 1
ORDER BY candidate_count DESC
LIMIT 20
""").fetchdf()

display(ambiguous)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,source1_entity_id,candidate_count,best_score,second_score
0,S1-448383797,2040,0.944179,0.871256
1,S1-165227339,2017,0.952094,0.858462
2,S1-275969212,2017,0.866026,0.827188
3,S1-210903361,2017,0.933147,0.874655
4,S1-882984604,2014,0.820765,0.799160
5,S1-135735980,2014,0.853250,0.832973
6,S1-832339061,2013,0.835586,0.756776
7,S1-402661874,2013,0.854132,0.853510
8,S1-69020827,2012,0.728205,0.723577
9,S1-302248901,2012,0.926819,0.844378


In [58]:
result = con.execute("""
SELECT
    COUNT(*) AS total_selected,
    COUNT(DISTINCT source1_entity_id) AS source1_with_selected_matches,
    AVG(match_count) AS avg_matches_per_source1,
    MAX(match_count) AS max_matches_per_source1
FROM (
    SELECT
        source1_entity_id,
        COUNT(*) AS match_count
    FROM scored_candidates
    WHERE
        (0.6 * name_similarity + 0.4 * address_similarity) >= 0.85
        AND name_similarity >= 0.70
        AND address_similarity >= 0.60
    GROUP BY source1_entity_id
)
""").fetchdf()

display(result)

,total_selected,source1_with_selected_matches,avg_matches_per_source1,max_matches_per_source1
0,698698,698698,1.873488,121


In [59]:
high_match = con.execute("""
SELECT
    source1_entity_id,
    COUNT(*) AS selected_match_count,
    MAX(0.6 * name_similarity + 0.4 * address_similarity) AS best_score,
    MIN(0.6 * name_similarity + 0.4 * address_similarity) AS lowest_selected_score
FROM scored_candidates
WHERE
    (0.6 * name_similarity + 0.4 * address_similarity) >= 0.85
    AND name_similarity >= 0.70
    AND address_similarity >= 0.60
GROUP BY source1_entity_id
HAVING COUNT(*) >= 10
ORDER BY selected_match_count DESC
LIMIT 20
""").fetchdf()

display(high_match)

,source1_entity_id,selected_match_count,best_score,lowest_selected_score
0,S1-129802256,121,0.953488,0.850980
1,S1-824744583,121,0.977011,0.850505
2,S1-903178083,119,0.945455,0.851163
3,S1-461910564,119,0.945917,0.851163
4,S1-703957850,118,0.946667,0.850016
5,S1-728180405,117,0.946988,0.850024
6,S1-332668334,117,0.956627,0.850087
7,S1-351293630,117,0.947368,0.850926
8,S1-165562403,117,0.962791,0.851230
9,S1-885612789,117,0.940230,0.850503


In [60]:
check = con.execute("""
SELECT
    sc.source1_entity_id,
    s1.business_name AS source1_name,
    s1.business_address AS source1_address,
    sc.candidate_entity_id,
    CASE
        WHEN sc.candidate_entity_id LIKE 'S2-%' THEN s2.business_name
        ELSE s3.business_name
    END AS candidate_name,
    CASE
        WHEN sc.candidate_entity_id LIKE 'S2-%' THEN s2.business_address
        ELSE s3.business_address
    END AS candidate_address,
    sc.name_similarity,
    sc.address_similarity,
    (0.6 * sc.name_similarity + 0.4 * sc.address_similarity) AS combined_score
FROM scored_candidates sc
JOIN source1 s1
    ON sc.source1_entity_id = s1.entity_id
LEFT JOIN source2 s2
    ON sc.candidate_entity_id = s2.entity_id
LEFT JOIN source3 s3
    ON sc.candidate_entity_id = s3.entity_id
WHERE sc.source1_entity_id = 'S1-129802256'
  AND (0.6 * sc.name_similarity + 0.4 * sc.address_similarity) >= 0.85
ORDER BY combined_score DESC
LIMIT 20
""").fetchdf()

display(check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,source1_entity_id,source1_name,source1_address,candidate_entity_id,candidate_name,candidate_address,name_similarity,address_similarity,combined_score
0,S1-129802256,Bordeaux Club SARL,"26 Rue Achard, Bordeaux, Nouvelle-Aquitaine",S2-123976911,Bordeaux Club Sarl,"42 RUE DE CARDOZE, BORDEAUX, Nouvelle-Aquitaine",1.0,0.883721,0.953488
1,S1-129802256,Bordeaux Club SARL,"26 Rue Achard, Bordeaux, Nouvelle-Aquitaine",S2-917123241,Bordeaux Club SARL,"26 R. GARAT, BORDEAUX, Nouvelle-Aquitaine",1.0,0.877900,0.951160
2,S1-129802256,Bordeaux Club SARL,"26 Rue Achard, Bordeaux, Nouvelle-Aquitaine",S2-204925338,Bordeaux Club SARL,"32 RUE COLBERT, BORDEAUX, Nouvelle-Aquitaine",1.0,0.867470,0.946988
3,S1-129802256,Bordeaux Club SARL,"26 Rue Achard, Bordeaux, Nouvelle-Aquitaine",S2-499730025,Bordeaux Club SARL,"227 R MOUNEYRA, BORDEAUX, Nouvelle-Aquitaine",1.0,0.861049,0.944420
4,S1-129802256,Bordeaux Club SARL,"26 Rue Achard, Bordeaux, Nouvelle-Aquitaine",S2-2152348,Bordeaux Club SARL,"N° 32 R. DE CAROS, BORDEAUX, Nouvelle-Aquitaine",1.0,0.857143,0.942857
5,S1-129802256,Bordeaux Club SARL,"26 Rue Achard, Bordeaux, Nouvelle-Aquitaine",S3-288678668,Bordeaux Club SARL,"4 Rue Dauzats, Bordeaux, Nouvelle-Aquitaine",1.0,0.853659,0.941463
6,S1-129802256,Bordeaux Club SARL,"26 Rue Achard, Bordeaux, Nouvelle-Aquitaine",S3-747772448,Bordeaux Club SARL,"7 Rue Luflade, Bordeaux, Nouvelle-Aquitaine",1.0,0.853659,0.941463
7,S1-129802256,Bordeaux Club SARL,"26 Rue Achard, Bordeaux, Nouvelle-Aquitaine",S2-209233570,Bordeaux Club SARL,"58 R GODARD, BORDEAUX, Nouvelle-Aquitaine",1.0,0.850000,0.940000
8,S1-129802256,Bordeaux Club SARL,"26 Rue Achard, Bordeaux, Nouvelle-Aquitaine",S3-842725455,Bordeaux Club SARL,"00227 Rue Malrec, Bordeaux, Nouvelle-Aquitaine",1.0,0.847059,0.938824
9,S1-129802256,Bordeaux Club SARL,"26 Rue Achard, Bordeaux, Nouvelle-Aquitaine",S2-615810088,Bordeaux Club SARL,"35 RUE BARENES, BORDEAUX, Nouvelle-Aquitaine",1.0,0.843373,0.937349


In [61]:
result = con.execute("""
SELECT
    COUNT(*) AS selected_pairs,
    COUNT(DISTINCT source1_entity_id) AS source1_with_matches,
    AVG(match_count) AS avg_matches_per_source1,
    MAX(match_count) AS max_matches_per_source1
FROM (
    SELECT
        source1_entity_id,
        COUNT(*) AS match_count
    FROM scored_candidates
    WHERE
        name_similarity >= 0.85
        AND address_similarity >= 0.85
        AND (0.6 * name_similarity + 0.4 * address_similarity) >= 0.90
    GROUP BY source1_entity_id
)
""").fetchdf()

display(result)

,selected_pairs,source1_with_matches,avg_matches_per_source1,max_matches_per_source1
0,376417,376417,1.197135,16


In [62]:
result = con.execute("""
WITH selected AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS match_count
    FROM scored_candidates
    WHERE
        name_similarity >= 0.85
        AND address_similarity >= 0.85
        AND (0.6 * name_similarity + 0.4 * address_similarity) >= 0.90
    GROUP BY source1_entity_id
)
SELECT
    SUM(match_count) AS actual_selected_pairs,
    COUNT(*) AS source1_with_matches,
    SUM(CASE WHEN match_count = 1 THEN 1 ELSE 0 END) AS one_match,
    SUM(CASE WHEN match_count > 1 THEN 1 ELSE 0 END) AS multiple_matches,
    MAX(match_count) AS max_matches
FROM selected
""").fetchdf()

display(result)

,actual_selected_pairs,source1_with_matches,one_match,multiple_matches,max_matches
0,450622.0,376417,312550.0,63867.0,16


In [63]:
import os

print("Current directory:", os.getcwd())
print("\nFiles/folders:")
for item in os.listdir("."):
    print(item)

Current directory: /home/sagemaker-user

Files/folders:
.bashrc
.agent
.kiro
.claude
.sagemaker_sql_editor_api_cache
.local
.ipython
.npm
.jupyter
untitled.chat
Untitled.ipynb
.ipynb_checkpoints
.cache
.config
business_entity_resolution.ipynb
output
.duckdb
tmp_duckdb
matching_work
.virtual_documents


In [64]:
import duckdb

con = duckdb.connect("matching_work/matching.duckdb")

con.execute("SET memory_limit='600MB';")
con.execute("SET threads=1;")
con.execute("SET preserve_insertion_order=false;")
con.execute("SET temp_directory='matching_work/duckdb_tmp2';")
con.execute("SET max_temp_directory_size='500MB';")

print("Database connected successfully.")

print("\nTables:")
display(con.execute("SHOW TABLES").fetchdf())

print("\nScored candidates:")
print(
    con.execute("SELECT COUNT(*) FROM scored_candidates").fetchone()[0]
)

NotImplementedException: Not implemented Error: Cannot switch temporary directory after the current one has been used

In [65]:
print("Database connection is still active.")

print("\nTables:")
display(con.execute("SHOW TABLES").fetchdf())

print("\nScored candidates:")
print(
    con.execute("SELECT COUNT(*) FROM scored_candidates").fetchone()[0]
)

Database connection is still active.

Tables:


,name
0,candidates
1,scored_candidates
2,source1
3,source2
4,source3



Scored candidates:
9765660


In [66]:
import os

matches = []

for root, dirs, files in os.walk("/home/sagemaker-user"):
    # Don't waste time searching hidden/system folders
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    
    if "validate_submission.py" in files:
        matches.append(os.path.join(root, "validate_submission.py"))

print("Validator files found:")
for path in matches:
    print(path)

Validator files found:


In [67]:
import os

for base in ["/home/sagemaker-user", "/opt/ml", "/mnt"]:
    if os.path.exists(base):
        print(f"\n--- {base} ---")
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if not d.startswith(".")]
            
            for f in files:
                if f == "validate_submission.py":
                    print(os.path.join(root, f))


--- /home/sagemaker-user ---

--- /opt/ml ---

--- /mnt ---


In [68]:
import os

print("=== output/ ===")
if os.path.exists("output"):
    for item in os.listdir("output"):
        print(item)
else:
    print("output folder not found")

print("\n=== Search for challenge files ===")
for root, dirs, files in os.walk("/home/sagemaker-user"):
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    
    for f in files:
        if f in [
            "train_ground_truth.tsv",
            "test_source1.tsv",
            "test_source2.tsv",
            "test_source3.tsv",
            "Documentation_template.md"
        ]:
            print(os.path.join(root, f))

=== output/ ===
candidate_pairs.tsv

=== Search for challenge files ===
/home/sagemaker-user/matching_work/test_source1.tsv
/home/sagemaker-user/matching_work/test_source2.tsv
/home/sagemaker-user/matching_work/test_source3.tsv


In [69]:
import pandas as pd
import os

path = "output/candidate_pairs.tsv"

print("File exists:", os.path.exists(path))
print("File size (MB):", round(os.path.getsize(path) / (1024**2), 2))

candidate_preview = pd.read_csv(path, sep="\t", nrows=10)

print("\nColumns:")
print(candidate_preview.columns.tolist())

print("\nFirst 10 rows:")
display(candidate_preview)

File exists: True
File size (MB): 0.0


EmptyDataError: No columns to parse from file

In [70]:
import json

notebook_path = "business_entity_resolution.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("Relevant notebook cells:\n")

for i, cell in enumerate(nb["cells"]):
    text = "".join(cell.get("source", []))
    
    if any(word in text.lower() for word in [
        "candidate_pairs",
        "matching_results",
        "matched_entity",
        "candidate_entity"
    ]):
        print("=" * 70)
        print("CELL", i)
        print(text[:8000])

Relevant notebook cells:

CELL 18
# Get the true matches for our 20 sample Source 1 records
truth_dict = dict(
    zip(
        sample_truth["source1_entity_id"],
        sample_truth["matched_entity_ids"].fillna("")
    )
)

# Show how many true matches each sample entity has
for entity_id in sample["entity_id"]:
    matches = truth_dict.get(entity_id, "")
    
    if matches:
        match_list = [x.strip() for x in matches.split(",") if x.strip()]
        print(entity_id, "->", len(match_list), "true matches")
    else:
        print(entity_id, "-> 0 true matches")
CELL 71
# Recreate ground_truth safely and verify the sample IDs

import pandas as pd

bucket = "sagemaker-eu-north-1-152025327817"

# Load only the ground-truth file
ground_truth = pd.read_csv(
    f"s3://{bucket}/dataset/train/train_ground_truth.tsv",
    sep="\t"
)

# Make sure sample_ids exists
if "sample_ids" not in globals():
    sample_ids = [
        "S1-925783039",
        "S1-773889195",
        "S1-377745466",


In [71]:
import json

with open("business_entity_resolution.ipynb", "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "candidate_pairs.tsv",
    "matching_results.tsv",
    "candidate_entity",
    "matched_entity"
]

for i, cell in enumerate(nb["cells"]):
    text = "".join(cell.get("source", []))
    
    lines = []
    for line in text.splitlines():
        if any(k.lower() in line.lower() for k in keywords):
            lines.append(line.strip())

    if lines:
        print(f"\n--- CELL {i} ---")
        for line in lines:
            print(line)


--- CELL 18 ---
sample_truth["matched_entity_ids"].fillna("")

--- CELL 71 ---
sample_gt["true_matches"] = sample_gt["matched_entity_ids"].apply(parse_matches)
["source1_entity_id", "matched_entity_ids"]

--- CELL 91 ---
CANDIDATE_FILE = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")
"candidate_entity_ids": [

--- CELL 92 ---
path = "output/candidate_pairs.tsv"

--- CELL 99 ---
CANDIDATE_FILE = "output/candidate_pairs.tsv"
) AS candidate_entity_ids

--- CELL 100 ---
path = "output/candidate_pairs.tsv"

--- CELL 109 ---
s2.entity_id AS candidate_entity_id
s3.entity_id AS candidate_entity_id

--- CELL 111 ---
s2.entity_id AS candidate_entity_id
s3.entity_id AS candidate_entity_id

--- CELL 114 ---
COUNT(DISTINCT source1_entity_id || '|' || candidate_entity_id) AS unique_pairs
SELECT source1_entity_id, candidate_entity_id
'candidate_entity_id':'VARCHAR'
SELECT source1_entity_id, candidate_entity_id
'candidate_entity_id':'VARCHAR'

--- CELL 115 ---
candidate_entity_id
candidate_entity_id

In [72]:
import json

with open("business_entity_resolution.ipynb", "r", encoding="utf-8") as f:
    nb = json.load(f)

for i in [91, 99, 229, 230]:
    print("\n" + "=" * 80)
    print("CELL", i)
    print("".join(nb["cells"][i].get("source", [])))


CELL 91
import os
import re
import pandas as pd
from unidecode import unidecode

# Paths
BUCKET = "sagemaker-eu-north-1-152025327817"

TEST_S1 = f"s3://{BUCKET}/dataset/test/test_source1.tsv"

S2_INDEX = "candidate_index/source2_index.tsv"
S3_INDEX = "candidate_index/source3_index.tsv"

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CANDIDATE_FILE = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")

# Safety checks
print("Checking files...")

for path in [S2_INDEX, S3_INDEX]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing index: {path}")
    print(path, "OK")

print("Starting candidate generation...")
print("This uses existing indexes; no new large index will be created.")

# Same normalization used when the indexes were created
def clean_series(s):
    s = s.fillna("").astype(str).str.lower()
    s = s.map(unidecode)
    s = s.str.replace(r"[^a-z0-9\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

def make_k

In [73]:
print("First 20 candidate rows:")
display(
    con.execute("""
        SELECT source1_entity_id, candidate_entity_id
        FROM candidates
        LIMIT 20
    """).fetchdf()
)

print("\nCandidate table count:")
print(
    con.execute("SELECT COUNT(*) FROM candidates").fetchone()[0]
)

First 20 candidate rows:


,source1_entity_id,candidate_entity_id
0,S1-607177411,S2-89663826
1,S1-10991820,S2-884102769
2,S1-439579149,S2-375151053
3,S1-984582949,S2-767997013
4,S1-199924207,S2-292567038
5,S1-655789370,S2-378846408
6,S1-714865224,S2-504393659
7,S1-45537298,S2-600460285
8,S1-255955599,S2-560710588
9,S1-565271347,S2-262646344



Candidate table count:
9765660


In [74]:
print("Creating final candidate_pairs.tsv...")

# Remove the empty placeholder
import os

candidate_file = "output/candidate_pairs.tsv"

if os.path.exists(candidate_file):
    os.remove(candidate_file)

con.execute("""
COPY (
    SELECT
        s1.entity_id AS source1_entity_id,
        COALESCE(c.candidate_entity_ids, '') AS candidate_entity_ids
    FROM source1 s1
    LEFT JOIN (
        SELECT
            source1_entity_id,
            string_agg(
                candidate_entity_id,
                ',' ORDER BY candidate_entity_id
            ) AS candidate_entity_ids
        FROM candidates
        GROUP BY source1_entity_id
    ) c
    ON s1.entity_id = c.source1_entity_id
    ORDER BY s1.entity_id
)
TO 'output/candidate_pairs.tsv'
WITH (
    HEADER,
    DELIMITER '\t'
)
""")

print("DONE")
print("File size (MB):",
      round(os.path.getsize(candidate_file) / (1024**2), 2))

Creating final candidate_pairs.tsv...


OutOfMemoryException: Out of Memory Error: failed to allocate data of size 32.0 KiB (572.2 MiB/572.2 MiB used)

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

In [75]:
import os
import pandas as pd

candidate_file = "output/candidate_pairs.tsv"

# Remove the failed/empty output
if os.path.exists(candidate_file):
    os.remove(candidate_file)

BATCH_SIZE = 10_000

# Get all Source1 IDs
s1_ids = con.execute("""
    SELECT entity_id
    FROM source1
    ORDER BY entity_id
""").fetchdf()["entity_id"].tolist()

total_s1 = len(s1_ids)

print("Source1 entities:", total_s1)
print("Batch size:", BATCH_SIZE)

first_write = True

for start in range(0, total_s1, BATCH_SIZE):

    batch_ids = s1_ids[start:start + BATCH_SIZE]

    df = con.execute("""
        SELECT
            s1.entity_id AS source1_entity_id,
            COALESCE(
                string_agg(
                    c.candidate_entity_id,
                    ',' ORDER BY c.candidate_entity_id
                ),
                ''
            ) AS candidate_entity_ids
        FROM (
            SELECT UNNEST(?) AS entity_id
        ) s1
        LEFT JOIN candidates c
            ON c.source1_entity_id = s1.entity_id
        GROUP BY s1.entity_id
        ORDER BY s1.entity_id
    """, [batch_ids]).fetchdf()

    df.to_csv(
        candidate_file,
        sep="\t",
        index=False,
        mode="w" if first_write else "a",
        header=first_write
    )

    first_write = False

    done = min(start + BATCH_SIZE, total_s1)

    if done % 100_000 == 0 or done == total_s1:
        print(f"Written {done:,} / {total_s1:,}", flush=True)

print("\nDONE")
print(
    "File size (MB):",
    round(os.path.getsize(candidate_file) / (1024**2), 2)
)

Source1 entities: 1732544
Batch size: 10000
Written 100,000 / 1,732,544
Written 200,000 / 1,732,544
Written 300,000 / 1,732,544
Written 400,000 / 1,732,544
Written 500,000 / 1,732,544
Written 600,000 / 1,732,544
Written 700,000 / 1,732,544
Written 800,000 / 1,732,544
Written 900,000 / 1,732,544
Written 1,000,000 / 1,732,544
Written 1,100,000 / 1,732,544
Written 1,200,000 / 1,732,544
Written 1,300,000 / 1,732,544
Written 1,400,000 / 1,732,544
Written 1,500,000 / 1,732,544
Written 1,600,000 / 1,732,544
Written 1,700,000 / 1,732,544
Written 1,732,544 / 1,732,544

DONE
File size (MB): 141.96


In [76]:
import pandas as pd
import os

path = "output/candidate_pairs.tsv"

print("File size (MB):", round(os.path.getsize(path)/(1024**2), 2))

preview = pd.read_csv(path, sep="\t", nrows=5, dtype=str)

print("\nColumns:")
print(preview.columns.tolist())

print("\nFirst 5 rows:")
display(preview)

print("\nTotal rows:")
print(
    sum(1 for _ in open(path, "r", encoding="utf-8")) - 1
)

File size (MB): 141.96

Columns:
['source1_entity_id', 'candidate_entity_ids']

First 5 rows:


,source1_entity_id,candidate_entity_ids
0,S1-100000556,"S2-109665397,S2-11431119,S2-119912657,S2-12323..."
1,S1-100001013,"S2-10453354,S2-135727199,S2-184717257,S2-43981..."
2,S1-100001295,"S2-334300764,S2-560501252,S3-409832146"
3,S1-100001834,NaN
4,S1-100003037,"S2-196478030,S2-207887213,S2-24257526,S2-41163..."



Total rows:
1732544


In [77]:
import os

matching_file = "output/matching_results.tsv"

if os.path.exists(matching_file):
    os.remove(matching_file)

print("Creating matching_results.tsv...")

# Write header
with open(matching_file, "w", encoding="utf-8") as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")

BATCH_SIZE = 10_000

# Get all Source1 IDs
s1_ids = con.execute("""
    SELECT entity_id
    FROM source1
    ORDER BY entity_id
""").fetchdf()["entity_id"].tolist()

total_s1 = len(s1_ids)

for start in range(0, total_s1, BATCH_SIZE):

    batch_ids = s1_ids[start:start + BATCH_SIZE]

    matches = con.execute("""
        SELECT
            source1_entity_id,
            candidate_entity_id
        FROM scored_candidates
        WHERE
            source1_entity_id IN (SELECT UNNEST(?))
            AND name_similarity >= 0.85
            AND address_similarity >= 0.85
            AND (
                0.6 * name_similarity
                + 0.4 * address_similarity
            ) >= 0.90
        ORDER BY source1_entity_id, candidate_entity_id
    """, [batch_ids]).fetchdf()

    # Group selected matches
    match_map = {}

    for row in matches.itertuples(index=False):
        match_map.setdefault(
            row.source1_entity_id,
            []
        ).append(row.candidate_entity_id)

    # Write ALL Source1 entities, including no-match entities
    with open(matching_file, "a", encoding="utf-8") as f:
        for s1_id in batch_ids:
            matched = match_map.get(s1_id, [])
            f.write(
                s1_id
                + "\t"
                + ",".join(matched)
                + "\n"
            )

    done = min(start + BATCH_SIZE, total_s1)

    if done % 100_000 == 0 or done == total_s1:
        print(
            f"Written {done:,} / {total_s1:,}",
            flush=True
        )

print("\nDONE")
print(
    "File size (MB):",
    round(os.path.getsize(matching_file)/(1024**2), 2)
)

Creating matching_results.tsv...
Written 100,000 / 1,732,544
Written 200,000 / 1,732,544
Written 300,000 / 1,732,544
Written 400,000 / 1,732,544
Written 500,000 / 1,732,544
Written 600,000 / 1,732,544
Written 700,000 / 1,732,544
Written 800,000 / 1,732,544
Written 900,000 / 1,732,544
Written 1,000,000 / 1,732,544
Written 1,100,000 / 1,732,544
Written 1,200,000 / 1,732,544
Written 1,300,000 / 1,732,544
Written 1,400,000 / 1,732,544
Written 1,500,000 / 1,732,544
Written 1,600,000 / 1,732,544
Written 1,700,000 / 1,732,544
Written 1,732,544 / 1,732,544

DONE
File size (MB): 28.13


In [ ]:
import pandas as pd
import os

candidate_file = "output/candidate_pairs.tsv"
matching_file = "output/matching_results.tsv"

print("=== FILE CHECK ===")
for path in [candidate_file, matching_file]:
    print(
        os.path.basename(path),
        "|",
        round(os.path.getsize(path)/(1024**2), 2),
        "MB"
    )

# Read only the two ID columns
cand = pd.read_csv(
    candidate_file,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

match = pd.read_csv(
    matching_file,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("\nCandidate columns:", cand.columns.tolist())
print("Matching columns:", match.columns.tolist())

print("\nCandidate rows:", len(cand))
print("Matching rows:", len(match))

print("\nUnique Source1 IDs:")
print("Candidate:", cand["source1_entity_id"].nunique())
print("Matching:", match["source1_entity_id"].nunique())

# Check that every matching ID is present in the corresponding candidate list
candidate_map = dict(
    zip(cand["source1_entity_id"], cand["candidate_entity_ids"])
)

bad_rows = []

for row in match.itertuples(index=False):
    s1 = row.source1_entity_id
    matched_ids = [
        x for x in row.matched_entity_ids.split(",")
        if x
    ]

    candidate_ids = set(
        x for x in candidate_map.get(s1, "").split(",")
        if x
    )

    for mid in matched_ids:
        if mid not in candidate_ids:
            bad_rows.append((s1, mid))
            break

print("\nMatched IDs outside candidate lists:", len(bad_rows))

# Count Source1 entities with at least one final match
matched_count = sum(
    bool(x)
    for x in match["matched_entity_ids"]
)

print("Source1 entities with >=1 match:", matched_count)
print(
    "Source1 entities with 0 matches:",
    len(match) - matched_count
)

=== FILE CHECK ===
candidate_pairs.tsv | 141.96 MB
matching_results.tsv | 28.13 MB


In [1]:
import os

candidate_file = "output/candidate_pairs.tsv"
matching_file = "output/matching_results.tsv"

def count_lines(path):
    with open(path, "rb") as f:
        return sum(1 for _ in f) - 1  # subtract header

print("=== ROW COUNTS ===")

candidate_rows = count_lines(candidate_file)
matching_rows = count_lines(matching_file)

print("Candidate rows:", candidate_rows)
print("Matching rows:", matching_rows)

print("\nExpected:", 1_732_544)

print("\n=== HEADERS ===")

with open(candidate_file, "r", encoding="utf-8") as f:
    print("Candidate:", f.readline().strip())

with open(matching_file, "r", encoding="utf-8") as f:
    print("Matching:", f.readline().strip())

print("\n=== FILE STATUS ===")

if candidate_rows == 1_732_544 and matching_rows == 1_732_544:
    print("PASS: Both files contain all Source1 rows.")
else:
    print("CHECK NEEDED: Row counts do not match expected count.")

=== ROW COUNTS ===
Candidate rows: 1732544
Matching rows: 1732544

Expected: 1732544

=== HEADERS ===
Candidate: source1_entity_id	candidate_entity_ids
Matching: source1_entity_id	matched_entity_ids

=== FILE STATUS ===
PASS: Both files contain all Source1 rows.


In [2]:
candidate_file = "output/candidate_pairs.tsv"
matching_file = "output/matching_results.tsv"

bad_count = 0
match_count = 0

with open(candidate_file, "r", encoding="utf-8") as cf, \
     open(matching_file, "r", encoding="utf-8") as mf:

    # Skip headers
    next(cf)
    next(mf)

    for cand_line, match_line in zip(cf, mf):

        cand_s1, cand_ids = cand_line.rstrip("\n").split("\t", 1)
        match_s1, match_ids = match_line.rstrip("\n").split("\t", 1)

        if cand_s1 != match_s1:
            print("ERROR: Source1 order mismatch:", cand_s1, match_s1)
            break

        candidate_set = set(
            x for x in cand_ids.split(",") if x
        )

        for mid in match_ids.split(","):
            if mid:
                match_count += 1

                if mid not in candidate_set:
                    bad_count += 1

                    if bad_count <= 10:
                        print(
                            "Invalid match:",
                            cand_s1,
                            mid
                        )

print("Total final matched pairs:", match_count)
print("Matches outside candidate lists:", bad_count)

if bad_count == 0:
    print("\nPASS: Every final match is contained in its candidate list.")

Total final matched pairs: 450622
Matches outside candidate lists: 0

PASS: Every final match is contained in its candidate list.


In [3]:
import os

print("=== OUTPUT ===")
for name in sorted(os.listdir("output")):
    path = os.path.join("output", name)
    if os.path.isfile(path):
        print(
            f"{name:35} "
            f"{os.path.getsize(path)/(1024**2):.2f} MB"
        )

print("\n=== NOTEBOOK ===")
print(
    "business_entity_resolution.ipynb:",
    round(
        os.path.getsize("business_entity_resolution.ipynb")
        /(1024**2), 2
    ),
    "MB"
)

=== OUTPUT ===
candidate_pairs.tsv                 141.96 MB
matching_results.tsv                28.13 MB

=== NOTEBOOK ===
business_entity_resolution.ipynb: 1.34 MB


In [4]:
import os

print("=== Documentation template ===")
found_doc = []

for base in ["/home/sagemaker-user", "/opt/ml"]:
    if os.path.exists(base):
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if not d.startswith(".")]
            if "Documentation_template.md" in files:
                found_doc.append(
                    os.path.join(root, "Documentation_template.md")
                )

for x in found_doc:
    print(x)

print("\n=== Existing code files ===")
code_ext = {".py", ".sh", ".md"}

for root, dirs, files in os.walk("/home/sagemaker-user"):
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    for f in files:
        if os.path.splitext(f)[1].lower() in code_ext:
            path = os.path.join(root, f)
            if "matching_work" not in path:
                print(path)

=== Documentation template ===

=== Existing code files ===
